In [1]:
import os

# Change to your project directory (update the path as needed)
os.chdir(r"E:\ET_Article_With_MM_SA\Data_For_plots")

# Verify the change
print(os.getcwd())

E:\ET_Article_With_MM_SA\Data_For_plots


In [2]:
import os

# Change to your project directory (update the path as needed)
os.chdir(r"E:\ET_Article_With_MM_SA\Future data ST")

# Verify the change
print(os.getcwd())

E:\ET_Article_With_MM_SA\Future data ST


In [9]:
# -*- coding: utf-8 -*-

import os
import xarray as xr
import pandas as pd
import numpy as np

DATA_DIR = ""

files = {
    "CWatM": os.path.join(DATA_DIR, "ET_CWatM_mm_day.nc"),
    "GLDAS": os.path.join(DATA_DIR, "GLDAS_ET_2000_2019.nc"),
    "SMAP": os.path.join(DATA_DIR, "SMAP_ET_2000_2019.nc"),
}

def inspect_netcdf(label, path):
    print("\n" + "="*80)
    print(f"DATASET: {label}")
    print(f"PATH: {path}")
    print("="*80)

    ds = xr.open_dataset(path)

    print("\n--- FULL DATASET INFO ---")
    print(ds)

    print("\n--- DIMENSIONS ---")
    print(dict(ds.dims))

    print("\n--- COORDINATES ---")
    for c in ds.coords:
        print(f"\nCoordinate: {c}")
        print(f"  dims: {ds[c].dims}")
        print(f"  shape: {ds[c].shape}")
        print(f"  dtype: {ds[c].dtype}")
        print(f"  attrs: {ds[c].attrs}")

        try:
            vals = ds[c].values
            if vals.size > 0:
                print(f"  first values: {vals.flatten()[:5]}")
                print(f"  last values : {vals.flatten()[-5:]}")
        except Exception as e:
            print(f"  could not print values: {e}")

    print("\n--- DATA VARIABLES ---")
    for v in ds.data_vars:
        da = ds[v]
        print(f"\nVariable: {v}")
        print(f"  dims: {da.dims}")
        print(f"  shape: {da.shape}")
        print(f"  dtype: {da.dtype}")
        print(f"  attrs: {da.attrs}")

        try:
            arr = da.values
            print(f"  min: {np.nanmin(arr)}")
            print(f"  max: {np.nanmax(arr)}")
            print(f"  mean: {np.nanmean(arr)}")
        except Exception as e:
            print(f"  could not calculate min/max/mean: {e}")

    print("\n--- POSSIBLE TIME VARIABLES ---")
    possible_time_names = [
        "time", "Time", "TIME",
        "date", "Date",
        "month", "Month",
        "year", "Year",
        "valid_time",
        "t"
    ]

    found_any = False

    for name in possible_time_names:
        if name in ds.variables:
            found_any = True
            print(f"\n{name}:")
            print(ds[name])
            try:
                vals = ds[name].values
                print(f"  first values: {vals.flatten()[:10]}")
                print(f"  last values : {vals.flatten()[-10:]}")
            except Exception as e:
                print(f"  could not print values: {e}")

    if not found_any:
        print("No obvious time variable found from common names.")

    print("\n--- GLOBAL ATTRIBUTES ---")
    print(ds.attrs)

    ds.close()


for label, path in files.items():
    inspect_netcdf(label, path)


DATASET: CWatM
PATH: ET_CWatM_mm_day.nc

--- FULL DATASET INFO ---
<xarray.Dataset> Size: 685kB
Dimensions:          (time: 468, lat: 13, lon: 14)
Coordinates:
  * time             (time) datetime64[ns] 4kB 1981-01-01 ... 2019-12-01
  * lat              (lat) float64 104B 36.75 36.25 35.75 ... 31.75 31.25 30.75
  * lon              (lon) float64 112B 45.25 45.75 46.25 ... 50.75 51.25 51.75
Data variables:
    ET_final_mm_day  (time, lat, lon) float64 681kB ...

--- DIMENSIONS ---
{'time': 468, 'lat': 13, 'lon': 14}

--- COORDINATES ---

Coordinate: lon
  dims: ('lon',)
  shape: (14,)
  dtype: float64
  attrs: {'standard_name': 'longitude', 'long_name': 'longitude', 'units': 'degrees_east', 'axis': 'X'}
  first values: [45.25 45.75 46.25 46.75 47.25]
  last values : [49.75 50.25 50.75 51.25 51.75]

Coordinate: lat
  dims: ('lat',)
  shape: (13,)
  dtype: float64
  attrs: {'standard_name': 'latitude', 'long_name': 'latitude', 'units': 'degrees_north', 'axis': 'Y'}
  first values: [36.75

In [26]:
# -*- coding: utf-8 -*-

"""
Final Historical-Future ET Time Series

Historical:
    CWatM
    GLDAS
    SMAP

Future climate models:
    GFDL
    IPSL
    MPI
    MRI
    UKESM

Scenarios:
    126 = Optimistic / SSP126
    370 = Medium / SSP370
    585 = Pessimistic / SSP585

Future mode:
    totalET_monthavg only

Important final logic:
    1. Read raw data.
    2. Convert units.
    3. Apply bias correction to future models.
    4. Slice historical: 2000-01 to 2019-12.
    5. Slice future: 2020-01 to 2099-12.
    6. Apply 12-month rolling mean.
    7. AFTER rolling:
        Historical:
            CWatM 2019-12 = mean(2019-09, 2019-10, 2019-11)
            GLDAS 2019-12 = mean(2019-09, 2019-10, 2019-11)

        Future:
            2020-01 = mean(2020-02, 2020-03, 2020-04)
            2099-12 = mean(2099-09, 2099-10, 2099-11)

    8. Plot the corrected rolling series.
"""

import os
import re
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt


# ============================================================
# 1. PATH SETTINGS
# ============================================================

BASE_DIR = os.getcwd()

CWATM_FILE = os.path.join(BASE_DIR, "ET_CWatM_mm_day.nc")
GLDAS_FILE = os.path.join(BASE_DIR, "GLDAS_ET_2000_2019.nc")
SMAP_FILE = os.path.join(BASE_DIR, "SMAP_ET_2000_2019.nc")

FUTURE_BASE_DIR = BASE_DIR


# ============================================================
# 2. MODELS AND SCENARIOS
# ============================================================

CLIMATE_MODELS = [
    "GFDL",
    "IPSL",
    "MPI",
    "MRI",
    "UKESM",
]

SCENARIOS = {
    "126": "Optimistic / SSP126",
    "370": "Medium / SSP370",
    "585": "Pessimistic / SSP585",
}

FUTURE_TOTAL_ET_FILE = "totalET_monthavg.nc"
FUTURE_TOTAL_ET_VAR = "totalET_monthavg"


# ============================================================
# 3. OUTPUT SETTINGS
# ============================================================

OUT_DIR = os.path.join(BASE_DIR, "outputs_historical_future_ET_final_after_rolling_fix")
TABLE_DIR = os.path.join(OUT_DIR, "tables")
FIG_DIR = os.path.join(OUT_DIR, "figures")

for d in [OUT_DIR, TABLE_DIR, FIG_DIR]:
    os.makedirs(d, exist_ok=True)


# ============================================================
# 4. ANALYSIS SETTINGS
# ============================================================

PLOT_START_DATE = "2000-01-01"
PLOT_END_DATE = "2100-01-01"

HISTORICAL_START_DATE = "2000-01-01"
HISTORICAL_END_DATE = "2019-12-01"

FUTURE_START_DATE = "2020-01-01"
FUTURE_END_DATE = "2099-12-01"

HISTORICAL_UNIT = "mm/day"

FUTURE_CONVERT_M_TO_MM = True

MASK_NEGATIVE_HISTORICAL_ET = True
MASK_NEGATIVE_FUTURE_ET = True

APPLY_BIAS_CORRECTION = True
BIAS_REFERENCE = "CWatM"
BIAS_METHOD = "multiplicative_mean_ratio"

# 12-month moving average for final figure
ROLLING_MONTHS = 12
ROLLING_CENTER = True
ROLLING_MIN_PERIODS = 6

SAVE_ANNUAL_SERIES = True

# Corrections are applied AFTER rolling
CORRECT_AFTER_ROLLING = True

# Historical correction after rolling
HISTORICAL_FIXED_COLUMNS = ["CWatM", "GLDAS"]
HISTORICAL_TARGET_DATE = "2019-12-01"
HISTORICAL_SOURCE_DATES = ["2019-09-01", "2019-10-01", "2019-11-01"]

# Future correction after rolling
FUTURE_FIRST_TARGET_DATE = "2020-01-01"
FUTURE_FIRST_SOURCE_DATES = ["2020-02-01", "2020-03-01", "2020-04-01"]

FUTURE_LAST_TARGET_DATE = "2099-12-01"
FUTURE_LAST_SOURCE_DATES = ["2099-09-01", "2099-10-01", "2099-11-01"]


# ============================================================
# 5. FONT AND FIGURE SETTINGS
# ============================================================

FONT_FAMILY = "DejaVu Sans"

TITLE_FONT_SIZE = 20
AXIS_LABEL_FONT_SIZE = 18
TICK_FONT_SIZE = 15
LEGEND_FONT_SIZE = 12
PERIOD_LABEL_FONT_SIZE = 14

FIG_WIDTH = 18
FIG_HEIGHT = 7

LINE_WIDTH_HISTORICAL = 2
LINE_WIDTH_FUTURE_MODEL = 1.2
LINE_WIDTH_ENSEMBLE = 2.5

ALPHA_FUTURE_MODEL = 0.45

plt.rcParams["font.family"] = FONT_FAMILY
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.dpi"] = 150
plt.rcParams["axes.titlesize"] = TITLE_FONT_SIZE
plt.rcParams["axes.labelsize"] = AXIS_LABEL_FONT_SIZE
plt.rcParams["xtick.labelsize"] = TICK_FONT_SIZE
plt.rcParams["ytick.labelsize"] = TICK_FONT_SIZE
plt.rcParams["legend.fontsize"] = LEGEND_FONT_SIZE


# ============================================================
# 6. BASIC FUNCTIONS
# ============================================================

def ensure_month_start_index(series_or_df):
    obj = series_or_df.copy()
    obj.index = pd.to_datetime(obj.index)
    obj.index = obj.index.to_period("M").to_timestamp()
    obj = obj.sort_index()
    return obj


def apply_rolling(series, months=12):
    if months is None or months <= 1:
        return series.copy()

    return series.rolling(
        window=months,
        center=ROLLING_CENTER,
        min_periods=ROLLING_MIN_PERIODS
    ).mean()


def apply_rolling_to_dataframe(df, months=12):
    out = pd.DataFrame(index=df.index)

    for col in df.columns:
        out[col] = apply_rolling(df[col], months=months)

    return out


def get_y_label():
    return "Mean monthly daily ET (mm/day)"


def save_series_annual(df, out_csv):
    annual_df = df.copy()
    annual_df.index = pd.to_datetime(annual_df.index)
    annual_df = annual_df.resample("YE").mean()
    annual_df.index = annual_df.index.year
    annual_df.index.name = "Year"
    annual_df.to_csv(out_csv, encoding="utf-8-sig")
    return annual_df


def replace_target_month_with_mean(series, target_date, source_dates):
    """
    Replace target month with mean of specified source months.
    This function can be applied to raw or rolling series.
    """

    s = series.copy()

    target_date = pd.Timestamp(target_date).to_period("M").to_timestamp()
    source_dates = [
        pd.Timestamp(d).to_period("M").to_timestamp()
        for d in source_dates
    ]

    if target_date not in s.index:
        return s, np.nan, np.nan, "target_date_not_found"

    available_source_dates = [d for d in source_dates if d in s.index]

    if len(available_source_dates) == 0:
        return s, s.loc[target_date], np.nan, "no_source_dates_found"

    source_values = s.loc[available_source_dates].dropna()

    if len(source_values) == 0:
        return s, s.loc[target_date], np.nan, "source_values_all_nan"

    original_value = s.loc[target_date]
    replacement_value = source_values.mean()

    s.loc[target_date] = replacement_value

    return s, original_value, replacement_value, "replaced"


# ============================================================
# 7. HISTORICAL DATA READER
# ============================================================

def extract_dates_from_long_name(long_name_attr):
    if isinstance(long_name_attr, str):
        long_names = [long_name_attr]
    else:
        long_names = list(long_name_attr)

    dates = []

    for item in long_names:
        item = str(item)
        match = re.search(r"(\d{4})_(\d{2})", item)

        if match is None:
            raise ValueError(f"Cannot extract date from long_name item: {item}")

        dates.append(
            pd.Timestamp(
                year=int(match.group(1)),
                month=int(match.group(2)),
                day=1
            )
        )

    return pd.DatetimeIndex(dates)


def standardize_xy_to_latlon(da):
    rename_dict = {}

    for name in list(da.dims) + list(da.coords):
        low = str(name).lower()

        if low in ["x", "longitude"]:
            rename_dict[name] = "lon"

        elif low in ["y", "latitude"]:
            rename_dict[name] = "lat"

    if rename_dict:
        da = da.rename(rename_dict)

    if "lat" in da.coords:
        da = da.sortby("lat")

    if "lon" in da.coords:
        da = da.sortby("lon")

    return da


def open_historical_et(path, dataset_name):
    print("\n" + "=" * 80)
    print(f"Opening historical dataset: {dataset_name}")
    print(path)
    print("=" * 80)

    if not os.path.exists(path):
        raise FileNotFoundError(path)

    ds = xr.open_dataset(path)

    candidate_vars = [
        v for v in ds.data_vars
        if str(v).lower() not in ["spatial_ref", "crs"]
    ]

    if len(candidate_vars) == 0:
        raise ValueError(f"No variable found in {dataset_name}")

    if "ET_final_mm_day" in candidate_vars:
        var_name = "ET_final_mm_day"
    elif "__xarray_dataarray_variable__" in candidate_vars:
        var_name = "__xarray_dataarray_variable__"
    else:
        var_name = candidate_vars[0]

    da = ds[var_name]

    if "time" in da.dims:
        da["time"] = pd.to_datetime(da.time.values)

    elif "band" in da.dims:
        long_name = da.attrs.get("long_name", None)

        if long_name is None:
            raise ValueError(f"{dataset_name} has band dimension but no long_name.")

        dates = extract_dates_from_long_name(long_name)

        if len(dates) != da.sizes["band"]:
            raise ValueError(
                f"{dataset_name}: extracted dates do not match band size."
            )

        da = da.rename({"band": "time"})
        da = da.assign_coords(time=dates)

    else:
        raise ValueError(f"No time or band dimension found in {dataset_name}")

    da = standardize_xy_to_latlon(da)

    for d in ["time", "lat", "lon"]:
        if d not in da.dims:
            raise ValueError(f"{dataset_name}: required dimension '{d}' not found.")

    da = da.astype("float64")
    da = da.where(np.isfinite(da))

    if MASK_NEGATIVE_HISTORICAL_ET:
        da = da.where(da >= 0)

    da.name = dataset_name
    da.attrs["unit"] = HISTORICAL_UNIT

    print(f"Variable: {var_name}")
    print(f"Dims: {da.dims}")
    print(f"Shape: {da.shape}")
    print(
        "Time:",
        pd.to_datetime(da.time.values[0]).strftime("%Y-%m"),
        "to",
        pd.to_datetime(da.time.values[-1]).strftime("%Y-%m")
    )
    print(f"Mean: {float(da.mean(skipna=True)):.6f}")

    return da


# ============================================================
# 8. CELL AREA AND AREA-WEIGHTED MEAN
# ============================================================

def infer_bounds_from_centers(values):
    values = np.asarray(values, dtype=float)

    if len(values) < 2:
        raise ValueError("At least two coordinate values are needed.")

    values_sorted = np.sort(values)
    mid = (values_sorted[:-1] + values_sorted[1:]) / 2.0

    first = values_sorted[0] - (mid[0] - values_sorted[0])
    last = values_sorted[-1] + (values_sorted[-1] - mid[-1])

    return np.concatenate([[first], mid, [last]])


def calculate_latlon_cell_area(lat, lon, radius=6371000.0):
    lat_values = np.asarray(lat.values if hasattr(lat, "values") else lat, dtype=float)
    lon_values = np.asarray(lon.values if hasattr(lon, "values") else lon, dtype=float)

    lat_ascending = np.all(np.diff(lat_values) > 0)
    lon_ascending = np.all(np.diff(lon_values) > 0)

    lat_sorted = np.sort(lat_values)
    lon_sorted = np.sort(lon_values)

    lat_bounds = infer_bounds_from_centers(lat_sorted)
    lon_bounds = infer_bounds_from_centers(lon_sorted)

    lat_bounds_rad = np.deg2rad(lat_bounds)
    lon_bounds_rad = np.deg2rad(lon_bounds)

    dlon = np.diff(lon_bounds_rad)
    sin_lat_diff = np.diff(np.sin(lat_bounds_rad))

    area_sorted = (radius ** 2) * sin_lat_diff[:, None] * dlon[None, :]
    area_sorted = np.abs(area_sorted)

    if not lat_ascending:
        area_sorted = area_sorted[::-1, :]

    if not lon_ascending:
        area_sorted = area_sorted[:, ::-1]

    area = xr.DataArray(
        area_sorted,
        dims=("lat", "lon"),
        coords={
            "lat": lat_values,
            "lon": lon_values,
        },
        name="cell_area"
    )

    area.attrs["units"] = "m2"

    return area


def area_weighted_mean(da):
    area = calculate_latlon_cell_area(da.lat, da.lon)
    valid = da.notnull()

    weighted_sum = (da * area).where(valid).sum(dim=("lat", "lon"), skipna=True)
    weight_sum = area.where(valid).sum(dim=("lat", "lon"), skipna=True)

    out = weighted_sum / weight_sum
    out.name = da.name

    return out


def to_monthly_area_series(da):
    s = area_weighted_mean(da).to_series()
    s = ensure_month_start_index(s)
    return s


# ============================================================
# 9. DOMAIN FUNCTIONS
# ============================================================

def get_historical_domain(*arrays):
    lat_min = max([float(da.lat.min()) for da in arrays])
    lat_max = min([float(da.lat.max()) for da in arrays])
    lon_min = max([float(da.lon.min()) for da in arrays])
    lon_max = min([float(da.lon.max()) for da in arrays])

    return lat_min, lat_max, lon_min, lon_max


def clip_to_domain(da, domain):
    lat_min, lat_max, lon_min, lon_max = domain

    da = da.sortby("lat")
    da = da.sortby("lon")

    return da.sel(
        lat=slice(lat_min, lat_max),
        lon=slice(lon_min, lon_max)
    )


# ============================================================
# 10. FUTURE TOTAL ET READER
# ============================================================

def open_future_total_et(model_name, scenario, hist_domain):
    scen_dir = os.path.join(FUTURE_BASE_DIR, model_name, scenario)
    total_path = os.path.join(scen_dir, FUTURE_TOTAL_ET_FILE)

    if not os.path.exists(total_path):
        raise FileNotFoundError(total_path)

    ds = xr.open_dataset(total_path)

    candidate_vars = [
        v for v in ds.data_vars
        if str(v).lower() not in ["spatial_ref", "crs"]
    ]

    if FUTURE_TOTAL_ET_VAR in candidate_vars:
        var_name = FUTURE_TOTAL_ET_VAR
    elif len(candidate_vars) == 1:
        var_name = candidate_vars[0]
    else:
        raise ValueError(
            f"Cannot select future total ET variable from {total_path}. "
            f"Candidates: {candidate_vars}"
        )

    da = ds[var_name]

    if "time" not in da.dims:
        raise ValueError(f"No time dimension found in {total_path}")

    da["time"] = pd.to_datetime(da.time.values)

    da = standardize_xy_to_latlon(da)

    for d in ["time", "lat", "lon"]:
        if d not in da.dims:
            raise ValueError(f"{model_name}/{scenario}: dimension '{d}' not found.")

    da = da.astype("float64")
    da = da.where(np.isfinite(da))

    da = clip_to_domain(da, hist_domain)

    if FUTURE_CONVERT_M_TO_MM:
        da = da * 1000.0

    if MASK_NEGATIVE_FUTURE_ET:
        da = da.where(da >= 0)

    da.name = f"{model_name}_{scenario}_totalET"

    return da


# ============================================================
# 11. BIAS CORRECTION
# ============================================================

def bias_correct_future_series(future_series, reference_series):
    """
    Multiplicative mean-ratio bias correction over overlap period.
    """

    df = pd.concat([future_series, reference_series], axis=1).dropna()
    df.columns = ["future", "reference"]

    n_overlap = len(df)

    if n_overlap < 12:
        return future_series, np.nan, n_overlap

    future_mean = df["future"].mean()
    reference_mean = df["reference"].mean()

    if not np.isfinite(future_mean) or future_mean == 0:
        return future_series, np.nan, n_overlap

    factor = reference_mean / future_mean

    corrected = future_series * factor
    corrected.name = future_series.name

    return corrected, factor, n_overlap


# ============================================================
# 12. AFTER-ROLLING CORRECTIONS
# ============================================================

def correct_historical_after_rolling(historical_plot_df):
    """
    Apply corrections to rolling historical series:
        CWatM 2019-12 = mean(2019-09, 2019-10, 2019-11)
        GLDAS 2019-12 = mean(2019-09, 2019-10, 2019-11)
    """

    df = historical_plot_df.copy()
    records = []

    if not CORRECT_AFTER_ROLLING:
        return df, records

    for col in HISTORICAL_FIXED_COLUMNS:
        if col in df.columns:
            corrected_series, original_value, replacement_value, status = replace_target_month_with_mean(
                df[col],
                target_date=HISTORICAL_TARGET_DATE,
                source_dates=HISTORICAL_SOURCE_DATES
            )

            df[col] = corrected_series

            final_value = (
                df[col].loc[pd.Timestamp(HISTORICAL_TARGET_DATE)]
                if pd.Timestamp(HISTORICAL_TARGET_DATE) in df.index
                else np.nan
            )

            records.append({
                "stage": "after_rolling",
                "series": col,
                "target_date": HISTORICAL_TARGET_DATE,
                "source_dates": ", ".join(HISTORICAL_SOURCE_DATES),
                "method": "target_equals_mean_of_sources",
                "original_value": original_value,
                "replacement_value": replacement_value,
                "final_value": final_value,
                "status": status,
            })

    return df, records


def correct_future_after_rolling(future_plot_df, scenario):
    """
    Apply corrections to rolling future model series:
        2020-01 = mean(2020-02, 2020-03, 2020-04)
        2099-12 = mean(2099-09, 2099-10, 2099-11)
    """

    df = future_plot_df.copy()
    records = []

    if not CORRECT_AFTER_ROLLING:
        return df, records

    for col in df.columns:

        corrected_series, original_value, replacement_value, status = replace_target_month_with_mean(
            df[col],
            target_date=FUTURE_FIRST_TARGET_DATE,
            source_dates=FUTURE_FIRST_SOURCE_DATES
        )

        df[col] = corrected_series

        final_value = (
            df[col].loc[pd.Timestamp(FUTURE_FIRST_TARGET_DATE)]
            if pd.Timestamp(FUTURE_FIRST_TARGET_DATE) in df.index
            else np.nan
        )

        records.append({
            "stage": "after_rolling",
            "scenario": scenario,
            "model": col,
            "target_date": FUTURE_FIRST_TARGET_DATE,
            "source_dates": ", ".join(FUTURE_FIRST_SOURCE_DATES),
            "method": "target_equals_mean_of_sources",
            "original_value": original_value,
            "replacement_value": replacement_value,
            "final_value": final_value,
            "status": status,
        })

        corrected_series, original_value, replacement_value, status = replace_target_month_with_mean(
            df[col],
            target_date=FUTURE_LAST_TARGET_DATE,
            source_dates=FUTURE_LAST_SOURCE_DATES
        )

        df[col] = corrected_series

        final_value = (
            df[col].loc[pd.Timestamp(FUTURE_LAST_TARGET_DATE)]
            if pd.Timestamp(FUTURE_LAST_TARGET_DATE) in df.index
            else np.nan
        )

        records.append({
            "stage": "after_rolling",
            "scenario": scenario,
            "model": col,
            "target_date": FUTURE_LAST_TARGET_DATE,
            "source_dates": ", ".join(FUTURE_LAST_SOURCE_DATES),
            "method": "target_equals_mean_of_sources",
            "original_value": original_value,
            "replacement_value": replacement_value,
            "final_value": final_value,
            "status": status,
        })

    return df, records


# ============================================================
# 13. PERIOD LINES AND LABELS
# ============================================================

PERIOD_LINES = [
    pd.Timestamp("2025-01-01"),
    pd.Timestamp("2050-01-01"),
    pd.Timestamp("2075-01-01"),
    pd.Timestamp("2100-01-01"),
]

PERIOD_LABELS = [
    ("Historical", pd.Timestamp("2000-01-01"), pd.Timestamp("2019-12-01")),
    ("Near future", pd.Timestamp("2025-01-01"), pd.Timestamp("2049-12-01")),
    ("Mid-century", pd.Timestamp("2050-01-01"), pd.Timestamp("2074-12-01")),
    ("Far future", pd.Timestamp("2075-01-01"), pd.Timestamp("2099-12-01")),
]


# ============================================================
# 14. PLOTTING
# ============================================================

def plot_scenario_timeseries(
    scenario_label,
    historical_plot_df,
    future_plot_df,
    ensemble_plot_series,
    out_png
):
    fig, ax = plt.subplots(figsize=(FIG_WIDTH, FIG_HEIGHT))

    hist_colors = {
        "CWatM": "black",
        "GLDAS": "tab:blue",
        "SMAP": "tab:green",
    }

    model_colors = {
        "GFDL": "tab:blue",
        "IPSL": "tab:orange",
        "MPI": "tab:green",
        "MRI": "tab:purple",
        "UKESM": "tab:brown",
    }

    for col in historical_plot_df.columns:
        ax.plot(
            historical_plot_df.index,
            historical_plot_df[col],
            linewidth=LINE_WIDTH_HISTORICAL,
            label=f"Historical {col}",
            color=hist_colors.get(col, None),
            zorder=3
        )

    for col in future_plot_df.columns:
        ax.plot(
            future_plot_df.index,
            future_plot_df[col],
            linewidth=LINE_WIDTH_FUTURE_MODEL,
            alpha=ALPHA_FUTURE_MODEL,
            color=model_colors.get(col, None),
            label=f"Future {col}",
            zorder=4
        )

    ax.plot(
        ensemble_plot_series.index,
        ensemble_plot_series.values,
        linewidth=LINE_WIDTH_ENSEMBLE,
        color="red",
        alpha=0.95,
        label="Future ensemble mean",
        zorder=5
    )

    for dt in PERIOD_LINES:
        ax.axvline(
            dt,
            color="gray",
            linestyle="--",
            linewidth=1.2,
            alpha=0.50,
            zorder=1
        )

    ax.set_xlim(
        pd.Timestamp(PLOT_START_DATE),
        pd.Timestamp(PLOT_END_DATE)
    )

    ymin, ymax = ax.get_ylim()

    for label, start, end in PERIOD_LABELS:
        mid = start + (end - start) / 2

        ax.text(
            mid,
            ymax - 0.035 * (ymax - ymin),
            label,
            ha="center",
            va="top",
            fontsize=PERIOD_LABEL_FONT_SIZE,
            color="gray"
        )

    title_extra = "bias-corrected to historical CWatM" if APPLY_BIAS_CORRECTION else "raw future"

    ax.set_title(
        f"Historical and Future Monthly ET Time Series - {scenario_label}\n"
        f"Future mode: totalET only; {title_extra}",
        fontsize=TITLE_FONT_SIZE
    )

    ax.set_xlabel("Time", fontsize=AXIS_LABEL_FONT_SIZE)
    ax.set_ylabel(get_y_label(), fontsize=AXIS_LABEL_FONT_SIZE)

    ax.tick_params(axis="both", labelsize=TICK_FONT_SIZE)
    ax.grid(True, alpha=0.25)
    ax.legend(
    ncol=3,
    fontsize=LEGEND_FONT_SIZE,
    loc="lower right",
    frameon=True,
    framealpha=0.85
)

    plt.tight_layout()
    plt.savefig(out_png, dpi=300)
    plt.close()


# ============================================================
# 15. MAIN WORKFLOW
# ============================================================

print("\nLoading historical ET datasets...")

cwatm = open_historical_et(CWATM_FILE, "CWatM")
gldas = open_historical_et(GLDAS_FILE, "GLDAS")
smap = open_historical_et(SMAP_FILE, "SMAP")

hist_domain = get_historical_domain(cwatm, gldas, smap)

print("\nHistorical domain intersection:")
print(f"lat: {hist_domain[0]:.3f} to {hist_domain[1]:.3f}")
print(f"lon: {hist_domain[2]:.3f} to {hist_domain[3]:.3f}")

cwatm_clip = clip_to_domain(cwatm, hist_domain)
gldas_clip = clip_to_domain(gldas, hist_domain)
smap_clip = clip_to_domain(smap, hist_domain)

historical_raw_df = pd.concat(
    [
        to_monthly_area_series(cwatm_clip).rename("CWatM"),
        to_monthly_area_series(gldas_clip).rename("GLDAS"),
        to_monthly_area_series(smap_clip).rename("SMAP"),
    ],
    axis=1
)

historical_raw_df = historical_raw_df.sort_index()

historical_raw_df = historical_raw_df.loc[
    (historical_raw_df.index >= pd.Timestamp(HISTORICAL_START_DATE)) &
    (historical_raw_df.index <= pd.Timestamp(HISTORICAL_END_DATE))
]

# Rolling first
historical_plot_df = apply_rolling_to_dataframe(
    historical_raw_df,
    months=ROLLING_MONTHS
)

# Then correction after rolling
historical_plot_df, historical_after_rolling_records = correct_historical_after_rolling(
    historical_plot_df
)

historical_raw_csv = os.path.join(TABLE_DIR, "historical_area_mean_ET_raw_final.csv")
historical_plot_csv = os.path.join(TABLE_DIR, "historical_area_mean_ET_rolling_corrected_final.csv")

historical_raw_df.to_csv(historical_raw_csv, encoding="utf-8-sig")
historical_plot_df.to_csv(historical_plot_csv, encoding="utf-8-sig")

if SAVE_ANNUAL_SERIES:
    save_series_annual(
        historical_raw_df,
        os.path.join(TABLE_DIR, "historical_area_mean_ET_raw_annual_final.csv")
    )


# ============================================================
# 16. FUTURE WORKFLOW
# ============================================================

all_future_summary = []
bias_records = []
future_after_rolling_records_all = []
combined_future_raw_records = []
combined_future_plot_records = []

for scenario, scenario_label in SCENARIOS.items():

    print("\n" + "=" * 80)
    print(f"Processing scenario: {scenario} - {scenario_label}")
    print("=" * 80)

    future_raw_series_list = []

    for model in CLIMATE_MODELS:

        print(f"\nProcessing model: {model}, scenario: {scenario}")

        try:
            future_da = open_future_total_et(
                model_name=model,
                scenario=scenario,
                hist_domain=hist_domain
            )

            future_series_raw = to_monthly_area_series(future_da)
            future_series_raw.name = model

            if APPLY_BIAS_CORRECTION:
                future_series_used, factor, n_overlap = bias_correct_future_series(
                    future_series=future_series_raw,
                    reference_series=historical_raw_df[BIAS_REFERENCE]
                )
            else:
                future_series_used = future_series_raw
                factor = np.nan
                n_overlap = 0

            future_series_used = future_series_used.loc[
                (future_series_used.index >= pd.Timestamp(FUTURE_START_DATE)) &
                (future_series_used.index <= pd.Timestamp(FUTURE_END_DATE))
            ].copy()

            future_series_used.name = model
            future_raw_series_list.append(future_series_used)

            bias_records.append({
                "model": model,
                "scenario": scenario,
                "bias_reference": BIAS_REFERENCE,
                "bias_method": BIAS_METHOD if APPLY_BIAS_CORRECTION else "none",
                "bias_factor": factor,
                "n_overlap_months": n_overlap,
            })

            all_future_summary.append({
                "model": model,
                "scenario": scenario,
                "scenario_label": scenario_label,
                "mode": "totalET_only",
                "bias_corrected": APPLY_BIAS_CORRECTION,
                "bias_factor": factor,
                "n_overlap_months": n_overlap,
                "raw_start": future_series_used.dropna().index.min(),
                "raw_end": future_series_used.dropna().index.max(),
                "raw_n_months": future_series_used.dropna().shape[0],
                "raw_mean": future_series_used.mean(),
                "raw_min": future_series_used.min(),
                "raw_max": future_series_used.max(),
                "domain_lat_min": hist_domain[0],
                "domain_lat_max": hist_domain[1],
                "domain_lon_min": hist_domain[2],
                "domain_lon_max": hist_domain[3],
            })

        except Exception as e:
            print(f"ERROR for {model} / {scenario}: {e}")

    if len(future_raw_series_list) == 0:
        print(f"No valid future data for scenario {scenario}. Plot skipped.")
        continue

    future_raw_df = pd.concat(future_raw_series_list, axis=1)
    future_raw_df = future_raw_df.sort_index()

    available_cols = [m for m in CLIMATE_MODELS if m in future_raw_df.columns]
    future_raw_df = future_raw_df[available_cols]

    # Rolling first
    future_plot_df = apply_rolling_to_dataframe(
        future_raw_df,
        months=ROLLING_MONTHS
    )

    # Then correction after rolling
    future_plot_df, future_after_rolling_records = correct_future_after_rolling(
        future_plot_df,
        scenario=scenario
    )

    future_after_rolling_records_all.extend(future_after_rolling_records)

    # Ensemble must be calculated AFTER rolling and AFTER corrections
    ensemble_plot_series = future_plot_df.mean(axis=1, skipna=True)
    ensemble_plot_series.name = "Future_Ensemble_Mean"

    # Save raw and plot data
    future_raw_csv = os.path.join(
        TABLE_DIR,
        f"future_area_mean_ET_raw_scenario_{scenario}_final.csv"
    )

    future_plot_csv = os.path.join(
        TABLE_DIR,
        f"future_area_mean_ET_rolling_corrected_scenario_{scenario}_final.csv"
    )

    ensemble_plot_csv = os.path.join(
        TABLE_DIR,
        f"future_ensemble_rolling_corrected_scenario_{scenario}_final.csv"
    )

    future_raw_df.to_csv(future_raw_csv, encoding="utf-8-sig")
    future_plot_df.to_csv(future_plot_csv, encoding="utf-8-sig")
    ensemble_plot_series.to_frame().to_csv(ensemble_plot_csv, encoding="utf-8-sig")

    if SAVE_ANNUAL_SERIES:
        save_series_annual(
            future_raw_df,
            os.path.join(
                TABLE_DIR,
                f"future_area_mean_ET_raw_scenario_{scenario}_annual_final.csv"
            )
        )

    # Combined records
    temp_raw = future_raw_df.copy()
    temp_raw["scenario"] = scenario
    temp_raw["scenario_label"] = scenario_label
    temp_raw = temp_raw.reset_index().rename(columns={"index": "time"})
    combined_future_raw_records.append(temp_raw)

    temp_plot = future_plot_df.copy()
    temp_plot["Ensemble_Mean"] = ensemble_plot_series
    temp_plot["scenario"] = scenario
    temp_plot["scenario_label"] = scenario_label
    temp_plot = temp_plot.reset_index().rename(columns={"index": "time"})
    combined_future_plot_records.append(temp_plot)

    # Plot
    out_png = os.path.join(
        FIG_DIR,
        f"historical_future_ET_scenario_{scenario}_final.png"
    )

    plot_scenario_timeseries(
        scenario_label=scenario_label,
        historical_plot_df=historical_plot_df,
        future_plot_df=future_plot_df,
        ensemble_plot_series=ensemble_plot_series,
        out_png=out_png
    )


# ============================================================
# 17. SAVE SUMMARY TABLES
# ============================================================

summary_df = pd.DataFrame(all_future_summary)
summary_csv = os.path.join(TABLE_DIR, "future_models_summary_final.csv")
summary_df.to_csv(summary_csv, index=False, encoding="utf-8-sig")

bias_df = pd.DataFrame(bias_records)
bias_csv = os.path.join(TABLE_DIR, "future_bias_correction_factors_final.csv")
bias_df.to_csv(bias_csv, index=False, encoding="utf-8-sig")

historical_after_rolling_df = pd.DataFrame(historical_after_rolling_records)
historical_after_rolling_csv = os.path.join(
    TABLE_DIR,
    "historical_after_rolling_fixed_months_final.csv"
)
historical_after_rolling_df.to_csv(
    historical_after_rolling_csv,
    index=False,
    encoding="utf-8-sig"
)

future_after_rolling_df = pd.DataFrame(future_after_rolling_records_all)
future_after_rolling_csv = os.path.join(
    TABLE_DIR,
    "future_after_rolling_fixed_months_final.csv"
)
future_after_rolling_df.to_csv(
    future_after_rolling_csv,
    index=False,
    encoding="utf-8-sig"
)

if len(combined_future_raw_records) > 0:
    combined_future_raw_df = pd.concat(combined_future_raw_records, axis=0)
    combined_future_raw_csv = os.path.join(
        TABLE_DIR,
        "future_area_mean_ET_all_models_all_scenarios_raw_final.csv"
    )
    combined_future_raw_df.to_csv(combined_future_raw_csv, index=False, encoding="utf-8-sig")
else:
    combined_future_raw_csv = None

if len(combined_future_plot_records) > 0:
    combined_future_plot_df = pd.concat(combined_future_plot_records, axis=0)
    combined_future_plot_csv = os.path.join(
        TABLE_DIR,
        "future_area_mean_ET_all_models_all_scenarios_rolling_corrected_final.csv"
    )
    combined_future_plot_df.to_csv(combined_future_plot_csv, index=False, encoding="utf-8-sig")
else:
    combined_future_plot_csv = None

config_df = pd.DataFrame([
    {"parameter": "FUTURE_ET_MODE", "value": "totalET_only"},
    {"parameter": "FUTURE_BASE_DIR", "value": FUTURE_BASE_DIR},
    {"parameter": "CLIMATE_MODELS", "value": ", ".join(CLIMATE_MODELS)},
    {"parameter": "SCENARIOS", "value": ", ".join(SCENARIOS.keys())},
    {"parameter": "PLOT_START_DATE", "value": PLOT_START_DATE},
    {"parameter": "PLOT_END_DATE", "value": PLOT_END_DATE},
    {"parameter": "HISTORICAL_START_DATE", "value": HISTORICAL_START_DATE},
    {"parameter": "HISTORICAL_END_DATE", "value": HISTORICAL_END_DATE},
    {"parameter": "FUTURE_START_DATE", "value": FUTURE_START_DATE},
    {"parameter": "FUTURE_END_DATE", "value": FUTURE_END_DATE},
    {"parameter": "APPLY_BIAS_CORRECTION", "value": APPLY_BIAS_CORRECTION},
    {"parameter": "BIAS_REFERENCE", "value": BIAS_REFERENCE},
    {"parameter": "BIAS_METHOD", "value": BIAS_METHOD},
    {"parameter": "ROLLING_MONTHS", "value": ROLLING_MONTHS},
    {"parameter": "ROLLING_CENTER", "value": ROLLING_CENTER},
    {"parameter": "ROLLING_MIN_PERIODS", "value": ROLLING_MIN_PERIODS},
    {"parameter": "CORRECT_AFTER_ROLLING", "value": CORRECT_AFTER_ROLLING},
    {"parameter": "HISTORICAL_FIXED_COLUMNS", "value": ", ".join(HISTORICAL_FIXED_COLUMNS)},
    {"parameter": "HISTORICAL_TARGET_DATE", "value": HISTORICAL_TARGET_DATE},
    {"parameter": "HISTORICAL_SOURCE_DATES", "value": ", ".join(HISTORICAL_SOURCE_DATES)},
    {"parameter": "FUTURE_FIRST_TARGET_DATE", "value": FUTURE_FIRST_TARGET_DATE},
    {"parameter": "FUTURE_FIRST_SOURCE_DATES", "value": ", ".join(FUTURE_FIRST_SOURCE_DATES)},
    {"parameter": "FUTURE_LAST_TARGET_DATE", "value": FUTURE_LAST_TARGET_DATE},
    {"parameter": "FUTURE_LAST_SOURCE_DATES", "value": ", ".join(FUTURE_LAST_SOURCE_DATES)},
    {"parameter": "domain_lat_min", "value": hist_domain[0]},
    {"parameter": "domain_lat_max", "value": hist_domain[1]},
    {"parameter": "domain_lon_min", "value": hist_domain[2]},
    {"parameter": "domain_lon_max", "value": hist_domain[3]},
])

config_csv = os.path.join(TABLE_DIR, "run_configuration_final.csv")
config_df.to_csv(config_csv, index=False, encoding="utf-8-sig")


# ============================================================
# 18. FINAL REPORT
# ============================================================

print("\n" + "=" * 80)
print("DONE")
print("=" * 80)

print("\nMain output folder:")
print(OUT_DIR)

print("\nMain tables:")
print(historical_raw_csv)
print(historical_plot_csv)
print(summary_csv)
print(bias_csv)
print(historical_after_rolling_csv)
print(future_after_rolling_csv)
print(config_csv)

if combined_future_raw_csv is not None:
    print(combined_future_raw_csv)

if combined_future_plot_csv is not None:
    print(combined_future_plot_csv)

print("\nFigures:")
for scenario in SCENARIOS:
    print(
        os.path.join(
            FIG_DIR,
            f"historical_future_ET_scenario_{scenario}_final.png"
        )
    )

print("\nNotes:")
print("Rolling mean is applied first.")
print("Fixed-month corrections are applied after rolling.")
print("Historical CWatM and GLDAS 2019-12 are corrected after rolling.")
print("Future model 2020-01 and 2099-12 are corrected after rolling.")
print("Ensemble mean is calculated after rolling and after corrections.")


Loading historical ET datasets...

Opening historical dataset: CWatM
E:\ET_Article_With_MM_SA\Future data ST\ET_CWatM_mm_day.nc
Variable: ET_final_mm_day
Dims: ('time', 'lat', 'lon')
Shape: (468, 13, 14)
Time: 1981-01 to 2019-12
Mean: 1.102419

Opening historical dataset: GLDAS
E:\ET_Article_With_MM_SA\Future data ST\GLDAS_ET_2000_2019.nc
Variable: __xarray_dataarray_variable__
Dims: ('time', 'lat', 'lon')
Shape: (240, 13, 14)
Time: 2000-01 to 2019-12
Mean: 0.793585

Opening historical dataset: SMAP
E:\ET_Article_With_MM_SA\Future data ST\SMAP_ET_2000_2019.nc
Variable: __xarray_dataarray_variable__
Dims: ('time', 'lat', 'lon')
Shape: (58, 13, 14)
Time: 2015-03 to 2019-12
Mean: 1.057844

Historical domain intersection:
lat: 30.750 to 36.750
lon: 45.250 to 51.750

Processing scenario: 126 - Optimistic / SSP126

Processing model: GFDL, scenario: 126

Processing model: IPSL, scenario: 126

Processing model: MPI, scenario: 126

Processing model: MRI, scenario: 126

Processing model: UKESM,

In [25]:
# -*- coding: utf-8 -*-

"""
Annual Historical-Future ET Time Series

Historical:
    CWatM
    GLDAS
    SMAP

Future climate models:
    GFDL
    IPSL
    MPI
    MRI
    UKESM

Scenarios:
    126 = Optimistic / SSP126
    370 = Medium / SSP370
    585 = Pessimistic / SSP585

Future mode:
    totalET_monthavg only

This version:
    - No moving average / no rolling window
    - Monthly data are converted to annual mean ET
    - Historical annual period: 2000-2019
    - Future annual period: 2020-2099
    - X-axis: 2000 to 2100
    - Vertical lines: 2025, 2050, 2075, 2100
"""

import os
import re
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt


# ============================================================
# 1. PATH SETTINGS
# ============================================================

BASE_DIR = os.getcwd()

CWATM_FILE = os.path.join(BASE_DIR, "ET_CWatM_mm_day.nc")
GLDAS_FILE = os.path.join(BASE_DIR, "GLDAS_ET_2000_2019.nc")
SMAP_FILE = os.path.join(BASE_DIR, "SMAP_ET_2000_2019.nc")

FUTURE_BASE_DIR = BASE_DIR


# ============================================================
# 2. MODELS AND SCENARIOS
# ============================================================

CLIMATE_MODELS = [
    "GFDL",
    "IPSL",
    "MPI",
    "MRI",
    "UKESM",
]

SCENARIOS = {
    "126": "Optimistic / SSP126",
    "370": "Medium / SSP370",
    "585": "Pessimistic / SSP585",
}

FUTURE_TOTAL_ET_FILE = "totalET_monthavg.nc"
FUTURE_TOTAL_ET_VAR = "totalET_monthavg"


# ============================================================
# 3. OUTPUT SETTINGS
# ============================================================

OUT_DIR = os.path.join(BASE_DIR, "outputs_historical_future_ET_annual_mean")
TABLE_DIR = os.path.join(OUT_DIR, "tables")
FIG_DIR = os.path.join(OUT_DIR, "figures")

for d in [OUT_DIR, TABLE_DIR, FIG_DIR]:
    os.makedirs(d, exist_ok=True)


# ============================================================
# 4. ANALYSIS SETTINGS
# ============================================================

PLOT_START_DATE = "2000-01-01"
PLOT_END_DATE = "2100-01-01"

HISTORICAL_START_DATE = "2000-01-01"
HISTORICAL_END_DATE = "2019-12-01"

FUTURE_START_DATE = "2020-01-01"
FUTURE_END_DATE = "2099-12-01"

HISTORICAL_UNIT = "mm/day"

FUTURE_CONVERT_M_TO_MM = True

MASK_NEGATIVE_HISTORICAL_ET = True
MASK_NEGATIVE_FUTURE_ET = True

APPLY_BIAS_CORRECTION = True
BIAS_REFERENCE = "CWatM"
BIAS_METHOD = "multiplicative_mean_ratio"

SAVE_MONTHLY_TABLES = True
SAVE_ANNUAL_TABLES = True

# Monthly edge corrections before annual averaging
CORRECT_MONTHLY_EDGE_VALUES = True

HISTORICAL_FIXED_COLUMNS = ["CWatM", "GLDAS"]

HISTORICAL_TARGET_DATE = "2019-12-01"
HISTORICAL_SOURCE_DATES = [
    "2019-09-01",
    "2019-10-01",
    "2019-11-01",
]

FUTURE_FIRST_TARGET_DATE = "2020-01-01"
FUTURE_FIRST_SOURCE_DATES = [
    "2020-02-01",
    "2020-03-01",
    "2020-04-01",
]

FUTURE_LAST_TARGET_DATE = "2099-12-01"
FUTURE_LAST_SOURCE_DATES = [
    "2099-09-01",
    "2099-10-01",
    "2099-11-01",
]


# ============================================================
# 5. FONT AND FIGURE SETTINGS
# ============================================================

FONT_FAMILY = "DejaVu Sans"

TITLE_FONT_SIZE = 20
AXIS_LABEL_FONT_SIZE = 18
TICK_FONT_SIZE = 15
LEGEND_FONT_SIZE = 12
PERIOD_LABEL_FONT_SIZE = 14

FIG_WIDTH = 18
FIG_HEIGHT = 7

LINE_WIDTH_HISTORICAL = 1
LINE_WIDTH_FUTURE_MODEL = 1
LINE_WIDTH_ENSEMBLE = 2

ALPHA_FUTURE_MODEL = 0.45

plt.rcParams["font.family"] = FONT_FAMILY
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.dpi"] = 150
plt.rcParams["axes.titlesize"] = TITLE_FONT_SIZE
plt.rcParams["axes.labelsize"] = AXIS_LABEL_FONT_SIZE
plt.rcParams["xtick.labelsize"] = TICK_FONT_SIZE
plt.rcParams["ytick.labelsize"] = TICK_FONT_SIZE
plt.rcParams["legend.fontsize"] = LEGEND_FONT_SIZE


# ============================================================
# 6. BASIC FUNCTIONS
# ============================================================

def ensure_month_start_index(series_or_df):
    obj = series_or_df.copy()
    obj.index = pd.to_datetime(obj.index)
    obj.index = obj.index.to_period("M").to_timestamp()
    obj = obj.sort_index()
    return obj


def monthly_to_annual_mean(series_or_df):
    """
    Convert monthly series/dataframe to annual mean.
    Annual timestamp is set to January 1 of each year.
    """

    obj = series_or_df.copy()
    obj.index = pd.to_datetime(obj.index)

    annual = obj.resample("YS").mean()

    return annual


def get_y_label():
    return "Annual mean daily ET (mm/day)"


def replace_target_month_with_mean(series, target_date, source_dates):
    """
    Replace one target month with mean of selected source months.
    """

    s = series.copy()

    target_date = pd.Timestamp(target_date).to_period("M").to_timestamp()
    source_dates = [
        pd.Timestamp(d).to_period("M").to_timestamp()
        for d in source_dates
    ]

    if target_date not in s.index:
        return s, np.nan, np.nan, "target_date_not_found"

    available_source_dates = [d for d in source_dates if d in s.index]

    if len(available_source_dates) == 0:
        return s, s.loc[target_date], np.nan, "no_source_dates_found"

    source_values = s.loc[available_source_dates].dropna()

    if len(source_values) == 0:
        return s, s.loc[target_date], np.nan, "source_values_all_nan"

    original_value = s.loc[target_date]
    replacement_value = source_values.mean()

    s.loc[target_date] = replacement_value

    return s, original_value, replacement_value, "replaced"


# ============================================================
# 7. HISTORICAL DATA READER
# ============================================================

def extract_dates_from_long_name(long_name_attr):
    if isinstance(long_name_attr, str):
        long_names = [long_name_attr]
    else:
        long_names = list(long_name_attr)

    dates = []

    for item in long_names:
        item = str(item)
        match = re.search(r"(\d{4})_(\d{2})", item)

        if match is None:
            raise ValueError(f"Cannot extract date from long_name item: {item}")

        dates.append(
            pd.Timestamp(
                year=int(match.group(1)),
                month=int(match.group(2)),
                day=1
            )
        )

    return pd.DatetimeIndex(dates)


def standardize_xy_to_latlon(da):
    rename_dict = {}

    for name in list(da.dims) + list(da.coords):
        low = str(name).lower()

        if low in ["x", "longitude"]:
            rename_dict[name] = "lon"

        elif low in ["y", "latitude"]:
            rename_dict[name] = "lat"

    if rename_dict:
        da = da.rename(rename_dict)

    if "lat" in da.coords:
        da = da.sortby("lat")

    if "lon" in da.coords:
        da = da.sortby("lon")

    return da


def open_historical_et(path, dataset_name):
    print("\n" + "=" * 80)
    print(f"Opening historical dataset: {dataset_name}")
    print(path)
    print("=" * 80)

    if not os.path.exists(path):
        raise FileNotFoundError(path)

    ds = xr.open_dataset(path)

    candidate_vars = [
        v for v in ds.data_vars
        if str(v).lower() not in ["spatial_ref", "crs"]
    ]

    if len(candidate_vars) == 0:
        raise ValueError(f"No variable found in {dataset_name}")

    if "ET_final_mm_day" in candidate_vars:
        var_name = "ET_final_mm_day"

    elif "__xarray_dataarray_variable__" in candidate_vars:
        var_name = "__xarray_dataarray_variable__"

    else:
        var_name = candidate_vars[0]

    da = ds[var_name]

    if "time" in da.dims:
        da["time"] = pd.to_datetime(da.time.values)

    elif "band" in da.dims:
        long_name = da.attrs.get("long_name", None)

        if long_name is None:
            raise ValueError(f"{dataset_name} has band dimension but no long_name.")

        dates = extract_dates_from_long_name(long_name)

        if len(dates) != da.sizes["band"]:
            raise ValueError(
                f"{dataset_name}: extracted dates do not match band size."
            )

        da = da.rename({"band": "time"})
        da = da.assign_coords(time=dates)

    else:
        raise ValueError(f"No time or band dimension found in {dataset_name}")

    da = standardize_xy_to_latlon(da)

    for d in ["time", "lat", "lon"]:
        if d not in da.dims:
            raise ValueError(f"{dataset_name}: required dimension '{d}' not found.")

    da = da.astype("float64")
    da = da.where(np.isfinite(da))

    if MASK_NEGATIVE_HISTORICAL_ET:
        da = da.where(da >= 0)

    da.name = dataset_name
    da.attrs["unit"] = HISTORICAL_UNIT

    print(f"Variable: {var_name}")
    print(f"Dims: {da.dims}")
    print(f"Shape: {da.shape}")
    print(
        "Time:",
        pd.to_datetime(da.time.values[0]).strftime("%Y-%m"),
        "to",
        pd.to_datetime(da.time.values[-1]).strftime("%Y-%m")
    )
    print(f"Mean: {float(da.mean(skipna=True)):.6f}")

    return da


# ============================================================
# 8. CELL AREA AND AREA-WEIGHTED MEAN
# ============================================================

def infer_bounds_from_centers(values):
    values = np.asarray(values, dtype=float)

    if len(values) < 2:
        raise ValueError("At least two coordinate values are needed.")

    values_sorted = np.sort(values)
    mid = (values_sorted[:-1] + values_sorted[1:]) / 2.0

    first = values_sorted[0] - (mid[0] - values_sorted[0])
    last = values_sorted[-1] + (values_sorted[-1] - mid[-1])

    return np.concatenate([[first], mid, [last]])


def calculate_latlon_cell_area(lat, lon, radius=6371000.0):
    lat_values = np.asarray(lat.values if hasattr(lat, "values") else lat, dtype=float)
    lon_values = np.asarray(lon.values if hasattr(lon, "values") else lon, dtype=float)

    lat_ascending = np.all(np.diff(lat_values) > 0)
    lon_ascending = np.all(np.diff(lon_values) > 0)

    lat_sorted = np.sort(lat_values)
    lon_sorted = np.sort(lon_values)

    lat_bounds = infer_bounds_from_centers(lat_sorted)
    lon_bounds = infer_bounds_from_centers(lon_sorted)

    lat_bounds_rad = np.deg2rad(lat_bounds)
    lon_bounds_rad = np.deg2rad(lon_bounds)

    dlon = np.diff(lon_bounds_rad)
    sin_lat_diff = np.diff(np.sin(lat_bounds_rad))

    area_sorted = (radius ** 2) * sin_lat_diff[:, None] * dlon[None, :]
    area_sorted = np.abs(area_sorted)

    if not lat_ascending:
        area_sorted = area_sorted[::-1, :]

    if not lon_ascending:
        area_sorted = area_sorted[:, ::-1]

    area = xr.DataArray(
        area_sorted,
        dims=("lat", "lon"),
        coords={
            "lat": lat_values,
            "lon": lon_values,
        },
        name="cell_area"
    )

    area.attrs["units"] = "m2"

    return area


def area_weighted_mean(da):
    area = calculate_latlon_cell_area(da.lat, da.lon)
    valid = da.notnull()

    weighted_sum = (da * area).where(valid).sum(dim=("lat", "lon"), skipna=True)
    weight_sum = area.where(valid).sum(dim=("lat", "lon"), skipna=True)

    out = weighted_sum / weight_sum
    out.name = da.name

    return out


def to_monthly_area_series(da):
    s = area_weighted_mean(da).to_series()
    s = ensure_month_start_index(s)
    return s


# ============================================================
# 9. DOMAIN FUNCTIONS
# ============================================================

def get_historical_domain(*arrays):
    lat_min = max([float(da.lat.min()) for da in arrays])
    lat_max = min([float(da.lat.max()) for da in arrays])
    lon_min = max([float(da.lon.min()) for da in arrays])
    lon_max = min([float(da.lon.max()) for da in arrays])

    return lat_min, lat_max, lon_min, lon_max


def clip_to_domain(da, domain):
    lat_min, lat_max, lon_min, lon_max = domain

    da = da.sortby("lat")
    da = da.sortby("lon")

    return da.sel(
        lat=slice(lat_min, lat_max),
        lon=slice(lon_min, lon_max)
    )


# ============================================================
# 10. FUTURE TOTAL ET READER
# ============================================================

def open_future_total_et(model_name, scenario, hist_domain):
    scen_dir = os.path.join(FUTURE_BASE_DIR, model_name, scenario)
    total_path = os.path.join(scen_dir, FUTURE_TOTAL_ET_FILE)

    if not os.path.exists(total_path):
        raise FileNotFoundError(total_path)

    ds = xr.open_dataset(total_path)

    candidate_vars = [
        v for v in ds.data_vars
        if str(v).lower() not in ["spatial_ref", "crs"]
    ]

    if FUTURE_TOTAL_ET_VAR in candidate_vars:
        var_name = FUTURE_TOTAL_ET_VAR

    elif len(candidate_vars) == 1:
        var_name = candidate_vars[0]

    else:
        raise ValueError(
            f"Cannot select future total ET variable from {total_path}. "
            f"Candidates: {candidate_vars}"
        )

    da = ds[var_name]

    if "time" not in da.dims:
        raise ValueError(f"No time dimension found in {total_path}")

    da["time"] = pd.to_datetime(da.time.values)

    da = standardize_xy_to_latlon(da)

    for d in ["time", "lat", "lon"]:
        if d not in da.dims:
            raise ValueError(f"{model_name}/{scenario}: dimension '{d}' not found.")

    da = da.astype("float64")
    da = da.where(np.isfinite(da))

    da = clip_to_domain(da, hist_domain)

    if FUTURE_CONVERT_M_TO_MM:
        da = da * 1000.0

    if MASK_NEGATIVE_FUTURE_ET:
        da = da.where(da >= 0)

    da.name = f"{model_name}_{scenario}_totalET"

    return da


# ============================================================
# 11. BIAS CORRECTION
# ============================================================

def bias_correct_future_series(future_series, reference_series):
    """
    Multiplicative mean-ratio bias correction over overlap period.
    """

    df = pd.concat([future_series, reference_series], axis=1).dropna()
    df.columns = ["future", "reference"]

    n_overlap = len(df)

    if n_overlap < 12:
        return future_series, np.nan, n_overlap

    future_mean = df["future"].mean()
    reference_mean = df["reference"].mean()

    if not np.isfinite(future_mean) or future_mean == 0:
        return future_series, np.nan, n_overlap

    factor = reference_mean / future_mean

    corrected = future_series * factor
    corrected.name = future_series.name

    return corrected, factor, n_overlap


# ============================================================
# 12. MONTHLY EDGE CORRECTIONS
# ============================================================

def correct_historical_monthly_edges(historical_monthly_df):
    df = historical_monthly_df.copy()
    records = []

    if not CORRECT_MONTHLY_EDGE_VALUES:
        return df, records

    for col in HISTORICAL_FIXED_COLUMNS:
        if col in df.columns:
            corrected_series, original_value, replacement_value, status = replace_target_month_with_mean(
                df[col],
                target_date=HISTORICAL_TARGET_DATE,
                source_dates=HISTORICAL_SOURCE_DATES
            )

            df[col] = corrected_series

            final_value = (
                df[col].loc[pd.Timestamp(HISTORICAL_TARGET_DATE)]
                if pd.Timestamp(HISTORICAL_TARGET_DATE) in df.index
                else np.nan
            )

            records.append({
                "stage": "monthly_before_annual_mean",
                "series": col,
                "target_date": HISTORICAL_TARGET_DATE,
                "source_dates": ", ".join(HISTORICAL_SOURCE_DATES),
                "method": "target_equals_mean_of_sources",
                "original_value": original_value,
                "replacement_value": replacement_value,
                "final_value": final_value,
                "status": status,
            })

    return df, records


def correct_future_monthly_edges(future_monthly_df, scenario):
    df = future_monthly_df.copy()
    records = []

    if not CORRECT_MONTHLY_EDGE_VALUES:
        return df, records

    for col in df.columns:

        corrected_series, original_value, replacement_value, status = replace_target_month_with_mean(
            df[col],
            target_date=FUTURE_FIRST_TARGET_DATE,
            source_dates=FUTURE_FIRST_SOURCE_DATES
        )

        df[col] = corrected_series

        final_value = (
            df[col].loc[pd.Timestamp(FUTURE_FIRST_TARGET_DATE)]
            if pd.Timestamp(FUTURE_FIRST_TARGET_DATE) in df.index
            else np.nan
        )

        records.append({
            "stage": "monthly_before_annual_mean",
            "scenario": scenario,
            "model": col,
            "target_date": FUTURE_FIRST_TARGET_DATE,
            "source_dates": ", ".join(FUTURE_FIRST_SOURCE_DATES),
            "method": "target_equals_mean_of_sources",
            "original_value": original_value,
            "replacement_value": replacement_value,
            "final_value": final_value,
            "status": status,
        })

        corrected_series, original_value, replacement_value, status = replace_target_month_with_mean(
            df[col],
            target_date=FUTURE_LAST_TARGET_DATE,
            source_dates=FUTURE_LAST_SOURCE_DATES
        )

        df[col] = corrected_series

        final_value = (
            df[col].loc[pd.Timestamp(FUTURE_LAST_TARGET_DATE)]
            if pd.Timestamp(FUTURE_LAST_TARGET_DATE) in df.index
            else np.nan
        )

        records.append({
            "stage": "monthly_before_annual_mean",
            "scenario": scenario,
            "model": col,
            "target_date": FUTURE_LAST_TARGET_DATE,
            "source_dates": ", ".join(FUTURE_LAST_SOURCE_DATES),
            "method": "target_equals_mean_of_sources",
            "original_value": original_value,
            "replacement_value": replacement_value,
            "final_value": final_value,
            "status": status,
        })

    return df, records


# ============================================================
# 13. PERIOD LINES AND LABELS
# ============================================================

PERIOD_LINES = [
    pd.Timestamp("2025-01-01"),
    pd.Timestamp("2050-01-01"),
    pd.Timestamp("2075-01-01"),
    pd.Timestamp("2100-01-01"),
]

PERIOD_LABELS = [
    ("Historical", pd.Timestamp("2000-01-01"), pd.Timestamp("2019-12-01")),
    ("Near future", pd.Timestamp("2025-01-01"), pd.Timestamp("2049-12-01")),
    ("Mid-century", pd.Timestamp("2050-01-01"), pd.Timestamp("2074-12-01")),
    ("Far future", pd.Timestamp("2075-01-01"), pd.Timestamp("2099-12-01")),
]


# ============================================================
# 14. PLOTTING
# ============================================================

def plot_scenario_annual_timeseries(
    scenario_label,
    historical_annual_df,
    future_annual_df,
    ensemble_annual_series,
    out_png
):
    fig, ax = plt.subplots(figsize=(FIG_WIDTH, FIG_HEIGHT))

    hist_colors = {
        "CWatM": "black",
        "GLDAS": "tab:blue",
        "SMAP": "tab:green",
    }

    model_colors = {
        "GFDL": "tab:blue",
        "IPSL": "tab:orange",
        "MPI": "tab:green",
        "MRI": "tab:purple",
        "UKESM": "tab:brown",
    }

    for col in historical_annual_df.columns:
        ax.plot(
            historical_annual_df.index,
            historical_annual_df[col],
            linewidth=LINE_WIDTH_HISTORICAL,
            marker="o",
            markersize=4,
            label=f"Historical {col}",
            color=hist_colors.get(col, None),
            zorder=3
        )

    for col in future_annual_df.columns:
        ax.plot(
            future_annual_df.index,
            future_annual_df[col],
            linewidth=LINE_WIDTH_FUTURE_MODEL,
            alpha=ALPHA_FUTURE_MODEL,
            marker="o",
            markersize=3,
            color=model_colors.get(col, None),
            label=f"Future {col}",
            zorder=4
        )

    ax.plot(
        ensemble_annual_series.index,
        ensemble_annual_series.values,
        linewidth=LINE_WIDTH_ENSEMBLE,
        marker="o",
        markersize=4,
        color="red",
        alpha=0.95,
        label="Future ensemble mean",
        zorder=5
    )

    for dt in PERIOD_LINES:
        ax.axvline(
            dt,
            color="gray",
            linestyle="--",
            linewidth=1.2,
            alpha=0.50,
            zorder=1
        )

    ax.set_xlim(
        pd.Timestamp(PLOT_START_DATE),
        pd.Timestamp(PLOT_END_DATE)
    )

    ymin, ymax = ax.get_ylim()

    for label, start, end in PERIOD_LABELS:
        mid = start + (end - start) / 2

        ax.text(
            mid,
            ymax - 0.035 * (ymax - ymin),
            label,
            ha="center",
            va="top",
            fontsize=PERIOD_LABEL_FONT_SIZE,
            color="gray"
        )

    title_extra = "bias-corrected to historical CWatM" if APPLY_BIAS_CORRECTION else "raw future"

    ax.set_title(
        f"Historical and Future Annual Mean ET Time Series - {scenario_label}\n"
        f"Future mode: totalET only; {title_extra}",
        fontsize=TITLE_FONT_SIZE
    )

    ax.set_xlabel("Time", fontsize=AXIS_LABEL_FONT_SIZE)
    ax.set_ylabel(get_y_label(), fontsize=AXIS_LABEL_FONT_SIZE)

    ax.tick_params(axis="both", labelsize=TICK_FONT_SIZE)
    ax.grid(True, alpha=0.25)
    ax.legend(ncol=3, fontsize=LEGEND_FONT_SIZE)

    plt.tight_layout()
    plt.savefig(out_png, dpi=300)
    plt.close()


# ============================================================
# 15. MAIN WORKFLOW
# ============================================================

print("\nLoading historical ET datasets...")

cwatm = open_historical_et(CWATM_FILE, "CWatM")
gldas = open_historical_et(GLDAS_FILE, "GLDAS")
smap = open_historical_et(SMAP_FILE, "SMAP")

hist_domain = get_historical_domain(cwatm, gldas, smap)

print("\nHistorical domain intersection:")
print(f"lat: {hist_domain[0]:.3f} to {hist_domain[1]:.3f}")
print(f"lon: {hist_domain[2]:.3f} to {hist_domain[3]:.3f}")

cwatm_clip = clip_to_domain(cwatm, hist_domain)
gldas_clip = clip_to_domain(gldas, hist_domain)
smap_clip = clip_to_domain(smap, hist_domain)

historical_monthly_df = pd.concat(
    [
        to_monthly_area_series(cwatm_clip).rename("CWatM"),
        to_monthly_area_series(gldas_clip).rename("GLDAS"),
        to_monthly_area_series(smap_clip).rename("SMAP"),
    ],
    axis=1
)

historical_monthly_df = historical_monthly_df.sort_index()

historical_monthly_df = historical_monthly_df.loc[
    (historical_monthly_df.index >= pd.Timestamp(HISTORICAL_START_DATE)) &
    (historical_monthly_df.index <= pd.Timestamp(HISTORICAL_END_DATE))
]

historical_monthly_corrected_df, historical_edge_records = correct_historical_monthly_edges(
    historical_monthly_df
)

historical_annual_df = monthly_to_annual_mean(historical_monthly_corrected_df)

if SAVE_MONTHLY_TABLES:
    historical_monthly_df.to_csv(
        os.path.join(TABLE_DIR, "historical_monthly_raw_final.csv"),
        encoding="utf-8-sig"
    )

    historical_monthly_corrected_df.to_csv(
        os.path.join(TABLE_DIR, "historical_monthly_corrected_final.csv"),
        encoding="utf-8-sig"
    )

if SAVE_ANNUAL_TABLES:
    historical_annual_df.to_csv(
        os.path.join(TABLE_DIR, "historical_annual_mean_final.csv"),
        encoding="utf-8-sig"
    )


# ============================================================
# 16. FUTURE WORKFLOW
# ============================================================

all_future_summary = []
bias_records = []
future_edge_records_all = []
combined_future_monthly_records = []
combined_future_annual_records = []

for scenario, scenario_label in SCENARIOS.items():

    print("\n" + "=" * 80)
    print(f"Processing scenario: {scenario} - {scenario_label}")
    print("=" * 80)

    future_monthly_series_list = []

    for model in CLIMATE_MODELS:

        print(f"\nProcessing model: {model}, scenario: {scenario}")

        try:
            future_da = open_future_total_et(
                model_name=model,
                scenario=scenario,
                hist_domain=hist_domain
            )

            future_series_raw = to_monthly_area_series(future_da)
            future_series_raw.name = model

            if APPLY_BIAS_CORRECTION:
                future_series_used, factor, n_overlap = bias_correct_future_series(
                    future_series=future_series_raw,
                    reference_series=historical_monthly_corrected_df[BIAS_REFERENCE]
                )
            else:
                future_series_used = future_series_raw
                factor = np.nan
                n_overlap = 0

            future_series_used = future_series_used.loc[
                (future_series_used.index >= pd.Timestamp(FUTURE_START_DATE)) &
                (future_series_used.index <= pd.Timestamp(FUTURE_END_DATE))
            ].copy()

            future_series_used.name = model
            future_monthly_series_list.append(future_series_used)

            bias_records.append({
                "model": model,
                "scenario": scenario,
                "bias_reference": BIAS_REFERENCE,
                "bias_method": BIAS_METHOD if APPLY_BIAS_CORRECTION else "none",
                "bias_factor": factor,
                "n_overlap_months": n_overlap,
            })

        except Exception as e:
            print(f"ERROR for {model} / {scenario}: {e}")

    if len(future_monthly_series_list) == 0:
        print(f"No valid future data for scenario {scenario}. Plot skipped.")
        continue

    future_monthly_df = pd.concat(future_monthly_series_list, axis=1)
    future_monthly_df = future_monthly_df.sort_index()

    available_cols = [m for m in CLIMATE_MODELS if m in future_monthly_df.columns]
    future_monthly_df = future_monthly_df[available_cols]

    future_monthly_corrected_df, future_edge_records = correct_future_monthly_edges(
        future_monthly_df,
        scenario=scenario
    )

    future_edge_records_all.extend(future_edge_records)

    future_annual_df = monthly_to_annual_mean(future_monthly_corrected_df)

    ensemble_annual_series = future_annual_df.mean(axis=1, skipna=True)
    ensemble_annual_series.name = "Future_Ensemble_Mean"

    for model in future_annual_df.columns:
        all_future_summary.append({
            "model": model,
            "scenario": scenario,
            "scenario_label": scenario_label,
            "mode": "totalET_only",
            "bias_corrected": APPLY_BIAS_CORRECTION,
            "start": future_annual_df[model].dropna().index.min(),
            "end": future_annual_df[model].dropna().index.max(),
            "n_years": future_annual_df[model].dropna().shape[0],
            "annual_mean": future_annual_df[model].mean(),
            "annual_min": future_annual_df[model].min(),
            "annual_max": future_annual_df[model].max(),
            "domain_lat_min": hist_domain[0],
            "domain_lat_max": hist_domain[1],
            "domain_lon_min": hist_domain[2],
            "domain_lon_max": hist_domain[3],
        })

    if SAVE_MONTHLY_TABLES:
        future_monthly_df.to_csv(
            os.path.join(TABLE_DIR, f"future_monthly_raw_scenario_{scenario}_final.csv"),
            encoding="utf-8-sig"
        )

        future_monthly_corrected_df.to_csv(
            os.path.join(TABLE_DIR, f"future_monthly_corrected_scenario_{scenario}_final.csv"),
            encoding="utf-8-sig"
        )

    if SAVE_ANNUAL_TABLES:
        future_annual_df.to_csv(
            os.path.join(TABLE_DIR, f"future_annual_mean_scenario_{scenario}_final.csv"),
            encoding="utf-8-sig"
        )

        ensemble_annual_series.to_frame().to_csv(
            os.path.join(TABLE_DIR, f"future_ensemble_annual_mean_scenario_{scenario}_final.csv"),
            encoding="utf-8-sig"
        )

    temp_monthly = future_monthly_corrected_df.copy()
    temp_monthly["scenario"] = scenario
    temp_monthly["scenario_label"] = scenario_label
    temp_monthly = temp_monthly.reset_index().rename(columns={"index": "time"})
    combined_future_monthly_records.append(temp_monthly)

    temp_annual = future_annual_df.copy()
    temp_annual["Ensemble_Mean"] = ensemble_annual_series
    temp_annual["scenario"] = scenario
    temp_annual["scenario_label"] = scenario_label
    temp_annual = temp_annual.reset_index().rename(columns={"index": "time"})
    combined_future_annual_records.append(temp_annual)

    out_png = os.path.join(
        FIG_DIR,
        f"annual_historical_future_ET_scenario_{scenario}_final.png"
    )

    plot_scenario_annual_timeseries(
        scenario_label=scenario_label,
        historical_annual_df=historical_annual_df,
        future_annual_df=future_annual_df,
        ensemble_annual_series=ensemble_annual_series,
        out_png=out_png
    )


# ============================================================
# 17. SAVE SUMMARY TABLES
# ============================================================

summary_df = pd.DataFrame(all_future_summary)
summary_csv = os.path.join(TABLE_DIR, "future_models_annual_summary_final.csv")
summary_df.to_csv(summary_csv, index=False, encoding="utf-8-sig")

bias_df = pd.DataFrame(bias_records)
bias_csv = os.path.join(TABLE_DIR, "future_bias_correction_factors_final.csv")
bias_df.to_csv(bias_csv, index=False, encoding="utf-8-sig")

historical_edge_df = pd.DataFrame(historical_edge_records)
historical_edge_csv = os.path.join(TABLE_DIR, "historical_monthly_edge_corrections_final.csv")
historical_edge_df.to_csv(historical_edge_csv, index=False, encoding="utf-8-sig")

future_edge_df = pd.DataFrame(future_edge_records_all)
future_edge_csv = os.path.join(TABLE_DIR, "future_monthly_edge_corrections_final.csv")
future_edge_df.to_csv(future_edge_csv, index=False, encoding="utf-8-sig")

if len(combined_future_monthly_records) > 0:
    combined_future_monthly_df = pd.concat(combined_future_monthly_records, axis=0)
    combined_future_monthly_csv = os.path.join(
        TABLE_DIR,
        "future_monthly_corrected_all_models_all_scenarios_final.csv"
    )
    combined_future_monthly_df.to_csv(
        combined_future_monthly_csv,
        index=False,
        encoding="utf-8-sig"
    )
else:
    combined_future_monthly_csv = None

if len(combined_future_annual_records) > 0:
    combined_future_annual_df = pd.concat(combined_future_annual_records, axis=0)
    combined_future_annual_csv = os.path.join(
        TABLE_DIR,
        "future_annual_mean_all_models_all_scenarios_final.csv"
    )
    combined_future_annual_df.to_csv(
        combined_future_annual_csv,
        index=False,
        encoding="utf-8-sig"
    )
else:
    combined_future_annual_csv = None

config_df = pd.DataFrame([
    {"parameter": "analysis_type", "value": "annual_mean_no_moving_window"},
    {"parameter": "future_mode", "value": "totalET_only"},
    {"parameter": "base_dir", "value": BASE_DIR},
    {"parameter": "future_base_dir", "value": FUTURE_BASE_DIR},
    {"parameter": "climate_models", "value": ", ".join(CLIMATE_MODELS)},
    {"parameter": "scenarios", "value": ", ".join(SCENARIOS.keys())},
    {"parameter": "plot_start_date", "value": PLOT_START_DATE},
    {"parameter": "plot_end_date", "value": PLOT_END_DATE},
    {"parameter": "historical_start_date", "value": HISTORICAL_START_DATE},
    {"parameter": "historical_end_date", "value": HISTORICAL_END_DATE},
    {"parameter": "future_start_date", "value": FUTURE_START_DATE},
    {"parameter": "future_end_date", "value": FUTURE_END_DATE},
    {"parameter": "apply_bias_correction", "value": APPLY_BIAS_CORRECTION},
    {"parameter": "bias_reference", "value": BIAS_REFERENCE},
    {"parameter": "bias_method", "value": BIAS_METHOD},
    {"parameter": "correct_monthly_edge_values_before_annual_mean", "value": CORRECT_MONTHLY_EDGE_VALUES},
    {"parameter": "historical_fixed_columns", "value": ", ".join(HISTORICAL_FIXED_COLUMNS)},
    {"parameter": "historical_target_date", "value": HISTORICAL_TARGET_DATE},
    {"parameter": "historical_source_dates", "value": ", ".join(HISTORICAL_SOURCE_DATES)},
    {"parameter": "future_first_target_date", "value": FUTURE_FIRST_TARGET_DATE},
    {"parameter": "future_first_source_dates", "value": ", ".join(FUTURE_FIRST_SOURCE_DATES)},
    {"parameter": "future_last_target_date", "value": FUTURE_LAST_TARGET_DATE},
    {"parameter": "future_last_source_dates", "value": ", ".join(FUTURE_LAST_SOURCE_DATES)},
    {"parameter": "domain_lat_min", "value": hist_domain[0]},
    {"parameter": "domain_lat_max", "value": hist_domain[1]},
    {"parameter": "domain_lon_min", "value": hist_domain[2]},
    {"parameter": "domain_lon_max", "value": hist_domain[3]},
])

config_csv = os.path.join(TABLE_DIR, "run_configuration_annual_final.csv")
config_df.to_csv(config_csv, index=False, encoding="utf-8-sig")


# ============================================================
# 18. FINAL REPORT
# ============================================================

print("\n" + "=" * 80)
print("DONE")
print("=" * 80)

print("\nMain output folder:")
print(OUT_DIR)

print("\nMain tables:")
print(summary_csv)
print(bias_csv)
print(historical_edge_csv)
print(future_edge_csv)
print(config_csv)

if combined_future_monthly_csv is not None:
    print(combined_future_monthly_csv)

if combined_future_annual_csv is not None:
    print(combined_future_annual_csv)

print("\nFigures:")
for scenario in SCENARIOS:
    print(
        os.path.join(
            FIG_DIR,
            f"annual_historical_future_ET_scenario_{scenario}_final.png"
        )
    )

print("\nNotes:")
print("This version uses annual mean ET, not a moving average.")
print("Monthly edge corrections are applied before annual averaging.")
print("Historical annual period: 2000-2019.")
print("Future annual period: 2020-2099.")
print("The x-axis boundary is 2100-01.")


Loading historical ET datasets...

Opening historical dataset: CWatM
E:\ET_Article_With_MM_SA\Future data ST\ET_CWatM_mm_day.nc
Variable: ET_final_mm_day
Dims: ('time', 'lat', 'lon')
Shape: (468, 13, 14)
Time: 1981-01 to 2019-12
Mean: 1.102419

Opening historical dataset: GLDAS
E:\ET_Article_With_MM_SA\Future data ST\GLDAS_ET_2000_2019.nc
Variable: __xarray_dataarray_variable__
Dims: ('time', 'lat', 'lon')
Shape: (240, 13, 14)
Time: 2000-01 to 2019-12
Mean: 0.793585

Opening historical dataset: SMAP
E:\ET_Article_With_MM_SA\Future data ST\SMAP_ET_2000_2019.nc
Variable: __xarray_dataarray_variable__
Dims: ('time', 'lat', 'lon')
Shape: (58, 13, 14)
Time: 2015-03 to 2019-12
Mean: 1.057844

Historical domain intersection:
lat: 30.750 to 36.750
lon: 45.250 to 51.750

Processing scenario: 126 - Optimistic / SSP126

Processing model: GFDL, scenario: 126

Processing model: IPSL, scenario: 126

Processing model: MPI, scenario: 126

Processing model: MRI, scenario: 126

Processing model: UKESM,

In [3]:
# -*- coding: utf-8 -*-

"""
Annual Historical-Future ET Time Series
with historical GLDAS and SMAP bias correction

Historical datasets:
    CWatM
    GLDAS
    SMAP

Historical bias correction:
    Reference: CWatM
    Method: additive mean bias

    Bias = mean(Product - CWatM)
    Corrected Product = Product - Bias

Future climate models:
    GFDL
    IPSL
    MPI
    MRI
    UKESM

Scenarios:
    126 = Optimistic / SSP126
    370 = Medium / SSP370
    585 = Pessimistic / SSP585

Main features:
    - Monthly spatial area-weighted ET
    - Monthly edge correction
    - GLDAS and SMAP additive bias correction relative to CWatM
    - Conversion from monthly to annual mean
    - Original historical lines: solid
    - Bias-corrected GLDAS/SMAP lines: dashed, same color
    - Legend in lower-right corner
    - Future climate-model bias correction relative to CWatM
"""

import os
import re
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt


# ============================================================
# 1. PATH SETTINGS
# ============================================================

BASE_DIR = os.getcwd()

CWATM_FILE = os.path.join(BASE_DIR, "ET_CWatM_mm_day.nc")
GLDAS_FILE = os.path.join(BASE_DIR, "GLDAS_ET_2000_2019.nc")
SMAP_FILE = os.path.join(BASE_DIR, "SMAP_ET_2000_2019.nc")

FUTURE_BASE_DIR = BASE_DIR


# ============================================================
# 2. MODELS AND SCENARIOS
# ============================================================

CLIMATE_MODELS = [
    "GFDL",
    "IPSL",
    "MPI",
    "MRI",
    "UKESM",
]

SCENARIOS = {
    "126": "Optimistic / SSP126",
    "370": "Medium / SSP370",
    "585": "Pessimistic / SSP585",
}

FUTURE_TOTAL_ET_FILE = "totalET_monthavg.nc"
FUTURE_TOTAL_ET_VAR = "totalET_monthavg"


# ============================================================
# 3. OUTPUT SETTINGS
# ============================================================

OUT_DIR = os.path.join(
    BASE_DIR,
    "outputs_historical_future_ET_annual_mean_historical_bias_corrected"
)

TABLE_DIR = os.path.join(OUT_DIR, "tables")
FIG_DIR = os.path.join(OUT_DIR, "figures")

for directory in [OUT_DIR, TABLE_DIR, FIG_DIR]:
    os.makedirs(directory, exist_ok=True)


# ============================================================
# 4. ANALYSIS SETTINGS
# ============================================================

PLOT_START_DATE = "2000-01-01"
PLOT_END_DATE = "2100-01-01"

HISTORICAL_START_DATE = "2000-01-01"
HISTORICAL_END_DATE = "2019-12-01"

FUTURE_START_DATE = "2020-01-01"
FUTURE_END_DATE = "2099-12-01"

HISTORICAL_UNIT = "mm/day"

FUTURE_CONVERT_M_TO_MM = True

MASK_NEGATIVE_HISTORICAL_ET = True
MASK_NEGATIVE_FUTURE_ET = True

# Future climate-model bias correction
APPLY_FUTURE_BIAS_CORRECTION = True
FUTURE_BIAS_REFERENCE = "CWatM"
FUTURE_BIAS_METHOD = "multiplicative_mean_ratio"

# Historical GLDAS/SMAP bias correction
APPLY_HISTORICAL_PRODUCT_BIAS_CORRECTION = True
HISTORICAL_BIAS_REFERENCE = "CWatM"

# چون اختلاف نسبتاً ثابت است، اصلاح افزایشی انتخاب شده است.
HISTORICAL_BIAS_METHOD = "additive_mean_bias"

HISTORICAL_PRODUCTS_TO_CORRECT = [
    "GLDAS",
    "SMAP",
]

# حداقل ماه مشترک لازم برای محاسبه بایاس
MIN_OVERLAP_MONTHS_FOR_HISTORICAL_BIAS = 12

SAVE_MONTHLY_TABLES = True
SAVE_ANNUAL_TABLES = True

# Monthly edge corrections before annual averaging
CORRECT_MONTHLY_EDGE_VALUES = True

HISTORICAL_FIXED_COLUMNS = [
    "CWatM",
    "GLDAS",
]

HISTORICAL_TARGET_DATE = "2019-12-01"

HISTORICAL_SOURCE_DATES = [
    "2019-09-01",
    "2019-10-01",
    "2019-11-01",
]

FUTURE_FIRST_TARGET_DATE = "2020-01-01"

FUTURE_FIRST_SOURCE_DATES = [
    "2020-02-01",
    "2020-03-01",
    "2020-04-01",
]

FUTURE_LAST_TARGET_DATE = "2099-12-01"

FUTURE_LAST_SOURCE_DATES = [
    "2099-09-01",
    "2099-10-01",
    "2099-11-01",
]


# ============================================================
# 5. FONT AND FIGURE SETTINGS
# ============================================================

FONT_FAMILY = "DejaVu Sans"

TITLE_FONT_SIZE = 20
AXIS_LABEL_FONT_SIZE = 18
TICK_FONT_SIZE = 15
LEGEND_FONT_SIZE = 11
PERIOD_LABEL_FONT_SIZE = 14

FIG_WIDTH = 18
FIG_HEIGHT = 7

LINE_WIDTH_HISTORICAL = 1.2
LINE_WIDTH_HISTORICAL_CORRECTED = 1.8
LINE_WIDTH_FUTURE_MODEL = 1.0
LINE_WIDTH_ENSEMBLE = 2.5

MARKER_SIZE_HISTORICAL = 4
MARKER_SIZE_HISTORICAL_CORRECTED = 3
MARKER_SIZE_FUTURE = 3
MARKER_SIZE_ENSEMBLE = 4

ALPHA_FUTURE_MODEL = 0.45

plt.rcParams["font.family"] = FONT_FAMILY
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.dpi"] = 150
plt.rcParams["axes.titlesize"] = TITLE_FONT_SIZE
plt.rcParams["axes.labelsize"] = AXIS_LABEL_FONT_SIZE
plt.rcParams["xtick.labelsize"] = TICK_FONT_SIZE
plt.rcParams["ytick.labelsize"] = TICK_FONT_SIZE
plt.rcParams["legend.fontsize"] = LEGEND_FONT_SIZE


# ============================================================
# 6. BASIC FUNCTIONS
# ============================================================

def ensure_month_start_index(series_or_df):
    obj = series_or_df.copy()

    obj.index = pd.to_datetime(obj.index)
    obj.index = obj.index.to_period("M").to_timestamp()

    obj = obj.sort_index()

    return obj


def monthly_to_annual_mean(series_or_df):
    """
    Convert monthly data to annual mean.

    Annual timestamps are assigned to January 1.
    """

    obj = series_or_df.copy()
    obj.index = pd.to_datetime(obj.index)

    return obj.resample("YS").mean()


def get_y_label():
    return "Annual mean daily ET (mm/day)"


def replace_target_month_with_mean(
    series,
    target_date,
    source_dates
):
    """
    Replace one target month with the mean of selected source months.
    """

    s = series.copy()

    target_date = (
        pd.Timestamp(target_date)
        .to_period("M")
        .to_timestamp()
    )

    source_dates = [
        pd.Timestamp(date).to_period("M").to_timestamp()
        for date in source_dates
    ]

    if target_date not in s.index:
        return s, np.nan, np.nan, "target_date_not_found"

    available_source_dates = [
        date for date in source_dates
        if date in s.index
    ]

    if len(available_source_dates) == 0:
        return (
            s,
            s.loc[target_date],
            np.nan,
            "no_source_dates_found"
        )

    source_values = s.loc[available_source_dates].dropna()

    if len(source_values) == 0:
        return (
            s,
            s.loc[target_date],
            np.nan,
            "source_values_all_nan"
        )

    original_value = s.loc[target_date]
    replacement_value = source_values.mean()

    s.loc[target_date] = replacement_value

    return (
        s,
        original_value,
        replacement_value,
        "replaced"
    )


# ============================================================
# 7. HISTORICAL DATA READER
# ============================================================

def extract_dates_from_long_name(long_name_attr):
    if isinstance(long_name_attr, str):
        long_names = [long_name_attr]
    else:
        long_names = list(long_name_attr)

    dates = []

    for item in long_names:
        item = str(item)

        match = re.search(r"(\d{4})_(\d{2})", item)

        if match is None:
            raise ValueError(
                f"Cannot extract date from long_name item: {item}"
            )

        dates.append(
            pd.Timestamp(
                year=int(match.group(1)),
                month=int(match.group(2)),
                day=1
            )
        )

    return pd.DatetimeIndex(dates)


def standardize_xy_to_latlon(da):
    rename_dict = {}

    for name in list(da.dims) + list(da.coords):
        name_lower = str(name).lower()

        if name_lower in ["x", "longitude"]:
            rename_dict[name] = "lon"

        elif name_lower in ["y", "latitude"]:
            rename_dict[name] = "lat"

    if rename_dict:
        da = da.rename(rename_dict)

    if "lat" in da.coords:
        da = da.sortby("lat")

    if "lon" in da.coords:
        da = da.sortby("lon")

    return da


def open_historical_et(path, dataset_name):
    print("\n" + "=" * 80)
    print(f"Opening historical dataset: {dataset_name}")
    print(path)
    print("=" * 80)

    if not os.path.exists(path):
        raise FileNotFoundError(path)

    ds = xr.open_dataset(path)

    candidate_vars = [
        variable
        for variable in ds.data_vars
        if str(variable).lower() not in ["spatial_ref", "crs"]
    ]

    if len(candidate_vars) == 0:
        raise ValueError(
            f"No variable found in {dataset_name}"
        )

    if "ET_final_mm_day" in candidate_vars:
        var_name = "ET_final_mm_day"

    elif "__xarray_dataarray_variable__" in candidate_vars:
        var_name = "__xarray_dataarray_variable__"

    else:
        var_name = candidate_vars[0]

    da = ds[var_name]

    if "time" in da.dims:
        da["time"] = pd.to_datetime(da.time.values)

    elif "band" in da.dims:
        long_name = da.attrs.get("long_name", None)

        if long_name is None:
            raise ValueError(
                f"{dataset_name} has band dimension but no long_name."
            )

        dates = extract_dates_from_long_name(long_name)

        if len(dates) != da.sizes["band"]:
            raise ValueError(
                f"{dataset_name}: extracted dates do not match band size."
            )

        da = da.rename({"band": "time"})
        da = da.assign_coords(time=dates)

    else:
        raise ValueError(
            f"No time or band dimension found in {dataset_name}"
        )

    da = standardize_xy_to_latlon(da)

    for dimension in ["time", "lat", "lon"]:
        if dimension not in da.dims:
            raise ValueError(
                f"{dataset_name}: required dimension "
                f"'{dimension}' not found."
            )

    da = da.astype("float64")
    da = da.where(np.isfinite(da))

    if MASK_NEGATIVE_HISTORICAL_ET:
        da = da.where(da >= 0)

    da.name = dataset_name
    da.attrs["unit"] = HISTORICAL_UNIT

    print(f"Variable: {var_name}")
    print(f"Dims: {da.dims}")
    print(f"Shape: {da.shape}")

    print(
        "Time:",
        pd.to_datetime(da.time.values[0]).strftime("%Y-%m"),
        "to",
        pd.to_datetime(da.time.values[-1]).strftime("%Y-%m")
    )

    print(
        f"Mean: {float(da.mean(skipna=True)):.6f}"
    )

    return da


# ============================================================
# 8. CELL AREA AND AREA-WEIGHTED MEAN
# ============================================================

def infer_bounds_from_centers(values):
    values = np.asarray(values, dtype=float)

    if len(values) < 2:
        raise ValueError(
            "At least two coordinate values are needed."
        )

    values_sorted = np.sort(values)

    midpoint = (
        values_sorted[:-1] +
        values_sorted[1:]
    ) / 2.0

    first = (
        values_sorted[0] -
        (midpoint[0] - values_sorted[0])
    )

    last = (
        values_sorted[-1] +
        (values_sorted[-1] - midpoint[-1])
    )

    return np.concatenate([
        [first],
        midpoint,
        [last]
    ])


def calculate_latlon_cell_area(
    lat,
    lon,
    radius=6371000.0
):
    lat_values = np.asarray(
        lat.values if hasattr(lat, "values") else lat,
        dtype=float
    )

    lon_values = np.asarray(
        lon.values if hasattr(lon, "values") else lon,
        dtype=float
    )

    lat_ascending = np.all(np.diff(lat_values) > 0)
    lon_ascending = np.all(np.diff(lon_values) > 0)

    lat_sorted = np.sort(lat_values)
    lon_sorted = np.sort(lon_values)

    lat_bounds = infer_bounds_from_centers(lat_sorted)
    lon_bounds = infer_bounds_from_centers(lon_sorted)

    lat_bounds_rad = np.deg2rad(lat_bounds)
    lon_bounds_rad = np.deg2rad(lon_bounds)

    dlon = np.diff(lon_bounds_rad)
    sin_lat_diff = np.diff(np.sin(lat_bounds_rad))

    area_sorted = (
        radius ** 2
    ) * sin_lat_diff[:, None] * dlon[None, :]

    area_sorted = np.abs(area_sorted)

    if not lat_ascending:
        area_sorted = area_sorted[::-1, :]

    if not lon_ascending:
        area_sorted = area_sorted[:, ::-1]

    area = xr.DataArray(
        area_sorted,
        dims=("lat", "lon"),
        coords={
            "lat": lat_values,
            "lon": lon_values,
        },
        name="cell_area"
    )

    area.attrs["units"] = "m2"

    return area


def area_weighted_mean(da):
    area = calculate_latlon_cell_area(
        da.lat,
        da.lon
    )

    valid = da.notnull()

    weighted_sum = (
        (da * area)
        .where(valid)
        .sum(
            dim=("lat", "lon"),
            skipna=True
        )
    )

    weight_sum = (
        area
        .where(valid)
        .sum(
            dim=("lat", "lon"),
            skipna=True
        )
    )

    output = weighted_sum / weight_sum
    output.name = da.name

    return output


def to_monthly_area_series(da):
    series = area_weighted_mean(da).to_series()

    series = ensure_month_start_index(series)

    return series


# ============================================================
# 9. DOMAIN FUNCTIONS
# ============================================================

def get_historical_domain(*arrays):
    lat_min = max([
        float(da.lat.min())
        for da in arrays
    ])

    lat_max = min([
        float(da.lat.max())
        for da in arrays
    ])

    lon_min = max([
        float(da.lon.min())
        for da in arrays
    ])

    lon_max = min([
        float(da.lon.max())
        for da in arrays
    ])

    return (
        lat_min,
        lat_max,
        lon_min,
        lon_max
    )


def clip_to_domain(da, domain):
    lat_min, lat_max, lon_min, lon_max = domain

    da = da.sortby("lat")
    da = da.sortby("lon")

    return da.sel(
        lat=slice(lat_min, lat_max),
        lon=slice(lon_min, lon_max)
    )


# ============================================================
# 10. FUTURE TOTAL ET READER
# ============================================================

def open_future_total_et(
    model_name,
    scenario,
    hist_domain
):
    scenario_dir = os.path.join(
        FUTURE_BASE_DIR,
        model_name,
        scenario
    )

    total_path = os.path.join(
        scenario_dir,
        FUTURE_TOTAL_ET_FILE
    )

    if not os.path.exists(total_path):
        raise FileNotFoundError(total_path)

    ds = xr.open_dataset(total_path)

    candidate_vars = [
        variable
        for variable in ds.data_vars
        if str(variable).lower() not in ["spatial_ref", "crs"]
    ]

    if FUTURE_TOTAL_ET_VAR in candidate_vars:
        var_name = FUTURE_TOTAL_ET_VAR

    elif len(candidate_vars) == 1:
        var_name = candidate_vars[0]

    else:
        raise ValueError(
            f"Cannot select future total ET variable from "
            f"{total_path}. Candidates: {candidate_vars}"
        )

    da = ds[var_name]

    if "time" not in da.dims:
        raise ValueError(
            f"No time dimension found in {total_path}"
        )

    da["time"] = pd.to_datetime(da.time.values)

    da = standardize_xy_to_latlon(da)

    for dimension in ["time", "lat", "lon"]:
        if dimension not in da.dims:
            raise ValueError(
                f"{model_name}/{scenario}: dimension "
                f"'{dimension}' not found."
            )

    da = da.astype("float64")
    da = da.where(np.isfinite(da))

    da = clip_to_domain(
        da,
        hist_domain
    )

    if FUTURE_CONVERT_M_TO_MM:
        da = da * 1000.0

    if MASK_NEGATIVE_FUTURE_ET:
        da = da.where(da >= 0)

    da.name = (
        f"{model_name}_{scenario}_totalET"
    )

    return da


# ============================================================
# 11. FUTURE BIAS CORRECTION
# ============================================================

def bias_correct_future_series(
    future_series,
    reference_series
):
    """
    Multiplicative mean-ratio correction for future models.
    """

    overlap_df = pd.concat(
        [
            future_series,
            reference_series
        ],
        axis=1
    ).dropna()

    overlap_df.columns = [
        "future",
        "reference"
    ]

    n_overlap = len(overlap_df)

    if n_overlap < 12:
        return (
            future_series,
            np.nan,
            n_overlap
        )

    future_mean = overlap_df["future"].mean()
    reference_mean = overlap_df["reference"].mean()

    if (
        not np.isfinite(future_mean)
        or future_mean == 0
    ):
        return (
            future_series,
            np.nan,
            n_overlap
        )

    factor = reference_mean / future_mean

    corrected = future_series * factor
    corrected.name = future_series.name

    return (
        corrected,
        factor,
        n_overlap
    )


# ============================================================
# 12. HISTORICAL PRODUCT BIAS CORRECTION
# ============================================================

def calculate_additive_historical_bias(
    product_series,
    reference_series,
    product_name,
    reference_name="CWatM"
):
    """
    Calculate and remove a constant additive bias.

    Bias:
        mean(Product - Reference)

    Correction:
        Product_corrected = Product - Bias
    """

    overlap = pd.concat(
        [
            reference_series.rename("reference"),
            product_series.rename("product")
        ],
        axis=1
    ).dropna()

    n_overlap = len(overlap)

    if n_overlap < MIN_OVERLAP_MONTHS_FOR_HISTORICAL_BIAS:
        corrected = product_series.copy()

        return corrected, {
            "product": product_name,
            "reference": reference_name,
            "method": HISTORICAL_BIAS_METHOD,
            "status": "insufficient_overlap",
            "overlap_start": (
                overlap.index.min()
                if n_overlap > 0
                else pd.NaT
            ),
            "overlap_end": (
                overlap.index.max()
                if n_overlap > 0
                else pd.NaT
            ),
            "n_overlap_months": n_overlap,
            "reference_mean": np.nan,
            "product_mean": np.nan,
            "additive_bias_mm_day": np.nan,
            "relative_bias_percent": np.nan,
            "mae_before": np.nan,
            "mae_after": np.nan,
            "rmse_before": np.nan,
            "rmse_after": np.nan,
        }

    reference_mean = overlap["reference"].mean()
    product_mean = overlap["product"].mean()

    additive_bias = (
        product_mean -
        reference_mean
    )

    if reference_mean != 0:
        relative_bias_percent = (
            additive_bias /
            reference_mean
        ) * 100.0
    else:
        relative_bias_percent = np.nan

    corrected = (
        product_series -
        additive_bias
    )

    corrected.name = (
        f"{product_name}_BiasCorrected"
    )

    corrected_overlap = pd.concat(
        [
            reference_series.rename("reference"),
            corrected.rename("corrected")
        ],
        axis=1
    ).dropna()

    error_before = (
        overlap["product"] -
        overlap["reference"]
    )

    error_after = (
        corrected_overlap["corrected"] -
        corrected_overlap["reference"]
    )

    mae_before = np.mean(
        np.abs(error_before)
    )

    mae_after = np.mean(
        np.abs(error_after)
    )

    rmse_before = np.sqrt(
        np.mean(error_before ** 2)
    )

    rmse_after = np.sqrt(
        np.mean(error_after ** 2)
    )

    record = {
        "product": product_name,
        "reference": reference_name,
        "method": HISTORICAL_BIAS_METHOD,
        "status": "corrected",
        "overlap_start": overlap.index.min(),
        "overlap_end": overlap.index.max(),
        "n_overlap_months": n_overlap,
        "reference_mean": reference_mean,
        "product_mean": product_mean,
        "additive_bias_mm_day": additive_bias,
        "relative_bias_percent": relative_bias_percent,
        "mae_before": mae_before,
        "mae_after": mae_after,
        "rmse_before": rmse_before,
        "rmse_after": rmse_after,
    }

    return corrected, record


def correct_historical_products(
    historical_monthly_df
):
    """
    Correct GLDAS and SMAP relative to CWatM.

    Original columns remain unchanged.
    Corrected columns are added separately.
    """

    output = historical_monthly_df.copy()
    records = []

    if not APPLY_HISTORICAL_PRODUCT_BIAS_CORRECTION:
        return output, records

    reference = output[
        HISTORICAL_BIAS_REFERENCE
    ]

    for product in HISTORICAL_PRODUCTS_TO_CORRECT:
        if product not in output.columns:
            continue

        corrected, record = (
            calculate_additive_historical_bias(
                product_series=output[product],
                reference_series=reference,
                product_name=product,
                reference_name=HISTORICAL_BIAS_REFERENCE
            )
        )

        output[
            f"{product}_BiasCorrected"
        ] = corrected

        records.append(record)

        print("\n" + "-" * 80)
        print(
            f"Historical bias correction: {product}"
        )
        print(
            f"Overlap: {record['overlap_start']} "
            f"to {record['overlap_end']}"
        )
        print(
            f"Number of common months: "
            f"{record['n_overlap_months']}"
        )
        print(
            f"Additive bias: "
            f"{record['additive_bias_mm_day']:.6f} mm/day"
        )
        print(
            f"Relative bias: "
            f"{record['relative_bias_percent']:.2f}%"
        )
        print(
            f"RMSE before: "
            f"{record['rmse_before']:.6f}"
        )
        print(
            f"RMSE after: "
            f"{record['rmse_after']:.6f}"
        )

    return output, records


# ============================================================
# 13. MONTHLY EDGE CORRECTIONS
# ============================================================

def correct_historical_monthly_edges(
    historical_monthly_df
):
    output = historical_monthly_df.copy()
    records = []

    if not CORRECT_MONTHLY_EDGE_VALUES:
        return output, records

    for column in HISTORICAL_FIXED_COLUMNS:
        if column not in output.columns:
            continue

        (
            corrected_series,
            original_value,
            replacement_value,
            status
        ) = replace_target_month_with_mean(
            output[column],
            target_date=HISTORICAL_TARGET_DATE,
            source_dates=HISTORICAL_SOURCE_DATES
        )

        output[column] = corrected_series

        final_value = (
            output[column].loc[
                pd.Timestamp(HISTORICAL_TARGET_DATE)
            ]
            if pd.Timestamp(HISTORICAL_TARGET_DATE)
            in output.index
            else np.nan
        )

        records.append({
            "stage": "monthly_before_annual_mean",
            "series": column,
            "target_date": HISTORICAL_TARGET_DATE,
            "source_dates": ", ".join(
                HISTORICAL_SOURCE_DATES
            ),
            "method": "target_equals_mean_of_sources",
            "original_value": original_value,
            "replacement_value": replacement_value,
            "final_value": final_value,
            "status": status,
        })

    return output, records


def correct_future_monthly_edges(
    future_monthly_df,
    scenario
):
    output = future_monthly_df.copy()
    records = []

    if not CORRECT_MONTHLY_EDGE_VALUES:
        return output, records

    for column in output.columns:

        (
            corrected_series,
            original_value,
            replacement_value,
            status
        ) = replace_target_month_with_mean(
            output[column],
            target_date=FUTURE_FIRST_TARGET_DATE,
            source_dates=FUTURE_FIRST_SOURCE_DATES
        )

        output[column] = corrected_series

        final_value = (
            output[column].loc[
                pd.Timestamp(FUTURE_FIRST_TARGET_DATE)
            ]
            if pd.Timestamp(FUTURE_FIRST_TARGET_DATE)
            in output.index
            else np.nan
        )

        records.append({
            "stage": "monthly_before_annual_mean",
            "scenario": scenario,
            "model": column,
            "target_date": FUTURE_FIRST_TARGET_DATE,
            "source_dates": ", ".join(
                FUTURE_FIRST_SOURCE_DATES
            ),
            "method": "target_equals_mean_of_sources",
            "original_value": original_value,
            "replacement_value": replacement_value,
            "final_value": final_value,
            "status": status,
        })

        (
            corrected_series,
            original_value,
            replacement_value,
            status
        ) = replace_target_month_with_mean(
            output[column],
            target_date=FUTURE_LAST_TARGET_DATE,
            source_dates=FUTURE_LAST_SOURCE_DATES
        )

        output[column] = corrected_series

        final_value = (
            output[column].loc[
                pd.Timestamp(FUTURE_LAST_TARGET_DATE)
            ]
            if pd.Timestamp(FUTURE_LAST_TARGET_DATE)
            in output.index
            else np.nan
        )

        records.append({
            "stage": "monthly_before_annual_mean",
            "scenario": scenario,
            "model": column,
            "target_date": FUTURE_LAST_TARGET_DATE,
            "source_dates": ", ".join(
                FUTURE_LAST_SOURCE_DATES
            ),
            "method": "target_equals_mean_of_sources",
            "original_value": original_value,
            "replacement_value": replacement_value,
            "final_value": final_value,
            "status": status,
        })

    return output, records


# ============================================================
# 14. PERIOD LINES AND LABELS
# ============================================================

PERIOD_LINES = [
    pd.Timestamp("2025-01-01"),
    pd.Timestamp("2050-01-01"),
    pd.Timestamp("2075-01-01"),
    pd.Timestamp("2100-01-01"),
]

PERIOD_LABELS = [
    (
        "Historical",
        pd.Timestamp("2000-01-01"),
        pd.Timestamp("2019-12-01")
    ),
    (
        "Near future",
        pd.Timestamp("2025-01-01"),
        pd.Timestamp("2049-12-01")
    ),
    (
        "Mid-century",
        pd.Timestamp("2050-01-01"),
        pd.Timestamp("2074-12-01")
    ),
    (
        "Far future",
        pd.Timestamp("2075-01-01"),
        pd.Timestamp("2099-12-01")
    ),
]


# ============================================================
# 15. PLOTTING
# ============================================================

def plot_scenario_annual_timeseries(
    scenario_label,
    historical_annual_df,
    future_annual_df,
    ensemble_annual_series,
    out_png
):
    fig, ax = plt.subplots(
        figsize=(FIG_WIDTH, FIG_HEIGHT)
    )

    historical_colors = {
        "CWatM": "black",
        "GLDAS": "tab:blue",
        "SMAP": "tab:green",
    }

    model_colors = {
        "GFDL": "tab:blue",
        "IPSL": "tab:orange",
        "MPI": "tab:green",
        "MRI": "tab:purple",
        "UKESM": "tab:brown",
    }

    # --------------------------------------------------------
    # Original historical lines
    # --------------------------------------------------------

    for column in ["CWatM", "GLDAS", "SMAP"]:
        if column not in historical_annual_df.columns:
            continue

        ax.plot(
            historical_annual_df.index,
            historical_annual_df[column],
            linewidth=LINE_WIDTH_HISTORICAL,
            marker="o",
            markersize=MARKER_SIZE_HISTORICAL,
            linestyle="-",
            label=f"Historical {column}",
            color=historical_colors[column],
            zorder=3
        )

    # --------------------------------------------------------
    # Bias-corrected historical lines
    # Same color, dashed line
    # --------------------------------------------------------

    corrected_line_settings = {
        "GLDAS_BiasCorrected": {
            "label": "GLDAS bias-corrected",
            "color": "tab:blue",
        },
        "SMAP_BiasCorrected": {
            "label": "SMAP bias-corrected",
            "color": "tab:green",
        },
    }

    for column, settings in corrected_line_settings.items():
        if column not in historical_annual_df.columns:
            continue

        ax.plot(
            historical_annual_df.index,
            historical_annual_df[column],
            linewidth=LINE_WIDTH_HISTORICAL_CORRECTED,
            marker="o",
            markersize=MARKER_SIZE_HISTORICAL_CORRECTED,
            linestyle="--",
            dashes=(5, 3),
            label=settings["label"],
            color=settings["color"],
            alpha=0.95,
            zorder=4
        )

    # --------------------------------------------------------
    # Future individual models
    # --------------------------------------------------------

    for column in future_annual_df.columns:
        ax.plot(
            future_annual_df.index,
            future_annual_df[column],
            linewidth=LINE_WIDTH_FUTURE_MODEL,
            alpha=ALPHA_FUTURE_MODEL,
            marker="o",
            markersize=MARKER_SIZE_FUTURE,
            linestyle="-",
            color=model_colors.get(column, None),
            label=f"Future {column}",
            zorder=4
        )

    # --------------------------------------------------------
    # Future ensemble mean
    # --------------------------------------------------------

    ax.plot(
        ensemble_annual_series.index,
        ensemble_annual_series.values,
        linewidth=LINE_WIDTH_ENSEMBLE,
        marker="o",
        markersize=MARKER_SIZE_ENSEMBLE,
        color="red",
        linestyle="-",
        alpha=0.95,
        label="Future ensemble mean",
        zorder=5
    )

    # --------------------------------------------------------
    # Period separators
    # --------------------------------------------------------

    for date in PERIOD_LINES:
        ax.axvline(
            date,
            color="gray",
            linestyle="--",
            linewidth=1.2,
            alpha=0.50,
            zorder=1
        )

    ax.set_xlim(
        pd.Timestamp(PLOT_START_DATE),
        pd.Timestamp(PLOT_END_DATE)
    )

    ymin, ymax = ax.get_ylim()

    for label, start, end in PERIOD_LABELS:
        midpoint = start + (end - start) / 2

        ax.text(
            midpoint,
            ymax - 0.035 * (ymax - ymin),
            label,
            ha="center",
            va="top",
            fontsize=PERIOD_LABEL_FONT_SIZE,
            color="gray"
        )

    if APPLY_FUTURE_BIAS_CORRECTION:
        title_extra = (
            "future bias-corrected to historical CWatM"
        )
    else:
        title_extra = "raw future"

    ax.set_title(
        f"Historical and Future Annual Mean ET Time Series "
        f"- {scenario_label}\n"
        f"Historical GLDAS/SMAP additive bias correction; "
        f"{title_extra}",
        fontsize=TITLE_FONT_SIZE
    )

    ax.set_xlabel(
        "Time",
        fontsize=AXIS_LABEL_FONT_SIZE
    )

    ax.set_ylabel(
        get_y_label(),
        fontsize=AXIS_LABEL_FONT_SIZE
    )

    ax.tick_params(
        axis="both",
        labelsize=TICK_FONT_SIZE
    )

    ax.grid(
        True,
        alpha=0.25
    )

    # Legend in lower-right corner
    ax.legend(
        ncol=3,
        fontsize=LEGEND_FONT_SIZE,
        loc="lower right",
        frameon=True,
        framealpha=0.88,
        edgecolor="gray"
    )

    plt.tight_layout()

    plt.savefig(
        out_png,
        dpi=300,
        bbox_inches="tight"
    )

    plt.close()


# ============================================================
# 16. LOAD HISTORICAL DATA
# ============================================================

print("\nLoading historical ET datasets...")

cwatm = open_historical_et(
    CWATM_FILE,
    "CWatM"
)

gldas = open_historical_et(
    GLDAS_FILE,
    "GLDAS"
)

smap = open_historical_et(
    SMAP_FILE,
    "SMAP"
)

hist_domain = get_historical_domain(
    cwatm,
    gldas,
    smap
)

print("\nHistorical domain intersection:")
print(
    f"lat: {hist_domain[0]:.3f} "
    f"to {hist_domain[1]:.3f}"
)
print(
    f"lon: {hist_domain[2]:.3f} "
    f"to {hist_domain[3]:.3f}"
)

cwatm_clip = clip_to_domain(
    cwatm,
    hist_domain
)

gldas_clip = clip_to_domain(
    gldas,
    hist_domain
)

smap_clip = clip_to_domain(
    smap,
    hist_domain
)

historical_monthly_raw_df = pd.concat(
    [
        to_monthly_area_series(
            cwatm_clip
        ).rename("CWatM"),

        to_monthly_area_series(
            gldas_clip
        ).rename("GLDAS"),

        to_monthly_area_series(
            smap_clip
        ).rename("SMAP"),
    ],
    axis=1
)

historical_monthly_raw_df = (
    historical_monthly_raw_df
    .sort_index()
)

historical_monthly_raw_df = (
    historical_monthly_raw_df.loc[
        (
            historical_monthly_raw_df.index >=
            pd.Timestamp(HISTORICAL_START_DATE)
        )
        &
        (
            historical_monthly_raw_df.index <=
            pd.Timestamp(HISTORICAL_END_DATE)
        )
    ]
)


# ============================================================
# 17. HISTORICAL EDGE CORRECTION
# ============================================================

(
    historical_monthly_edge_corrected_df,
    historical_edge_records
) = correct_historical_monthly_edges(
    historical_monthly_raw_df
)


# ============================================================
# 18. HISTORICAL GLDAS/SMAP BIAS CORRECTION
# ============================================================

(
    historical_monthly_all_df,
    historical_bias_records
) = correct_historical_products(
    historical_monthly_edge_corrected_df
)

historical_annual_df = monthly_to_annual_mean(
    historical_monthly_all_df
)


# ============================================================
# 19. SAVE HISTORICAL TABLES
# ============================================================

if SAVE_MONTHLY_TABLES:
    historical_monthly_raw_df.to_csv(
        os.path.join(
            TABLE_DIR,
            "historical_monthly_raw_final.csv"
        ),
        encoding="utf-8-sig"
    )

    historical_monthly_edge_corrected_df.to_csv(
        os.path.join(
            TABLE_DIR,
            "historical_monthly_edge_corrected_final.csv"
        ),
        encoding="utf-8-sig"
    )

    historical_monthly_all_df.to_csv(
        os.path.join(
            TABLE_DIR,
            "historical_monthly_with_bias_corrected_products_final.csv"
        ),
        encoding="utf-8-sig"
    )

if SAVE_ANNUAL_TABLES:
    historical_annual_df.to_csv(
        os.path.join(
            TABLE_DIR,
            "historical_annual_mean_with_bias_corrected_products_final.csv"
        ),
        encoding="utf-8-sig"
    )

historical_bias_df = pd.DataFrame(
    historical_bias_records
)

historical_bias_csv = os.path.join(
    TABLE_DIR,
    "historical_GLDAS_SMAP_bias_correction_summary.csv"
)

historical_bias_df.to_csv(
    historical_bias_csv,
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 20. FUTURE WORKFLOW
# ============================================================

all_future_summary = []
future_bias_records = []
future_edge_records_all = []
combined_future_monthly_records = []
combined_future_annual_records = []

for scenario, scenario_label in SCENARIOS.items():

    print("\n" + "=" * 80)
    print(
        f"Processing scenario: "
        f"{scenario} - {scenario_label}"
    )
    print("=" * 80)

    future_monthly_series_list = []

    for model in CLIMATE_MODELS:

        print(
            f"\nProcessing model: "
            f"{model}, scenario: {scenario}"
        )

        try:
            future_da = open_future_total_et(
                model_name=model,
                scenario=scenario,
                hist_domain=hist_domain
            )

            future_series_raw = (
                to_monthly_area_series(
                    future_da
                )
            )

            future_series_raw.name = model

            if APPLY_FUTURE_BIAS_CORRECTION:
                (
                    future_series_used,
                    factor,
                    n_overlap
                ) = bias_correct_future_series(
                    future_series=future_series_raw,
                    reference_series=(
                        historical_monthly_edge_corrected_df[
                            FUTURE_BIAS_REFERENCE
                        ]
                    )
                )

            else:
                future_series_used = (
                    future_series_raw.copy()
                )

                factor = np.nan
                n_overlap = 0

            future_series_used = (
                future_series_used.loc[
                    (
                        future_series_used.index >=
                        pd.Timestamp(FUTURE_START_DATE)
                    )
                    &
                    (
                        future_series_used.index <=
                        pd.Timestamp(FUTURE_END_DATE)
                    )
                ].copy()
            )

            future_series_used.name = model

            future_monthly_series_list.append(
                future_series_used
            )

            future_bias_records.append({
                "model": model,
                "scenario": scenario,
                "bias_reference": FUTURE_BIAS_REFERENCE,
                "bias_method": (
                    FUTURE_BIAS_METHOD
                    if APPLY_FUTURE_BIAS_CORRECTION
                    else "none"
                ),
                "bias_factor": factor,
                "n_overlap_months": n_overlap,
            })

        except Exception as error:
            print(
                f"ERROR for {model} / {scenario}: "
                f"{error}"
            )

    if len(future_monthly_series_list) == 0:
        print(
            f"No valid future data for scenario "
            f"{scenario}. Plot skipped."
        )
        continue

    future_monthly_df = pd.concat(
        future_monthly_series_list,
        axis=1
    )

    future_monthly_df = (
        future_monthly_df.sort_index()
    )

    available_columns = [
        model
        for model in CLIMATE_MODELS
        if model in future_monthly_df.columns
    ]

    future_monthly_df = future_monthly_df[
        available_columns
    ]

    (
        future_monthly_corrected_df,
        future_edge_records
    ) = correct_future_monthly_edges(
        future_monthly_df,
        scenario=scenario
    )

    future_edge_records_all.extend(
        future_edge_records
    )

    future_annual_df = monthly_to_annual_mean(
        future_monthly_corrected_df
    )

    ensemble_annual_series = (
        future_annual_df.mean(
            axis=1,
            skipna=True
        )
    )

    ensemble_annual_series.name = (
        "Future_Ensemble_Mean"
    )

    for model in future_annual_df.columns:
        valid_series = (
            future_annual_df[model]
            .dropna()
        )

        all_future_summary.append({
            "model": model,
            "scenario": scenario,
            "scenario_label": scenario_label,
            "mode": "totalET_only",
            "bias_corrected": (
                APPLY_FUTURE_BIAS_CORRECTION
            ),
            "start": (
                valid_series.index.min()
                if len(valid_series) > 0
                else pd.NaT
            ),
            "end": (
                valid_series.index.max()
                if len(valid_series) > 0
                else pd.NaT
            ),
            "n_years": len(valid_series),
            "annual_mean": valid_series.mean(),
            "annual_min": valid_series.min(),
            "annual_max": valid_series.max(),
            "domain_lat_min": hist_domain[0],
            "domain_lat_max": hist_domain[1],
            "domain_lon_min": hist_domain[2],
            "domain_lon_max": hist_domain[3],
        })

    if SAVE_MONTHLY_TABLES:
        future_monthly_df.to_csv(
            os.path.join(
                TABLE_DIR,
                f"future_monthly_raw_scenario_"
                f"{scenario}_final.csv"
            ),
            encoding="utf-8-sig"
        )

        future_monthly_corrected_df.to_csv(
            os.path.join(
                TABLE_DIR,
                f"future_monthly_corrected_scenario_"
                f"{scenario}_final.csv"
            ),
            encoding="utf-8-sig"
        )

    if SAVE_ANNUAL_TABLES:
        future_annual_df.to_csv(
            os.path.join(
                TABLE_DIR,
                f"future_annual_mean_scenario_"
                f"{scenario}_final.csv"
            ),
            encoding="utf-8-sig"
        )

        ensemble_annual_series.to_frame().to_csv(
            os.path.join(
                TABLE_DIR,
                f"future_ensemble_annual_mean_scenario_"
                f"{scenario}_final.csv"
            ),
            encoding="utf-8-sig"
        )

    temporary_monthly = (
        future_monthly_corrected_df.copy()
    )

    temporary_monthly["scenario"] = scenario
    temporary_monthly["scenario_label"] = scenario_label

    temporary_monthly = (
        temporary_monthly
        .reset_index()
        .rename(columns={"index": "time"})
    )

    combined_future_monthly_records.append(
        temporary_monthly
    )

    temporary_annual = future_annual_df.copy()

    temporary_annual[
        "Ensemble_Mean"
    ] = ensemble_annual_series

    temporary_annual["scenario"] = scenario
    temporary_annual["scenario_label"] = scenario_label

    temporary_annual = (
        temporary_annual
        .reset_index()
        .rename(columns={"index": "time"})
    )

    combined_future_annual_records.append(
        temporary_annual
    )

    output_figure = os.path.join(
        FIG_DIR,
        f"annual_historical_future_ET_scenario_"
        f"{scenario}_bias_corrected_final.png"
    )

    plot_scenario_annual_timeseries(
        scenario_label=scenario_label,
        historical_annual_df=historical_annual_df,
        future_annual_df=future_annual_df,
        ensemble_annual_series=ensemble_annual_series,
        out_png=output_figure
    )


# ============================================================
# 21. SAVE SUMMARY TABLES
# ============================================================

future_summary_df = pd.DataFrame(
    all_future_summary
)

future_summary_csv = os.path.join(
    TABLE_DIR,
    "future_models_annual_summary_final.csv"
)

future_summary_df.to_csv(
    future_summary_csv,
    index=False,
    encoding="utf-8-sig"
)

future_bias_df = pd.DataFrame(
    future_bias_records
)

future_bias_csv = os.path.join(
    TABLE_DIR,
    "future_bias_correction_factors_final.csv"
)

future_bias_df.to_csv(
    future_bias_csv,
    index=False,
    encoding="utf-8-sig"
)

historical_edge_df = pd.DataFrame(
    historical_edge_records
)

historical_edge_csv = os.path.join(
    TABLE_DIR,
    "historical_monthly_edge_corrections_final.csv"
)

historical_edge_df.to_csv(
    historical_edge_csv,
    index=False,
    encoding="utf-8-sig"
)

future_edge_df = pd.DataFrame(
    future_edge_records_all
)

future_edge_csv = os.path.join(
    TABLE_DIR,
    "future_monthly_edge_corrections_final.csv"
)

future_edge_df.to_csv(
    future_edge_csv,
    index=False,
    encoding="utf-8-sig"
)

if len(combined_future_monthly_records) > 0:
    combined_future_monthly_df = pd.concat(
        combined_future_monthly_records,
        axis=0
    )

    combined_future_monthly_csv = os.path.join(
        TABLE_DIR,
        "future_monthly_corrected_all_models_"
        "all_scenarios_final.csv"
    )

    combined_future_monthly_df.to_csv(
        combined_future_monthly_csv,
        index=False,
        encoding="utf-8-sig"
    )

else:
    combined_future_monthly_csv = None

if len(combined_future_annual_records) > 0:
    combined_future_annual_df = pd.concat(
        combined_future_annual_records,
        axis=0
    )

    combined_future_annual_csv = os.path.join(
        TABLE_DIR,
        "future_annual_mean_all_models_"
        "all_scenarios_final.csv"
    )

    combined_future_annual_df.to_csv(
        combined_future_annual_csv,
        index=False,
        encoding="utf-8-sig"
    )

else:
    combined_future_annual_csv = None


# ============================================================
# 22. SAVE CONFIGURATION
# ============================================================

configuration_df = pd.DataFrame([
    {
        "parameter": "analysis_type",
        "value": "annual_mean_with_historical_bias_correction"
    },
    {
        "parameter": "future_mode",
        "value": "totalET_only"
    },
    {
        "parameter": "base_dir",
        "value": BASE_DIR
    },
    {
        "parameter": "climate_models",
        "value": ", ".join(CLIMATE_MODELS)
    },
    {
        "parameter": "scenarios",
        "value": ", ".join(SCENARIOS.keys())
    },
    {
        "parameter": "historical_start_date",
        "value": HISTORICAL_START_DATE
    },
    {
        "parameter": "historical_end_date",
        "value": HISTORICAL_END_DATE
    },
    {
        "parameter": "future_start_date",
        "value": FUTURE_START_DATE
    },
    {
        "parameter": "future_end_date",
        "value": FUTURE_END_DATE
    },
    {
        "parameter": "historical_bias_reference",
        "value": HISTORICAL_BIAS_REFERENCE
    },
    {
        "parameter": "historical_bias_method",
        "value": HISTORICAL_BIAS_METHOD
    },
    {
        "parameter": "historical_products_corrected",
        "value": ", ".join(
            HISTORICAL_PRODUCTS_TO_CORRECT
        )
    },
    {
        "parameter": "apply_future_bias_correction",
        "value": APPLY_FUTURE_BIAS_CORRECTION
    },
    {
        "parameter": "future_bias_reference",
        "value": FUTURE_BIAS_REFERENCE
    },
    {
        "parameter": "future_bias_method",
        "value": FUTURE_BIAS_METHOD
    },
    {
        "parameter": "legend_location",
        "value": "lower right"
    },
    {
        "parameter": "domain_lat_min",
        "value": hist_domain[0]
    },
    {
        "parameter": "domain_lat_max",
        "value": hist_domain[1]
    },
    {
        "parameter": "domain_lon_min",
        "value": hist_domain[2]
    },
    {
        "parameter": "domain_lon_max",
        "value": hist_domain[3]
    },
])

configuration_csv = os.path.join(
    TABLE_DIR,
    "run_configuration_annual_bias_corrected_final.csv"
)

configuration_df.to_csv(
    configuration_csv,
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 23. FINAL REPORT
# ============================================================

print("\n" + "=" * 80)
print("DONE")
print("=" * 80)

print("\nHistorical bias correction summary:")
print(historical_bias_csv)

if len(historical_bias_df) > 0:
    print(
        historical_bias_df[
            [
                "product",
                "overlap_start",
                "overlap_end",
                "n_overlap_months",
                "additive_bias_mm_day",
                "relative_bias_percent",
                "rmse_before",
                "rmse_after",
            ]
        ].to_string(index=False)
    )

print("\nMain output folder:")
print(OUT_DIR)

print("\nMain tables:")
print(future_summary_csv)
print(future_bias_csv)
print(historical_bias_csv)
print(historical_edge_csv)
print(future_edge_csv)
print(configuration_csv)

if combined_future_monthly_csv is not None:
    print(combined_future_monthly_csv)

if combined_future_annual_csv is not None:
    print(combined_future_annual_csv)

print("\nFigures:")

for scenario in SCENARIOS:
    print(
        os.path.join(
            FIG_DIR,
            f"annual_historical_future_ET_scenario_"
            f"{scenario}_bias_corrected_final.png"
        )
    )

print("\nNotes:")
print(
    "GLDAS and SMAP additive biases are calculated "
    "from their common monthly periods with CWatM."
)
print(
    "Corrected GLDAS and SMAP annual series are shown "
    "with dashed lines in the same original colors."
)
print(
    "Original historical series remain visible as solid lines."
)
print(
    "The legend is located in the lower-right corner."
)


Loading historical ET datasets...

Opening historical dataset: CWatM
E:\ET_Article_With_MM_SA\Future data ST\ET_CWatM_mm_day.nc
Variable: ET_final_mm_day
Dims: ('time', 'lat', 'lon')
Shape: (468, 13, 14)
Time: 1981-01 to 2019-12
Mean: 1.102419

Opening historical dataset: GLDAS
E:\ET_Article_With_MM_SA\Future data ST\GLDAS_ET_2000_2019.nc
Variable: __xarray_dataarray_variable__
Dims: ('time', 'lat', 'lon')
Shape: (240, 13, 14)
Time: 2000-01 to 2019-12
Mean: 0.793585

Opening historical dataset: SMAP
E:\ET_Article_With_MM_SA\Future data ST\SMAP_ET_2000_2019.nc
Variable: __xarray_dataarray_variable__
Dims: ('time', 'lat', 'lon')
Shape: (58, 13, 14)
Time: 2015-03 to 2019-12
Mean: 1.057844

Historical domain intersection:
lat: 30.750 to 36.750
lon: 45.250 to 51.750

--------------------------------------------------------------------------------
Historical bias correction: GLDAS
Overlap: 2000-01-01 00:00:00 to 2019-12-01 00:00:00
Number of common months: 240
Additive bias: -0.295263 mm/da

In [27]:
# -*- coding: utf-8 -*-

"""
Boxplot comparison of monthly ET distributions

Goal:
    Compare monthly ET distribution between:
        1. Historical CWatM prediction model
        2. Five future climate models:
            GFDL, IPSL, MPI, MRI, UKESM

For each:
    - 3 scenarios: 126, 370, 585
    - 3 future periods:
        Near future : 2025-2049
        Mid-century : 2050-2074
        Far future  : 2075-2099

Total figures:
    3 scenarios × 3 future periods = 9 boxplots

Each boxplot has 6 boxes:
    CWatM historical + 5 climate models

Data:
    Monthly area-weighted ET
"""

import os
import re
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt


# ============================================================
# 1. PATH SETTINGS
# ============================================================

BASE_DIR = os.getcwd()

CWATM_FILE = os.path.join(BASE_DIR, "ET_CWatM_mm_day.nc")

FUTURE_BASE_DIR = BASE_DIR


# ============================================================
# 2. MODELS AND SCENARIOS
# ============================================================

CLIMATE_MODELS = [
    "GFDL",
    "IPSL",
    "MPI",
    "MRI",
    "UKESM",
]

SCENARIOS = {
    "126": "Optimistic / SSP126",
    "370": "Medium / SSP370",
    "585": "Pessimistic / SSP585",
}

FUTURE_TOTAL_ET_FILE = "totalET_monthavg.nc"
FUTURE_TOTAL_ET_VAR = "totalET_monthavg"


# ============================================================
# 3. OUTPUT SETTINGS
# ============================================================

OUT_DIR = os.path.join(BASE_DIR, "outputs_ET_boxplots_CWatM_vs_climate_models")
TABLE_DIR = os.path.join(OUT_DIR, "tables")
FIG_DIR = os.path.join(OUT_DIR, "figures")

for d in [OUT_DIR, TABLE_DIR, FIG_DIR]:
    os.makedirs(d, exist_ok=True)


# ============================================================
# 4. ANALYSIS SETTINGS
# ============================================================

# Historical CWatM comparison period
# اگر فایل CWatM شما تا 2019 باشد، کد خودش فقط داده‌های موجود را استفاده می‌کند.
CWATM_HIST_START = "1995-01-01"
CWATM_HIST_END = "2020-12-01"

# Future 25-year periods
# برای 25 سال کامل، بازه‌ها به صورت 2025-2049، 2050-2074 و 2075-2099 تعریف شده‌اند.
FUTURE_PERIODS = {
    "near_future": {
        "label": "Near future",
        "start": "2025-01-01",
        "end": "2049-12-01",
    },
    "mid_century": {
        "label": "Mid-century",
        "start": "2050-01-01",
        "end": "2074-12-01",
    },
    "far_future": {
        "label": "Far future",
        "start": "2075-01-01",
        "end": "2099-12-01",
    },
}

# Unit settings
HISTORICAL_UNIT = "mm/day"

# Future totalET_monthavg has unit m in your files.
# It is converted to mm by multiplying by 1000.
FUTURE_CONVERT_M_TO_MM = True

# Bias correction is recommended if future totalET scale differs from CWatM.
APPLY_BIAS_CORRECTION = True
BIAS_METHOD = "multiplicative_mean_ratio"

# Negative values
MASK_NEGATIVE_HISTORICAL_ET = True
MASK_NEGATIVE_FUTURE_ET = True

# Save extracted monthly values for each plot
SAVE_BOXPLOT_VALUES = True


# ============================================================
# 5. FIGURE SETTINGS
# ============================================================

FONT_FAMILY = "DejaVu Sans"

TITLE_FONT_SIZE = 18
AXIS_LABEL_FONT_SIZE = 16
TICK_FONT_SIZE = 13
BOX_LABEL_FONT_SIZE = 12
ANNOTATION_FONT_SIZE = 11

FIG_WIDTH = 12
FIG_HEIGHT = 7

BOX_WIDTH = 0.55
SHOW_FLIERS = True

Y_LABEL = "Monthly ET (mm/day)"

plt.rcParams["font.family"] = FONT_FAMILY
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.dpi"] = 150
plt.rcParams["axes.titlesize"] = TITLE_FONT_SIZE
plt.rcParams["axes.labelsize"] = AXIS_LABEL_FONT_SIZE
plt.rcParams["xtick.labelsize"] = TICK_FONT_SIZE
plt.rcParams["ytick.labelsize"] = TICK_FONT_SIZE


# ============================================================
# 6. BASIC FUNCTIONS
# ============================================================

def ensure_month_start_index(series_or_df):
    obj = series_or_df.copy()
    obj.index = pd.to_datetime(obj.index)
    obj.index = obj.index.to_period("M").to_timestamp()
    obj = obj.sort_index()
    return obj


def safe_name(text):
    text = str(text)
    text = text.replace("/", "_")
    text = text.replace(" ", "_")
    text = text.replace("-", "_")
    text = re.sub(r"[^\w]", "_", text)
    text = re.sub(r"_+", "_", text)
    return text.strip("_")


def describe_series(series):
    s = pd.Series(series).dropna()

    if len(s) == 0:
        return {
            "n": 0,
            "mean": np.nan,
            "median": np.nan,
            "std": np.nan,
            "min": np.nan,
            "q25": np.nan,
            "q75": np.nan,
            "max": np.nan,
        }

    return {
        "n": int(s.shape[0]),
        "mean": float(s.mean()),
        "median": float(s.median()),
        "std": float(s.std()),
        "min": float(s.min()),
        "q25": float(s.quantile(0.25)),
        "q75": float(s.quantile(0.75)),
        "max": float(s.max()),
    }


# ============================================================
# 7. NETCDF READER FUNCTIONS
# ============================================================

def extract_dates_from_long_name(long_name_attr):
    if isinstance(long_name_attr, str):
        long_names = [long_name_attr]
    else:
        long_names = list(long_name_attr)

    dates = []

    for item in long_names:
        item = str(item)
        match = re.search(r"(\d{4})_(\d{2})", item)

        if match is None:
            raise ValueError(f"Cannot extract date from long_name item: {item}")

        dates.append(
            pd.Timestamp(
                year=int(match.group(1)),
                month=int(match.group(2)),
                day=1
            )
        )

    return pd.DatetimeIndex(dates)


def standardize_xy_to_latlon(da):
    rename_dict = {}

    for name in list(da.dims) + list(da.coords):
        low = str(name).lower()

        if low in ["x", "longitude"]:
            rename_dict[name] = "lon"

        elif low in ["y", "latitude"]:
            rename_dict[name] = "lat"

    if rename_dict:
        da = da.rename(rename_dict)

    if "lat" in da.coords:
        da = da.sortby("lat")

    if "lon" in da.coords:
        da = da.sortby("lon")

    return da


def open_cwatm_et(path):
    print("\n" + "=" * 80)
    print("Opening historical CWatM")
    print(path)
    print("=" * 80)

    if not os.path.exists(path):
        raise FileNotFoundError(path)

    ds = xr.open_dataset(path)

    candidate_vars = [
        v for v in ds.data_vars
        if str(v).lower() not in ["spatial_ref", "crs"]
    ]

    if len(candidate_vars) == 0:
        raise ValueError("No variable found in CWatM file.")

    if "ET_final_mm_day" in candidate_vars:
        var_name = "ET_final_mm_day"
    else:
        var_name = candidate_vars[0]

    da = ds[var_name]

    if "time" not in da.dims:
        raise ValueError("No time dimension found in CWatM file.")

    da["time"] = pd.to_datetime(da.time.values)

    da = standardize_xy_to_latlon(da)

    for d in ["time", "lat", "lon"]:
        if d not in da.dims:
            raise ValueError(f"CWatM required dimension '{d}' not found.")

    da = da.astype("float64")
    da = da.where(np.isfinite(da))

    if MASK_NEGATIVE_HISTORICAL_ET:
        da = da.where(da >= 0)

    da.name = "CWatM"
    da.attrs["unit"] = HISTORICAL_UNIT

    print(f"Variable: {var_name}")
    print(f"Shape: {da.shape}")
    print(
        "Time:",
        pd.to_datetime(da.time.values[0]).strftime("%Y-%m"),
        "to",
        pd.to_datetime(da.time.values[-1]).strftime("%Y-%m")
    )
    print(f"Mean: {float(da.mean(skipna=True)):.6f}")

    return da


def open_future_total_et(model_name, scenario, hist_domain):
    scen_dir = os.path.join(FUTURE_BASE_DIR, model_name, scenario)
    total_path = os.path.join(scen_dir, FUTURE_TOTAL_ET_FILE)

    if not os.path.exists(total_path):
        raise FileNotFoundError(total_path)

    ds = xr.open_dataset(total_path)

    candidate_vars = [
        v for v in ds.data_vars
        if str(v).lower() not in ["spatial_ref", "crs"]
    ]

    if FUTURE_TOTAL_ET_VAR in candidate_vars:
        var_name = FUTURE_TOTAL_ET_VAR
    elif len(candidate_vars) == 1:
        var_name = candidate_vars[0]
    else:
        raise ValueError(
            f"Cannot select future ET variable from {total_path}. "
            f"Candidates: {candidate_vars}"
        )

    da = ds[var_name]

    if "time" not in da.dims:
        raise ValueError(f"No time dimension found in {total_path}")

    da["time"] = pd.to_datetime(da.time.values)

    da = standardize_xy_to_latlon(da)

    for d in ["time", "lat", "lon"]:
        if d not in da.dims:
            raise ValueError(f"{model_name}/{scenario}: dimension '{d}' not found.")

    da = da.astype("float64")
    da = da.where(np.isfinite(da))

    da = clip_to_domain(da, hist_domain)

    if FUTURE_CONVERT_M_TO_MM:
        da = da * 1000.0

    if MASK_NEGATIVE_FUTURE_ET:
        da = da.where(da >= 0)

    da.name = f"{model_name}_{scenario}"

    return da


# ============================================================
# 8. AREA FUNCTIONS
# ============================================================

def infer_bounds_from_centers(values):
    values = np.asarray(values, dtype=float)

    if len(values) < 2:
        raise ValueError("At least two coordinate values are needed.")

    values_sorted = np.sort(values)

    mid = (values_sorted[:-1] + values_sorted[1:]) / 2.0

    first = values_sorted[0] - (mid[0] - values_sorted[0])
    last = values_sorted[-1] + (values_sorted[-1] - mid[-1])

    return np.concatenate([[first], mid, [last]])


def calculate_latlon_cell_area(lat, lon, radius=6371000.0):
    lat_values = np.asarray(lat.values if hasattr(lat, "values") else lat, dtype=float)
    lon_values = np.asarray(lon.values if hasattr(lon, "values") else lon, dtype=float)

    lat_ascending = np.all(np.diff(lat_values) > 0)
    lon_ascending = np.all(np.diff(lon_values) > 0)

    lat_sorted = np.sort(lat_values)
    lon_sorted = np.sort(lon_values)

    lat_bounds = infer_bounds_from_centers(lat_sorted)
    lon_bounds = infer_bounds_from_centers(lon_sorted)

    lat_bounds_rad = np.deg2rad(lat_bounds)
    lon_bounds_rad = np.deg2rad(lon_bounds)

    dlon = np.diff(lon_bounds_rad)
    sin_lat_diff = np.diff(np.sin(lat_bounds_rad))

    area_sorted = (radius ** 2) * sin_lat_diff[:, None] * dlon[None, :]
    area_sorted = np.abs(area_sorted)

    if not lat_ascending:
        area_sorted = area_sorted[::-1, :]

    if not lon_ascending:
        area_sorted = area_sorted[:, ::-1]

    area = xr.DataArray(
        area_sorted,
        dims=("lat", "lon"),
        coords={
            "lat": lat_values,
            "lon": lon_values,
        },
        name="cell_area"
    )

    area.attrs["units"] = "m2"

    return area


def area_weighted_mean(da):
    area = calculate_latlon_cell_area(da.lat, da.lon)
    valid = da.notnull()

    weighted_sum = (da * area).where(valid).sum(dim=("lat", "lon"), skipna=True)
    weight_sum = area.where(valid).sum(dim=("lat", "lon"), skipna=True)

    out = weighted_sum / weight_sum
    out.name = da.name

    return out


def to_monthly_area_series(da):
    s = area_weighted_mean(da).to_series()
    s = ensure_month_start_index(s)
    return s


# ============================================================
# 9. DOMAIN AND BIAS FUNCTIONS
# ============================================================

def get_domain_from_array(da):
    lat_min = float(da.lat.min())
    lat_max = float(da.lat.max())
    lon_min = float(da.lon.min())
    lon_max = float(da.lon.max())

    return lat_min, lat_max, lon_min, lon_max


def clip_to_domain(da, domain):
    lat_min, lat_max, lon_min, lon_max = domain

    da = da.sortby("lat")
    da = da.sortby("lon")

    return da.sel(
        lat=slice(lat_min, lat_max),
        lon=slice(lon_min, lon_max)
    )


def bias_correct_future_series(future_series, reference_series):
    df = pd.concat([future_series, reference_series], axis=1).dropna()
    df.columns = ["future", "reference"]

    n_overlap = len(df)

    if n_overlap < 12:
        return future_series, np.nan, n_overlap

    future_mean = df["future"].mean()
    reference_mean = df["reference"].mean()

    if not np.isfinite(future_mean) or future_mean == 0:
        return future_series, np.nan, n_overlap

    factor = reference_mean / future_mean

    corrected = future_series * factor
    corrected.name = future_series.name

    return corrected, factor, n_overlap


# ============================================================
# 10. BOXPLOT FUNCTION
# ============================================================

def make_boxplot(
    data_dict,
    scenario,
    scenario_label,
    period_key,
    period_label,
    period_start,
    period_end,
    out_png
):
    labels = list(data_dict.keys())
    values = [pd.Series(data_dict[k]).dropna().values for k in labels]

    fig, ax = plt.subplots(figsize=(FIG_WIDTH, FIG_HEIGHT))

    bp = ax.boxplot(
        values,
        labels=labels,
        widths=BOX_WIDTH,
        patch_artist=True,
        showfliers=SHOW_FLIERS,
        medianprops=dict(linewidth=2),
        boxprops=dict(linewidth=1.4),
        whiskerprops=dict(linewidth=1.2),
        capprops=dict(linewidth=1.2),
        flierprops=dict(marker="o", markersize=3, alpha=0.45)
    )

    for patch in bp["boxes"]:
        patch.set_alpha(0.65)

    ax.set_title(
        f"Monthly ET distribution: CWatM vs climate models\n"
        f"{scenario_label} | {period_label}",
        fontsize=TITLE_FONT_SIZE
    )

    ax.set_ylabel(Y_LABEL, fontsize=AXIS_LABEL_FONT_SIZE)

    ax.tick_params(axis="x", labelsize=BOX_LABEL_FONT_SIZE, rotation=25)
    ax.tick_params(axis="y", labelsize=TICK_FONT_SIZE)

    ax.grid(axis="y", alpha=0.30)

    info_text = (
        f"CWatM: 1995-2020\n"
        f"Future: {pd.Timestamp(period_start).year}-{pd.Timestamp(period_end).year}\n"
        f"Monthly data"
    )

    ax.text(
        0.98,
        0.03,
        info_text,
        transform=ax.transAxes,
        ha="right",
        va="bottom",
        fontsize=ANNOTATION_FONT_SIZE,
        bbox=dict(boxstyle="round", facecolor="white", alpha=0.75)
    )

    plt.tight_layout()
    plt.savefig(out_png, dpi=300)
    plt.close()


# ============================================================
# 11. MAIN WORKFLOW
# ============================================================

print("\nLoading CWatM historical data...")

cwatm_da = open_cwatm_et(CWATM_FILE)

hist_domain = get_domain_from_array(cwatm_da)

print("\nDomain used for future clipping:")
print(f"lat: {hist_domain[0]:.3f} to {hist_domain[1]:.3f}")
print(f"lon: {hist_domain[2]:.3f} to {hist_domain[3]:.3f}")

cwatm_series_all = to_monthly_area_series(cwatm_da)

cwatm_hist_series = cwatm_series_all.loc[
    (cwatm_series_all.index >= pd.Timestamp(CWATM_HIST_START)) &
    (cwatm_series_all.index <= pd.Timestamp(CWATM_HIST_END))
].copy()

cwatm_hist_series.name = "CWatM"

print("\nCWatM comparison period:")
print(f"Requested: {CWATM_HIST_START} to {CWATM_HIST_END}")
print(
    "Available:",
    cwatm_hist_series.dropna().index.min(),
    "to",
    cwatm_hist_series.dropna().index.max(),
    "| n months:",
    cwatm_hist_series.dropna().shape[0]
)

cwatm_hist_csv = os.path.join(TABLE_DIR, "CWatM_historical_monthly_ET_1995_2020.csv")
cwatm_hist_series.to_frame().to_csv(cwatm_hist_csv, encoding="utf-8-sig")


all_summary_records = []
all_bias_records = []
all_boxplot_values_records = []


for scenario, scenario_label in SCENARIOS.items():

    print("\n" + "=" * 80)
    print(f"Processing scenario: {scenario} - {scenario_label}")
    print("=" * 80)

    future_model_series = {}

    for model in CLIMATE_MODELS:

        print(f"\nReading future model: {model}, scenario: {scenario}")

        try:
            future_da = open_future_total_et(
                model_name=model,
                scenario=scenario,
                hist_domain=hist_domain
            )

            future_series_raw = to_monthly_area_series(future_da)
            future_series_raw.name = model

            if APPLY_BIAS_CORRECTION:
                future_series_used, factor, n_overlap = bias_correct_future_series(
                    future_series=future_series_raw,
                    reference_series=cwatm_series_all
                )
            else:
                future_series_used = future_series_raw
                factor = np.nan
                n_overlap = 0

            future_series_used.name = model
            future_model_series[model] = future_series_used

            all_bias_records.append({
                "scenario": scenario,
                "scenario_label": scenario_label,
                "model": model,
                "apply_bias_correction": APPLY_BIAS_CORRECTION,
                "bias_method": BIAS_METHOD if APPLY_BIAS_CORRECTION else "none",
                "bias_factor": factor,
                "n_overlap_months": n_overlap,
            })

        except Exception as e:
            print(f"ERROR for {model} / {scenario}: {e}")

    for period_key, period_info in FUTURE_PERIODS.items():

        period_label = period_info["label"]
        period_start = period_info["start"]
        period_end = period_info["end"]

        print(f"\nCreating boxplot: {scenario} - {period_label}")

        data_dict = {
            "CWatM\n1995-2020": cwatm_hist_series.dropna()
        }

        for model in CLIMATE_MODELS:

            if model not in future_model_series:
                continue

            s = future_model_series[model].loc[
                (future_model_series[model].index >= pd.Timestamp(period_start)) &
                (future_model_series[model].index <= pd.Timestamp(period_end))
            ].copy()

            data_dict[model] = s.dropna()

        out_png = os.path.join(
            FIG_DIR,
            f"boxplot_ET_{scenario}_{period_key}.png"
        )

        make_boxplot(
            data_dict=data_dict,
            scenario=scenario,
            scenario_label=scenario_label,
            period_key=period_key,
            period_label=period_label,
            period_start=period_start,
            period_end=period_end,
            out_png=out_png
        )

        # Save values for this boxplot
        if SAVE_BOXPLOT_VALUES:
            value_rows = []

            for label, values in data_dict.items():
                temp = pd.Series(values).dropna().to_frame(name="ET")
                temp["series"] = label.replace("\n", " ")
                temp["scenario"] = scenario
                temp["scenario_label"] = scenario_label
                temp["future_period"] = period_key
                temp["future_period_label"] = period_label
                temp = temp.reset_index().rename(columns={"index": "time"})
                value_rows.append(temp)

                desc = describe_series(values)

                all_summary_records.append({
                    "scenario": scenario,
                    "scenario_label": scenario_label,
                    "future_period": period_key,
                    "future_period_label": period_label,
                    "future_start": period_start,
                    "future_end": period_end,
                    "series": label.replace("\n", " "),
                    **desc
                })

            if len(value_rows) > 0:
                box_values_df = pd.concat(value_rows, axis=0)

                box_values_csv = os.path.join(
                    TABLE_DIR,
                    f"boxplot_values_ET_{scenario}_{period_key}.csv"
                )

                box_values_df.to_csv(
                    box_values_csv,
                    index=False,
                    encoding="utf-8-sig"
                )

                all_boxplot_values_records.append(box_values_df)


# ============================================================
# 12. SAVE SUMMARY TABLES
# ============================================================

summary_df = pd.DataFrame(all_summary_records)

summary_csv = os.path.join(TABLE_DIR, "boxplot_summary_statistics_all.csv")
summary_df.to_csv(summary_csv, index=False, encoding="utf-8-sig")

bias_df = pd.DataFrame(all_bias_records)

bias_csv = os.path.join(TABLE_DIR, "future_bias_correction_factors_boxplot.csv")
bias_df.to_csv(bias_csv, index=False, encoding="utf-8-sig")

if len(all_boxplot_values_records) > 0:
    all_values_df = pd.concat(all_boxplot_values_records, axis=0)

    all_values_csv = os.path.join(TABLE_DIR, "boxplot_values_all.csv")
    all_values_df.to_csv(all_values_csv, index=False, encoding="utf-8-sig")
else:
    all_values_csv = None


config_df = pd.DataFrame([
    {"parameter": "analysis_type", "value": "monthly_boxplot_distribution_comparison"},
    {"parameter": "historical_reference", "value": "CWatM"},
    {"parameter": "cwatm_hist_start", "value": CWATM_HIST_START},
    {"parameter": "cwatm_hist_end", "value": CWATM_HIST_END},
    {"parameter": "future_models", "value": ", ".join(CLIMATE_MODELS)},
    {"parameter": "scenarios", "value": ", ".join(SCENARIOS.keys())},
    {"parameter": "apply_bias_correction", "value": APPLY_BIAS_CORRECTION},
    {"parameter": "bias_method", "value": BIAS_METHOD},
    {"parameter": "future_total_et_file", "value": FUTURE_TOTAL_ET_FILE},
    {"parameter": "future_total_et_var", "value": FUTURE_TOTAL_ET_VAR},
    {"parameter": "future_convert_m_to_mm", "value": FUTURE_CONVERT_M_TO_MM},
    {"parameter": "y_label", "value": Y_LABEL},
    {"parameter": "domain_lat_min", "value": hist_domain[0]},
    {"parameter": "domain_lat_max", "value": hist_domain[1]},
    {"parameter": "domain_lon_min", "value": hist_domain[2]},
    {"parameter": "domain_lon_max", "value": hist_domain[3]},
])

for key, info in FUTURE_PERIODS.items():
    config_df = pd.concat(
        [
            config_df,
            pd.DataFrame([
                {"parameter": f"{key}_label", "value": info["label"]},
                {"parameter": f"{key}_start", "value": info["start"]},
                {"parameter": f"{key}_end", "value": info["end"]},
            ])
        ],
        ignore_index=True
    )

config_csv = os.path.join(TABLE_DIR, "run_configuration_boxplot.csv")
config_df.to_csv(config_csv, index=False, encoding="utf-8-sig")


# ============================================================
# 13. FINAL REPORT
# ============================================================

print("\n" + "=" * 80)
print("DONE")
print("=" * 80)

print("\nOutput folder:")
print(OUT_DIR)

print("\nFigures folder:")
print(FIG_DIR)

print("\nCreated boxplot figures:")
for scenario in SCENARIOS:
    for period_key in FUTURE_PERIODS:
        print(os.path.join(FIG_DIR, f"boxplot_ET_{scenario}_{period_key}.png"))

print("\nMain tables:")
print(cwatm_hist_csv)
print(summary_csv)
print(bias_csv)
print(config_csv)

if all_values_csv is not None:
    print(all_values_csv)

print("\nNotes:")
print("Each figure has 6 boxes: CWatM historical + five future climate models.")
print("CWatM historical period requested: 1995-2020.")
print("Future periods are defined as 25-year periods: 2025-2049, 2050-2074, 2075-2099.")
print("Monthly area-weighted ET values are used.")
print("Future totalET is converted from m to mm and optionally bias-corrected to CWatM.")


Loading CWatM historical data...

Opening historical CWatM
E:\ET_Article_With_MM_SA\Future data ST\ET_CWatM_mm_day.nc
Variable: ET_final_mm_day
Shape: (468, 13, 14)
Time: 1981-01 to 2019-12
Mean: 1.102419

Domain used for future clipping:
lat: 30.750 to 36.750
lon: 45.250 to 51.750

CWatM comparison period:
Requested: 1995-01-01 to 2020-12-01
Available: 1995-01-01 00:00:00 to 2019-12-01 00:00:00 | n months: 300

Processing scenario: 126 - Optimistic / SSP126

Reading future model: GFDL, scenario: 126

Reading future model: IPSL, scenario: 126

Reading future model: MPI, scenario: 126

Reading future model: MRI, scenario: 126

Reading future model: UKESM, scenario: 126

Creating boxplot: 126 - Near future

Creating boxplot: 126 - Mid-century

Creating boxplot: 126 - Far future

Processing scenario: 370 - Medium / SSP370

Reading future model: GFDL, scenario: 370

Reading future model: IPSL, scenario: 370

Reading future model: MPI, scenario: 370

Reading future model: MRI, scenario: 37

In [29]:
# -*- coding: utf-8 -*-

"""
ET Change Summary Tables - Ensemble Mean Only

This version:
    - Does NOT create separate tables for each climate model.
    - First calculates the mean of 5 future climate models:
        GFDL, IPSL, MPI, MRI, UKESM
    - Then compares the future ensemble mean with historical CWatM.
    - Outputs annual, seasonal, and monthly summary tables.
    - Percent change is calculated as:
        ((Future - Historical) / Historical) * 100
"""

import os
import re
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import xarray as xr


# ============================================================
# 1. PATH SETTINGS
# ============================================================

BASE_DIR = os.getcwd()

CWATM_FILE = os.path.join(BASE_DIR, "ET_CWatM_mm_day.nc")

FUTURE_BASE_DIR = BASE_DIR


# ============================================================
# 2. MODELS AND SCENARIOS
# ============================================================

CLIMATE_MODELS = [
    "GFDL",
    "IPSL",
    "MPI",
    "MRI",
    "UKESM",
]

SCENARIOS = {
    "126": {
        "short": "Optimistic",
        "label": "Optimistic (SSP1-2.6)",
    },
    "370": {
        "short": "Average",
        "label": "Average (SSP3-7.0)",
    },
    "585": {
        "short": "Pessimistic",
        "label": "Pessimistic (SSP5-8.5)",
    },
}

FUTURE_TOTAL_ET_FILE = "totalET_monthavg.nc"
FUTURE_TOTAL_ET_VAR = "totalET_monthavg"


# ============================================================
# 3. OUTPUT SETTINGS
# ============================================================

OUT_DIR = os.path.join(BASE_DIR, "outputs_ET_change_summary_tables_ensemble_mean")
TABLE_DIR = os.path.join(OUT_DIR, "tables")
PIVOT_DIR = os.path.join(TABLE_DIR, "pivot_tables")

for d in [OUT_DIR, TABLE_DIR, PIVOT_DIR]:
    os.makedirs(d, exist_ok=True)


# ============================================================
# 4. PERIOD SETTINGS
# ============================================================

HISTORICAL_START = "1995-01-01"
HISTORICAL_END = "2020-12-01"

FUTURE_PERIODS = {
    "near_future": {
        "label": "Near future",
        "start": "2025-01-01",
        "end": "2049-12-01",
    },
    "mid_century": {
        "label": "Mid-century",
        "start": "2050-01-01",
        "end": "2074-12-01",
    },
    "far_future": {
        "label": "Far future",
        "start": "2075-01-01",
        "end": "2099-12-01",
    },
}


# ============================================================
# 5. ANALYSIS SETTINGS
# ============================================================

HISTORICAL_UNIT = "mm/day"

FUTURE_CONVERT_M_TO_MM = True

MASK_NEGATIVE_HISTORICAL_ET = True
MASK_NEGATIVE_FUTURE_ET = True

APPLY_BIAS_CORRECTION = True
BIAS_METHOD = "multiplicative_mean_ratio"

MIN_MONTHS_PER_YEAR = 12
MIN_MONTHS_PER_SEASON = 3

SEASONS = {
    "Winter": [1, 2, 3],
    "Spring": [4, 5, 6],
    "Summer": [7, 8, 9],
    "Autumn": [10, 11, 12],
}

METRICS = ["Mean", "Max", "Min"]

MONTH_NAMES = {
    1: "January",
    2: "February",
    3: "March",
    4: "April",
    5: "May",
    6: "June",
    7: "July",
    8: "August",
    9: "September",
    10: "October",
    11: "November",
    12: "December",
}


# ============================================================
# 6. BASIC FUNCTIONS
# ============================================================

def safe_name(text):
    text = str(text)
    text = text.replace("/", "_")
    text = text.replace(" ", "_")
    text = text.replace("-", "_")
    text = re.sub(r"[^\w]", "_", text)
    text = re.sub(r"_+", "_", text)
    return text.strip("_")


def ensure_month_start_index(series_or_df):
    obj = series_or_df.copy()
    obj.index = pd.to_datetime(obj.index)
    obj.index = obj.index.to_period("M").to_timestamp()
    obj = obj.sort_index()
    return obj


def percent_change(future_value, historical_value):
    if pd.isna(future_value) or pd.isna(historical_value):
        return np.nan

    if historical_value == 0:
        return np.nan

    return ((future_value - historical_value) / historical_value) * 100.0


def absolute_change(future_value, historical_value):
    if pd.isna(future_value) or pd.isna(historical_value):
        return np.nan

    return future_value - historical_value


def subset_series(series, start_date, end_date):
    s = series.copy()
    s.index = pd.to_datetime(s.index)

    return s.loc[
        (s.index >= pd.Timestamp(start_date)) &
        (s.index <= pd.Timestamp(end_date))
    ].dropna()


# ============================================================
# 7. NETCDF READER FUNCTIONS
# ============================================================

def standardize_xy_to_latlon(da):
    rename_dict = {}

    for name in list(da.dims) + list(da.coords):
        low = str(name).lower()

        if low in ["x", "longitude"]:
            rename_dict[name] = "lon"

        elif low in ["y", "latitude"]:
            rename_dict[name] = "lat"

    if rename_dict:
        da = da.rename(rename_dict)

    if "lat" in da.coords:
        da = da.sortby("lat")

    if "lon" in da.coords:
        da = da.sortby("lon")

    return da


def open_cwatm_et(path):
    print("\n" + "=" * 80)
    print("Opening historical CWatM")
    print(path)
    print("=" * 80)

    if not os.path.exists(path):
        raise FileNotFoundError(path)

    ds = xr.open_dataset(path)

    candidate_vars = [
        v for v in ds.data_vars
        if str(v).lower() not in ["spatial_ref", "crs"]
    ]

    if len(candidate_vars) == 0:
        raise ValueError("No variable found in CWatM file.")

    if "ET_final_mm_day" in candidate_vars:
        var_name = "ET_final_mm_day"
    else:
        var_name = candidate_vars[0]

    da = ds[var_name]

    if "time" not in da.dims:
        raise ValueError("No time dimension found in CWatM file.")

    da["time"] = pd.to_datetime(da.time.values)

    da = standardize_xy_to_latlon(da)

    for d in ["time", "lat", "lon"]:
        if d not in da.dims:
            raise ValueError(f"CWatM required dimension '{d}' not found.")

    da = da.astype("float64")
    da = da.where(np.isfinite(da))

    if MASK_NEGATIVE_HISTORICAL_ET:
        da = da.where(da >= 0)

    da.name = "CWatM"
    da.attrs["unit"] = HISTORICAL_UNIT

    print(f"Variable: {var_name}")
    print(f"Shape: {da.shape}")
    print(
        "Time:",
        pd.to_datetime(da.time.values[0]).strftime("%Y-%m"),
        "to",
        pd.to_datetime(da.time.values[-1]).strftime("%Y-%m")
    )
    print(f"Mean: {float(da.mean(skipna=True)):.6f}")

    return da


def open_future_total_et(model_name, scenario, hist_domain):
    scen_dir = os.path.join(FUTURE_BASE_DIR, model_name, scenario)
    total_path = os.path.join(scen_dir, FUTURE_TOTAL_ET_FILE)

    if not os.path.exists(total_path):
        raise FileNotFoundError(total_path)

    ds = xr.open_dataset(total_path)

    candidate_vars = [
        v for v in ds.data_vars
        if str(v).lower() not in ["spatial_ref", "crs"]
    ]

    if FUTURE_TOTAL_ET_VAR in candidate_vars:
        var_name = FUTURE_TOTAL_ET_VAR

    elif len(candidate_vars) == 1:
        var_name = candidate_vars[0]

    else:
        raise ValueError(
            f"Cannot select future ET variable from {total_path}. "
            f"Candidates: {candidate_vars}"
        )

    da = ds[var_name]

    if "time" not in da.dims:
        raise ValueError(f"No time dimension found in {total_path}")

    da["time"] = pd.to_datetime(da.time.values)

    da = standardize_xy_to_latlon(da)

    for d in ["time", "lat", "lon"]:
        if d not in da.dims:
            raise ValueError(f"{model_name}/{scenario}: dimension '{d}' not found.")

    da = da.astype("float64")
    da = da.where(np.isfinite(da))

    da = clip_to_domain(da, hist_domain)

    if FUTURE_CONVERT_M_TO_MM:
        da = da * 1000.0

    if MASK_NEGATIVE_FUTURE_ET:
        da = da.where(da >= 0)

    da.name = f"{model_name}_{scenario}"

    return da


# ============================================================
# 8. AREA-WEIGHTED MEAN FUNCTIONS
# ============================================================

def infer_bounds_from_centers(values):
    values = np.asarray(values, dtype=float)

    if len(values) < 2:
        raise ValueError("At least two coordinate values are needed.")

    values_sorted = np.sort(values)
    mid = (values_sorted[:-1] + values_sorted[1:]) / 2.0

    first = values_sorted[0] - (mid[0] - values_sorted[0])
    last = values_sorted[-1] + (values_sorted[-1] - mid[-1])

    return np.concatenate([[first], mid, [last]])


def calculate_latlon_cell_area(lat, lon, radius=6371000.0):
    lat_values = np.asarray(lat.values if hasattr(lat, "values") else lat, dtype=float)
    lon_values = np.asarray(lon.values if hasattr(lon, "values") else lon, dtype=float)

    lat_ascending = np.all(np.diff(lat_values) > 0)
    lon_ascending = np.all(np.diff(lon_values) > 0)

    lat_sorted = np.sort(lat_values)
    lon_sorted = np.sort(lon_values)

    lat_bounds = infer_bounds_from_centers(lat_sorted)
    lon_bounds = infer_bounds_from_centers(lon_sorted)

    lat_bounds_rad = np.deg2rad(lat_bounds)
    lon_bounds_rad = np.deg2rad(lon_bounds)

    dlon = np.diff(lon_bounds_rad)
    sin_lat_diff = np.diff(np.sin(lat_bounds_rad))

    area_sorted = (radius ** 2) * sin_lat_diff[:, None] * dlon[None, :]
    area_sorted = np.abs(area_sorted)

    if not lat_ascending:
        area_sorted = area_sorted[::-1, :]

    if not lon_ascending:
        area_sorted = area_sorted[:, ::-1]

    area = xr.DataArray(
        area_sorted,
        dims=("lat", "lon"),
        coords={
            "lat": lat_values,
            "lon": lon_values,
        },
        name="cell_area"
    )

    area.attrs["units"] = "m2"

    return area


def area_weighted_mean(da):
    area = calculate_latlon_cell_area(da.lat, da.lon)
    valid = da.notnull()

    weighted_sum = (da * area).where(valid).sum(dim=("lat", "lon"), skipna=True)
    weight_sum = area.where(valid).sum(dim=("lat", "lon"), skipna=True)

    out = weighted_sum / weight_sum
    out.name = da.name

    return out


def to_monthly_area_series(da):
    s = area_weighted_mean(da).to_series()
    s = ensure_month_start_index(s)
    return s


# ============================================================
# 9. DOMAIN AND BIAS FUNCTIONS
# ============================================================

def get_domain_from_array(da):
    lat_min = float(da.lat.min())
    lat_max = float(da.lat.max())
    lon_min = float(da.lon.min())
    lon_max = float(da.lon.max())

    return lat_min, lat_max, lon_min, lon_max


def clip_to_domain(da, domain):
    lat_min, lat_max, lon_min, lon_max = domain

    da = da.sortby("lat")
    da = da.sortby("lon")

    return da.sel(
        lat=slice(lat_min, lat_max),
        lon=slice(lon_min, lon_max)
    )


def bias_correct_future_series(future_series, reference_series):
    df = pd.concat([future_series, reference_series], axis=1).dropna()
    df.columns = ["future", "reference"]

    n_overlap = len(df)

    if n_overlap < 12:
        return future_series, np.nan, n_overlap

    future_mean = df["future"].mean()
    reference_mean = df["reference"].mean()

    if not np.isfinite(future_mean) or future_mean == 0:
        return future_series, np.nan, n_overlap

    factor = reference_mean / future_mean
    corrected = future_series * factor
    corrected.name = future_series.name

    return corrected, factor, n_overlap


# ============================================================
# 10. METRIC CALCULATION FUNCTIONS
# ============================================================

def calculate_annual_period_metrics(series, start_date, end_date):
    s = subset_series(series, start_date, end_date)

    if len(s) == 0:
        return {"Mean": np.nan, "Max": np.nan, "Min": np.nan, "n_units": 0}

    df = s.to_frame("ET")
    df["year"] = df.index.year

    yearly_count = df.groupby("year")["ET"].count()
    valid_years = yearly_count[yearly_count >= MIN_MONTHS_PER_YEAR].index

    df = df[df["year"].isin(valid_years)]

    if df.empty:
        return {"Mean": np.nan, "Max": np.nan, "Min": np.nan, "n_units": 0}

    annual_stats = df.groupby("year")["ET"].agg(
        annual_mean="mean",
        annual_max="max",
        annual_min="min"
    )

    return {
        "Mean": annual_stats["annual_mean"].mean(),
        "Max": annual_stats["annual_max"].mean(),
        "Min": annual_stats["annual_min"].mean(),
        "n_units": int(annual_stats.shape[0]),
    }


def calculate_monthly_period_metrics(series, start_date, end_date):
    s = subset_series(series, start_date, end_date)

    output = {}

    for month_num, month_name in MONTH_NAMES.items():
        sm = s[s.index.month == month_num].dropna()

        if len(sm) == 0:
            output[month_name] = {
                "Mean": np.nan,
                "Max": np.nan,
                "Min": np.nan,
                "n_units": 0,
            }
        else:
            output[month_name] = {
                "Mean": sm.mean(),
                "Max": sm.max(),
                "Min": sm.min(),
                "n_units": int(sm.shape[0]),
            }

    return output


def calculate_seasonal_period_metrics(series, start_date, end_date):
    s = subset_series(series, start_date, end_date)

    output = {}

    for season_name, months in SEASONS.items():

        records = []

        for year in sorted(s.index.year.unique()):
            sy = s[
                (s.index.year == year) &
                (s.index.month.isin(months))
            ].dropna()

            if len(sy) >= MIN_MONTHS_PER_SEASON:
                records.append({
                    "year": year,
                    "seasonal_mean": sy.mean(),
                    "seasonal_max": sy.max(),
                    "seasonal_min": sy.min(),
                })

        if len(records) == 0:
            output[season_name] = {
                "Mean": np.nan,
                "Max": np.nan,
                "Min": np.nan,
                "n_units": 0,
            }
        else:
            sdf = pd.DataFrame(records)

            output[season_name] = {
                "Mean": sdf["seasonal_mean"].mean(),
                "Max": sdf["seasonal_max"].mean(),
                "Min": sdf["seasonal_min"].mean(),
                "n_units": int(sdf.shape[0]),
            }

    return output


# ============================================================
# 11. COMPARISON FUNCTIONS
# ============================================================

def add_comparison_records(
    records,
    scale,
    unit_name,
    scenario_code,
    scenario_short,
    scenario_label,
    future_period_key,
    future_period_label,
    future_start,
    future_end,
    historical_metrics,
    future_metrics
):
    for metric in METRICS:
        hist_val = historical_metrics.get(metric, np.nan)
        fut_val = future_metrics.get(metric, np.nan)

        records.append({
            "scale": scale,
            "unit": unit_name,
            "metric": metric,
            "scenario_code": scenario_code,
            "scenario": scenario_short,
            "scenario_label": scenario_label,
            "future_period": future_period_key,
            "future_period_label": future_period_label,
            "future_start": future_start,
            "future_end": future_end,
            "future_source": "EnsembleMean_5_models",
            "historical_reference": "CWatM",
            "historical_start": HISTORICAL_START,
            "historical_end": HISTORICAL_END,
            "historical_value": hist_val,
            "future_value": fut_val,
            "absolute_change": absolute_change(fut_val, hist_val),
            "percent_change": percent_change(fut_val, hist_val),
            "historical_n_units": historical_metrics.get("n_units", np.nan),
            "future_n_units": future_metrics.get("n_units", np.nan),
        })


def build_comparisons_for_ensemble(
    records,
    ensemble_series,
    scenario_code,
    scenario_info,
    future_period_key,
    future_period_info,
    hist_annual_metrics,
    hist_monthly_metrics,
    hist_seasonal_metrics
):
    scenario_short = scenario_info["short"]
    scenario_label = scenario_info["label"]

    future_start = future_period_info["start"]
    future_end = future_period_info["end"]
    future_period_label = future_period_info["label"]

    future_annual_metrics = calculate_annual_period_metrics(
        ensemble_series,
        future_start,
        future_end
    )

    add_comparison_records(
        records=records,
        scale="Annual",
        unit_name="Annual",
        scenario_code=scenario_code,
        scenario_short=scenario_short,
        scenario_label=scenario_label,
        future_period_key=future_period_key,
        future_period_label=future_period_label,
        future_start=future_start,
        future_end=future_end,
        historical_metrics=hist_annual_metrics,
        future_metrics=future_annual_metrics
    )

    future_monthly_metrics = calculate_monthly_period_metrics(
        ensemble_series,
        future_start,
        future_end
    )

    for month_name in MONTH_NAMES.values():
        add_comparison_records(
            records=records,
            scale="Monthly",
            unit_name=month_name,
            scenario_code=scenario_code,
            scenario_short=scenario_short,
            scenario_label=scenario_label,
            future_period_key=future_period_key,
            future_period_label=future_period_label,
            future_start=future_start,
            future_end=future_end,
            historical_metrics=hist_monthly_metrics[month_name],
            future_metrics=future_monthly_metrics[month_name]
        )

    future_seasonal_metrics = calculate_seasonal_period_metrics(
        ensemble_series,
        future_start,
        future_end
    )

    for season_name in SEASONS.keys():
        add_comparison_records(
            records=records,
            scale="Seasonal",
            unit_name=season_name,
            scenario_code=scenario_code,
            scenario_short=scenario_short,
            scenario_label=scenario_label,
            future_period_key=future_period_key,
            future_period_label=future_period_label,
            future_start=future_start,
            future_end=future_end,
            historical_metrics=hist_seasonal_metrics[season_name],
            future_metrics=future_seasonal_metrics[season_name]
        )


# ============================================================
# 12. PIVOT TABLE FUNCTIONS
# ============================================================

def make_percent_pivot(long_df, scale, future_period_key):
    sub = long_df[
        (long_df["scale"] == scale) &
        (long_df["future_period"] == future_period_key)
    ].copy()

    if sub.empty:
        return pd.DataFrame()

    metric_label_map = {
        "Mean": "Mean%",
        "Max": "Max%",
        "Min": "Min%",
    }

    sub["metric_label"] = sub["metric"].map(metric_label_map)

    pivot = sub.pivot_table(
        index="unit",
        columns=["scenario", "metric_label"],
        values="percent_change",
        aggfunc="first"
    )

    scenario_order = ["Optimistic", "Average", "Pessimistic"]
    metric_order = ["Mean%", "Max%", "Min%"]

    ordered_columns = []

    for scen in scenario_order:
        for metric in metric_order:
            if (scen, metric) in pivot.columns:
                ordered_columns.append((scen, metric))

    pivot = pivot.loc[:, ordered_columns]

    if scale == "Monthly":
        pivot = pivot.reindex(list(MONTH_NAMES.values()))

    elif scale == "Seasonal":
        pivot = pivot.reindex(list(SEASONS.keys()))

    elif scale == "Annual":
        pivot = pivot.reindex(["Annual"])

    pivot = pivot.round(2)

    return pivot


def make_value_pivot(long_df, scale, future_period_key, value_column):
    sub = long_df[
        (long_df["scale"] == scale) &
        (long_df["future_period"] == future_period_key)
    ].copy()

    if sub.empty:
        return pd.DataFrame()

    pivot = sub.pivot_table(
        index="unit",
        columns=["scenario", "metric"],
        values=value_column,
        aggfunc="first"
    )

    scenario_order = ["Optimistic", "Average", "Pessimistic"]
    metric_order = ["Mean", "Max", "Min"]

    ordered_columns = []

    for scen in scenario_order:
        for metric in metric_order:
            if (scen, metric) in pivot.columns:
                ordered_columns.append((scen, metric))

    pivot = pivot.loc[:, ordered_columns]

    if scale == "Monthly":
        pivot = pivot.reindex(list(MONTH_NAMES.values()))

    elif scale == "Seasonal":
        pivot = pivot.reindex(list(SEASONS.keys()))

    elif scale == "Annual":
        pivot = pivot.reindex(["Annual"])

    pivot = pivot.round(4)

    return pivot


def save_pivot_tables(long_df, output_dir):
    os.makedirs(output_dir, exist_ok=True)

    scales = ["Annual", "Seasonal", "Monthly"]

    output_excel = os.path.join(
        output_dir,
        "ET_change_pivot_tables_ensemble_mean.xlsx"
    )

    with pd.ExcelWriter(output_excel, engine="openpyxl") as writer:

        for period_key in FUTURE_PERIODS.keys():
            for scale in scales:

                percent_pivot = make_percent_pivot(
                    long_df=long_df,
                    scale=scale,
                    future_period_key=period_key
                )

                if not percent_pivot.empty:
                    sheet_name = f"{scale[:3]}_{period_key[:10]}_pct"
                    sheet_name = sheet_name[:31]
                    percent_pivot.to_excel(writer, sheet_name=sheet_name)

                    csv_name = f"percent_{scale}_{period_key}_EnsembleMean.csv"
                    percent_pivot.to_csv(
                        os.path.join(output_dir, csv_name),
                        encoding="utf-8-sig"
                    )

                for value_column in [
                    "historical_value",
                    "future_value",
                    "absolute_change",
                ]:
                    pivot = make_value_pivot(
                        long_df=long_df,
                        scale=scale,
                        future_period_key=period_key,
                        value_column=value_column
                    )

                    if not pivot.empty:
                        csv_name = f"{value_column}_{scale}_{period_key}_EnsembleMean.csv"
                        pivot.to_csv(
                            os.path.join(output_dir, csv_name),
                            encoding="utf-8-sig"
                        )

    return output_excel


# ============================================================
# 13. MAIN WORKFLOW
# ============================================================

print("\nLoading CWatM historical data...")

cwatm_da = open_cwatm_et(CWATM_FILE)

hist_domain = get_domain_from_array(cwatm_da)

print("\nDomain used for future clipping:")
print(f"lat: {hist_domain[0]:.3f} to {hist_domain[1]:.3f}")
print(f"lon: {hist_domain[2]:.3f} to {hist_domain[3]:.3f}")

cwatm_series_all = to_monthly_area_series(cwatm_da)
cwatm_series_all.name = "CWatM"

cwatm_hist_series = subset_series(
    cwatm_series_all,
    HISTORICAL_START,
    HISTORICAL_END
)

cwatm_hist_csv = os.path.join(TABLE_DIR, "CWatM_historical_monthly_reference.csv")
cwatm_hist_series.to_frame(name="CWatM").to_csv(cwatm_hist_csv, encoding="utf-8-sig")

hist_annual_metrics = calculate_annual_period_metrics(
    cwatm_series_all,
    HISTORICAL_START,
    HISTORICAL_END
)

hist_monthly_metrics = calculate_monthly_period_metrics(
    cwatm_series_all,
    HISTORICAL_START,
    HISTORICAL_END
)

hist_seasonal_metrics = calculate_seasonal_period_metrics(
    cwatm_series_all,
    HISTORICAL_START,
    HISTORICAL_END
)


all_records = []
bias_records = []
ensemble_monthly_records = []


for scenario_code, scenario_info in SCENARIOS.items():

    print("\n" + "=" * 80)
    print(f"Processing scenario: {scenario_code} - {scenario_info['label']}")
    print("=" * 80)

    model_series_dict = {}

    for model in CLIMATE_MODELS:

        print(f"\nReading future model: {model}, scenario: {scenario_code}")

        try:
            future_da = open_future_total_et(
                model_name=model,
                scenario=scenario_code,
                hist_domain=hist_domain
            )

            future_series_raw = to_monthly_area_series(future_da)
            future_series_raw.name = model

            if APPLY_BIAS_CORRECTION:
                future_series_used, factor, n_overlap = bias_correct_future_series(
                    future_series=future_series_raw,
                    reference_series=cwatm_series_all
                )
            else:
                future_series_used = future_series_raw
                factor = np.nan
                n_overlap = 0

            future_series_used.name = model
            model_series_dict[model] = future_series_used

            bias_records.append({
                "scenario_code": scenario_code,
                "scenario": scenario_info["short"],
                "scenario_label": scenario_info["label"],
                "model": model,
                "apply_bias_correction": APPLY_BIAS_CORRECTION,
                "bias_method": BIAS_METHOD if APPLY_BIAS_CORRECTION else "none",
                "bias_factor": factor,
                "n_overlap_months": n_overlap,
            })

        except Exception as e:
            print(f"ERROR for {model} / {scenario_code}: {e}")

    if len(model_series_dict) == 0:
        print(f"No valid future models found for scenario {scenario_code}.")
        continue

    future_models_df = pd.concat(model_series_dict.values(), axis=1)
    future_models_df.columns = list(model_series_dict.keys())

    ensemble_series = future_models_df.mean(axis=1, skipna=True)
    ensemble_series.name = "EnsembleMean_5_models"

    temp_ens = ensemble_series.to_frame(name="ET")
    temp_ens["scenario_code"] = scenario_code
    temp_ens["scenario"] = scenario_info["short"]
    temp_ens["scenario_label"] = scenario_info["label"]
    temp_ens = temp_ens.reset_index().rename(columns={"index": "time"})
    ensemble_monthly_records.append(temp_ens)

    for future_period_key, future_period_info in FUTURE_PERIODS.items():

        print(f"\nCalculating ensemble comparisons for period: {future_period_info['label']}")

        build_comparisons_for_ensemble(
            records=all_records,
            ensemble_series=ensemble_series,
            scenario_code=scenario_code,
            scenario_info=scenario_info,
            future_period_key=future_period_key,
            future_period_info=future_period_info,
            hist_annual_metrics=hist_annual_metrics,
            hist_monthly_metrics=hist_monthly_metrics,
            hist_seasonal_metrics=hist_seasonal_metrics
        )


# ============================================================
# 14. SAVE OUTPUT TABLES
# ============================================================

long_df = pd.DataFrame(all_records)

long_df = long_df[
    [
        "scale",
        "unit",
        "metric",
        "scenario_code",
        "scenario",
        "scenario_label",
        "future_period",
        "future_period_label",
        "future_start",
        "future_end",
        "future_source",
        "historical_reference",
        "historical_start",
        "historical_end",
        "historical_value",
        "future_value",
        "absolute_change",
        "percent_change",
        "historical_n_units",
        "future_n_units",
    ]
]

long_csv = os.path.join(TABLE_DIR, "ET_change_summary_long_ensemble_mean.csv")
long_df.to_csv(long_csv, index=False, encoding="utf-8-sig")

long_excel = os.path.join(TABLE_DIR, "ET_change_summary_long_ensemble_mean.xlsx")
long_df.to_excel(long_excel, index=False)

bias_df = pd.DataFrame(bias_records)
bias_csv = os.path.join(TABLE_DIR, "future_bias_correction_factors_5_models.csv")
bias_df.to_csv(bias_csv, index=False, encoding="utf-8-sig")

if len(ensemble_monthly_records) > 0:
    ensemble_monthly_df = pd.concat(ensemble_monthly_records, axis=0)

    ensemble_monthly_csv = os.path.join(
        TABLE_DIR,
        "future_ensemble_monthly_area_mean_ET_all_scenarios.csv"
    )

    ensemble_monthly_df.to_csv(
        ensemble_monthly_csv,
        index=False,
        encoding="utf-8-sig"
    )
else:
    ensemble_monthly_csv = None


pivot_excel = save_pivot_tables(
    long_df=long_df,
    output_dir=PIVOT_DIR
)


config_df = pd.DataFrame([
    {"parameter": "analysis_type", "value": "ET_change_summary_tables_ensemble_mean_only"},
    {"parameter": "historical_reference", "value": "CWatM"},
    {"parameter": "historical_start", "value": HISTORICAL_START},
    {"parameter": "historical_end", "value": HISTORICAL_END},
    {"parameter": "future_models_used_for_ensemble", "value": ", ".join(CLIMATE_MODELS)},
    {"parameter": "future_source", "value": "Mean of 5 climate models"},
    {"parameter": "scenarios", "value": ", ".join(SCENARIOS.keys())},
    {"parameter": "future_total_et_file", "value": FUTURE_TOTAL_ET_FILE},
    {"parameter": "future_total_et_var", "value": FUTURE_TOTAL_ET_VAR},
    {"parameter": "future_convert_m_to_mm", "value": FUTURE_CONVERT_M_TO_MM},
    {"parameter": "apply_bias_correction", "value": APPLY_BIAS_CORRECTION},
    {"parameter": "bias_method", "value": BIAS_METHOD},
    {"parameter": "percent_change_formula", "value": "((Future - Historical) / Historical) * 100"},
    {"parameter": "min_months_per_year", "value": MIN_MONTHS_PER_YEAR},
    {"parameter": "min_months_per_season", "value": MIN_MONTHS_PER_SEASON},
    {"parameter": "season_definition", "value": str(SEASONS)},
    {"parameter": "domain_lat_min", "value": hist_domain[0]},
    {"parameter": "domain_lat_max", "value": hist_domain[1]},
    {"parameter": "domain_lon_min", "value": hist_domain[2]},
    {"parameter": "domain_lon_max", "value": hist_domain[3]},
])

for key, info in FUTURE_PERIODS.items():
    config_df = pd.concat(
        [
            config_df,
            pd.DataFrame([
                {"parameter": f"{key}_label", "value": info["label"]},
                {"parameter": f"{key}_start", "value": info["start"]},
                {"parameter": f"{key}_end", "value": info["end"]},
            ])
        ],
        ignore_index=True
    )

config_csv = os.path.join(TABLE_DIR, "run_configuration_ET_change_summary_ensemble_mean.csv")
config_df.to_csv(config_csv, index=False, encoding="utf-8-sig")


# ============================================================
# 15. FINAL REPORT
# ============================================================

print("\n" + "=" * 80)
print("DONE")
print("=" * 80)

print("\nMain output folder:")
print(OUT_DIR)

print("\nMain long tables:")
print(long_csv)
print(long_excel)

print("\nPivot tables:")
print(pivot_excel)

print("\nPivot folder:")
print(PIVOT_DIR)

print("\nBias correction table:")
print(bias_csv)

print("\nHistorical monthly reference:")
print(cwatm_hist_csv)

if ensemble_monthly_csv is not None:
    print("\nFuture ensemble monthly ET:")
    print(ensemble_monthly_csv)

print("\nConfiguration:")
print(config_csv)

print("\nNotes:")
print("Only the ensemble mean of the 5 climate models is compared with historical CWatM.")
print("No separate comparison table is produced for individual climate models.")
print("Percent change = ((Future - Historical) / Historical) * 100.")
print("Negative values indicate decrease relative to historical CWatM.")
print("Values greater than 100 indicate that the increase is larger than the historical reference value.")


Loading CWatM historical data...

Opening historical CWatM
E:\ET_Article_With_MM_SA\Future data ST\ET_CWatM_mm_day.nc
Variable: ET_final_mm_day
Shape: (468, 13, 14)
Time: 1981-01 to 2019-12
Mean: 1.102419

Domain used for future clipping:
lat: 30.750 to 36.750
lon: 45.250 to 51.750

Processing scenario: 126 - Optimistic (SSP1-2.6)

Reading future model: GFDL, scenario: 126

Reading future model: IPSL, scenario: 126

Reading future model: MPI, scenario: 126

Reading future model: MRI, scenario: 126

Reading future model: UKESM, scenario: 126

Calculating ensemble comparisons for period: Near future

Calculating ensemble comparisons for period: Mid-century

Calculating ensemble comparisons for period: Far future

Processing scenario: 370 - Average (SSP3-7.0)

Reading future model: GFDL, scenario: 370

Reading future model: IPSL, scenario: 370

Reading future model: MPI, scenario: 370

Reading future model: MRI, scenario: 370

Reading future model: UKESM, scenario: 370

Calculating ensem

In [4]:
# -*- coding: utf-8 -*-

"""
Annual Historical-Future ET Time Series
using the sum of three future ET components

Future ET:
    totalET_monthavg
    + EvapWaterBodyM_monthavg
    + EvapoChannel_monthavg converted from m3 to m

Unit conversion:
    totalET_monthavg        : m
    EvapWaterBodyM_monthavg : m
    EvapoChannel_monthavg   : m3

    EvapoChannel depth (m) =
        EvapoChannel volume (m3) / grid-cell area (m2)

    Combined ET (mm) =
        (
            totalET_monthavg
            + EvapWaterBodyM_monthavg
            + EvapoChannel_monthavg / cell_area
        ) * 1000

Historical datasets:
    CWatM
    GLDAS
    SMAP

Historical product correction:
    GLDAS and SMAP are corrected relative to CWatM
    using a constant additive mean bias.

Future models:
    GFDL, IPSL, MPI, MRI, UKESM

Scenarios:
    126, 370, 585

The plotting details are kept unchanged:
    - Annual mean ET
    - Original GLDAS/SMAP: solid lines
    - Bias-corrected GLDAS/SMAP: dashed lines
    - Future models: original colors
    - Future ensemble mean: red
    - Legend: lower right
"""

import os
import re
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt


# ============================================================
# 1. PATH SETTINGS
# ============================================================

BASE_DIR = os.getcwd()

CWATM_FILE = os.path.join(
    BASE_DIR,
    "ET_CWatM_mm_day.nc"
)

GLDAS_FILE = os.path.join(
    BASE_DIR,
    "GLDAS_ET_2000_2019.nc"
)

SMAP_FILE = os.path.join(
    BASE_DIR,
    "SMAP_ET_2000_2019.nc"
)

FUTURE_BASE_DIR = BASE_DIR


# ============================================================
# 2. MODELS AND SCENARIOS
# ============================================================

CLIMATE_MODELS = [
    "GFDL",
    "IPSL",
    "MPI",
    "MRI",
    "UKESM",
]

SCENARIOS = {
    "126": "Optimistic / SSP126",
    "370": "Medium / SSP370",
    "585": "Pessimistic / SSP585",
}


# ============================================================
# 3. FUTURE COMPONENT FILES AND VARIABLES
# ============================================================

FUTURE_TOTAL_ET_FILE = "totalET_monthavg.nc"
FUTURE_TOTAL_ET_VAR = "totalET_monthavg"

FUTURE_WATER_BODY_FILE = "EvapWaterBodyM_monthavg.nc"
FUTURE_WATER_BODY_VAR = "EvapWaterBodyM_monthavg"

FUTURE_CHANNEL_FILE = "EvapoChannel_monthavg.nc"
FUTURE_CHANNEL_VAR = "EvapoChannel_monthavg"

FUTURE_ET_MODE = "sum_of_three_components"


# ============================================================
# 4. OUTPUT SETTINGS
# ============================================================

OUT_DIR = os.path.join(
    BASE_DIR,
    "outputs_historical_future_ET_annual_three_components_test"
)

TABLE_DIR = os.path.join(
    OUT_DIR,
    "tables"
)

FIG_DIR = os.path.join(
    OUT_DIR,
    "figures"
)

for directory in [
    OUT_DIR,
    TABLE_DIR,
    FIG_DIR,
]:
    os.makedirs(
        directory,
        exist_ok=True
    )


# ============================================================
# 5. ANALYSIS SETTINGS
# ============================================================

PLOT_START_DATE = "2000-01-01"
PLOT_END_DATE = "2100-01-01"

HISTORICAL_START_DATE = "2000-01-01"
HISTORICAL_END_DATE = "2019-12-01"

FUTURE_START_DATE = "2020-01-01"
FUTURE_END_DATE = "2099-12-01"

HISTORICAL_UNIT = "mm/day"

MASK_NEGATIVE_HISTORICAL_ET = True
MASK_NEGATIVE_FUTURE_ET = True

# Future climate-model bias correction
APPLY_FUTURE_BIAS_CORRECTION = True
FUTURE_BIAS_REFERENCE = "CWatM"
FUTURE_BIAS_METHOD = "multiplicative_mean_ratio"

# Historical GLDAS/SMAP bias correction
APPLY_HISTORICAL_PRODUCT_BIAS_CORRECTION = True
HISTORICAL_BIAS_REFERENCE = "CWatM"
HISTORICAL_BIAS_METHOD = "additive_mean_bias"

HISTORICAL_PRODUCTS_TO_CORRECT = [
    "GLDAS",
    "SMAP",
]

MIN_OVERLAP_MONTHS_FOR_HISTORICAL_BIAS = 12

SAVE_MONTHLY_TABLES = True
SAVE_ANNUAL_TABLES = True
SAVE_COMPONENT_TABLES = True

# Monthly edge corrections before annual averaging
CORRECT_MONTHLY_EDGE_VALUES = True

HISTORICAL_FIXED_COLUMNS = [
    "CWatM",
    "GLDAS",
]

HISTORICAL_TARGET_DATE = "2019-12-01"

HISTORICAL_SOURCE_DATES = [
    "2019-09-01",
    "2019-10-01",
    "2019-11-01",
]

FUTURE_FIRST_TARGET_DATE = "2020-01-01"

FUTURE_FIRST_SOURCE_DATES = [
    "2020-02-01",
    "2020-03-01",
    "2020-04-01",
]

FUTURE_LAST_TARGET_DATE = "2099-12-01"

FUTURE_LAST_SOURCE_DATES = [
    "2099-09-01",
    "2099-10-01",
    "2099-11-01",
]


# ============================================================
# 6. FONT AND FIGURE SETTINGS
# ============================================================

FONT_FAMILY = "DejaVu Sans"

TITLE_FONT_SIZE = 20
AXIS_LABEL_FONT_SIZE = 18
TICK_FONT_SIZE = 15
LEGEND_FONT_SIZE = 11
PERIOD_LABEL_FONT_SIZE = 14

FIG_WIDTH = 18
FIG_HEIGHT = 7

LINE_WIDTH_HISTORICAL = 1.2
LINE_WIDTH_HISTORICAL_CORRECTED = 1.8
LINE_WIDTH_FUTURE_MODEL = 1.0
LINE_WIDTH_ENSEMBLE = 2.5

MARKER_SIZE_HISTORICAL = 4
MARKER_SIZE_HISTORICAL_CORRECTED = 3
MARKER_SIZE_FUTURE = 3
MARKER_SIZE_ENSEMBLE = 4

ALPHA_FUTURE_MODEL = 0.45

plt.rcParams["font.family"] = FONT_FAMILY
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.dpi"] = 150
plt.rcParams["axes.titlesize"] = TITLE_FONT_SIZE
plt.rcParams["axes.labelsize"] = AXIS_LABEL_FONT_SIZE
plt.rcParams["xtick.labelsize"] = TICK_FONT_SIZE
plt.rcParams["ytick.labelsize"] = TICK_FONT_SIZE
plt.rcParams["legend.fontsize"] = LEGEND_FONT_SIZE


# ============================================================
# 7. BASIC FUNCTIONS
# ============================================================

def ensure_month_start_index(series_or_df):
    obj = series_or_df.copy()

    obj.index = pd.to_datetime(obj.index)
    obj.index = obj.index.to_period("M").to_timestamp()

    return obj.sort_index()


def monthly_to_annual_mean(series_or_df):
    """
    Convert monthly data to annual means.

    The annual timestamp is January 1.
    """

    obj = series_or_df.copy()
    obj.index = pd.to_datetime(obj.index)

    return obj.resample("YS").mean()


def get_y_label():
    return "Annual mean daily ET (mm/day)"


def replace_target_month_with_mean(
    series,
    target_date,
    source_dates
):
    """
    Replace one target month with the mean of selected months.
    """

    output = series.copy()

    target_date = (
        pd.Timestamp(target_date)
        .to_period("M")
        .to_timestamp()
    )

    source_dates = [
        pd.Timestamp(date)
        .to_period("M")
        .to_timestamp()
        for date in source_dates
    ]

    if target_date not in output.index:
        return (
            output,
            np.nan,
            np.nan,
            "target_date_not_found"
        )

    available_dates = [
        date
        for date in source_dates
        if date in output.index
    ]

    if len(available_dates) == 0:
        return (
            output,
            output.loc[target_date],
            np.nan,
            "no_source_dates_found"
        )

    source_values = (
        output.loc[available_dates]
        .dropna()
    )

    if len(source_values) == 0:
        return (
            output,
            output.loc[target_date],
            np.nan,
            "source_values_all_nan"
        )

    original_value = output.loc[target_date]
    replacement_value = source_values.mean()

    output.loc[target_date] = replacement_value

    return (
        output,
        original_value,
        replacement_value,
        "replaced"
    )


# ============================================================
# 8. COORDINATE AND VARIABLE FUNCTIONS
# ============================================================

def extract_dates_from_long_name(long_name_attr):
    if isinstance(long_name_attr, str):
        long_names = [long_name_attr]
    else:
        long_names = list(long_name_attr)

    dates = []

    for item in long_names:
        item = str(item)

        match = re.search(
            r"(\d{4})_(\d{2})",
            item
        )

        if match is None:
            raise ValueError(
                f"Cannot extract date from long_name item: {item}"
            )

        dates.append(
            pd.Timestamp(
                year=int(match.group(1)),
                month=int(match.group(2)),
                day=1
            )
        )

    return pd.DatetimeIndex(dates)


def standardize_xy_to_latlon(da):
    rename_dict = {}

    for name in list(da.dims) + list(da.coords):
        name_lower = str(name).lower()

        if name_lower in [
            "x",
            "longitude",
        ]:
            rename_dict[name] = "lon"

        elif name_lower in [
            "y",
            "latitude",
        ]:
            rename_dict[name] = "lat"

    if rename_dict:
        da = da.rename(rename_dict)

    if "lat" in da.coords:
        da = da.sortby("lat")

    if "lon" in da.coords:
        da = da.sortby("lon")

    return da


def select_data_variable(
    ds,
    preferred_variable,
    file_path
):
    candidates = [
        variable
        for variable in ds.data_vars
        if str(variable).lower()
        not in [
            "spatial_ref",
            "crs",
        ]
    ]

    if preferred_variable in candidates:
        return preferred_variable

    if len(candidates) == 1:
        return candidates[0]

    raise ValueError(
        f"Cannot select variable from {file_path}. "
        f"Preferred variable: {preferred_variable}. "
        f"Candidates: {candidates}"
    )


# ============================================================
# 9. HISTORICAL DATA READER
# ============================================================

def open_historical_et(
    path,
    dataset_name
):
    print("\n" + "=" * 80)
    print(
        f"Opening historical dataset: {dataset_name}"
    )
    print(path)
    print("=" * 80)

    if not os.path.exists(path):
        raise FileNotFoundError(path)

    ds = xr.open_dataset(path)

    candidate_vars = [
        variable
        for variable in ds.data_vars
        if str(variable).lower()
        not in [
            "spatial_ref",
            "crs",
        ]
    ]

    if len(candidate_vars) == 0:
        raise ValueError(
            f"No variable found in {dataset_name}"
        )

    if "ET_final_mm_day" in candidate_vars:
        variable_name = "ET_final_mm_day"

    elif "__xarray_dataarray_variable__" in candidate_vars:
        variable_name = "__xarray_dataarray_variable__"

    else:
        variable_name = candidate_vars[0]

    da = ds[variable_name]

    if "time" in da.dims:
        da["time"] = pd.to_datetime(
            da.time.values
        )

    elif "band" in da.dims:
        long_name = da.attrs.get(
            "long_name",
            None
        )

        if long_name is None:
            raise ValueError(
                f"{dataset_name} has band dimension "
                f"but no long_name attribute."
            )

        dates = extract_dates_from_long_name(
            long_name
        )

        if len(dates) != da.sizes["band"]:
            raise ValueError(
                f"{dataset_name}: extracted dates "
                f"do not match band size."
            )

        da = da.rename({
            "band": "time"
        })

        da = da.assign_coords(
            time=dates
        )

    else:
        raise ValueError(
            f"No time or band dimension found in {dataset_name}"
        )

    da = standardize_xy_to_latlon(da)

    for dimension in [
        "time",
        "lat",
        "lon",
    ]:
        if dimension not in da.dims:
            raise ValueError(
                f"{dataset_name}: required dimension "
                f"'{dimension}' not found."
            )

    da = da.astype("float64")
    da = da.where(np.isfinite(da))

    if MASK_NEGATIVE_HISTORICAL_ET:
        da = da.where(da >= 0)

    da.name = dataset_name
    da.attrs["unit"] = HISTORICAL_UNIT

    print(f"Variable: {variable_name}")
    print(f"Shape: {da.shape}")

    print(
        "Time:",
        pd.to_datetime(
            da.time.values[0]
        ).strftime("%Y-%m"),
        "to",
        pd.to_datetime(
            da.time.values[-1]
        ).strftime("%Y-%m")
    )

    print(
        f"Mean: "
        f"{float(da.mean(skipna=True)):.6f}"
    )

    return da


# ============================================================
# 10. CELL AREA FUNCTIONS
# ============================================================

def infer_bounds_from_centers(values):
    values = np.asarray(
        values,
        dtype=float
    )

    if len(values) < 2:
        raise ValueError(
            "At least two coordinate values are required."
        )

    values_sorted = np.sort(values)

    midpoint = (
        values_sorted[:-1] +
        values_sorted[1:]
    ) / 2.0

    first = (
        values_sorted[0] -
        (
            midpoint[0] -
            values_sorted[0]
        )
    )

    last = (
        values_sorted[-1] +
        (
            values_sorted[-1] -
            midpoint[-1]
        )
    )

    return np.concatenate([
        [first],
        midpoint,
        [last],
    ])


def calculate_latlon_cell_area(
    lat,
    lon,
    radius=6371000.0
):
    """
    Calculate regular latitude-longitude cell areas in m2.
    """

    lat_values = np.asarray(
        lat.values
        if hasattr(lat, "values")
        else lat,
        dtype=float
    )

    lon_values = np.asarray(
        lon.values
        if hasattr(lon, "values")
        else lon,
        dtype=float
    )

    lat_ascending = np.all(
        np.diff(lat_values) > 0
    )

    lon_ascending = np.all(
        np.diff(lon_values) > 0
    )

    lat_sorted = np.sort(lat_values)
    lon_sorted = np.sort(lon_values)

    lat_bounds = infer_bounds_from_centers(
        lat_sorted
    )

    lon_bounds = infer_bounds_from_centers(
        lon_sorted
    )

    lat_bounds_rad = np.deg2rad(
        lat_bounds
    )

    lon_bounds_rad = np.deg2rad(
        lon_bounds
    )

    longitude_width = np.diff(
        lon_bounds_rad
    )

    latitude_factor = np.diff(
        np.sin(lat_bounds_rad)
    )

    area_sorted = (
        radius ** 2
        * latitude_factor[:, None]
        * longitude_width[None, :]
    )

    area_sorted = np.abs(
        area_sorted
    )

    if not lat_ascending:
        area_sorted = area_sorted[::-1, :]

    if not lon_ascending:
        area_sorted = area_sorted[:, ::-1]

    area = xr.DataArray(
        area_sorted,
        dims=(
            "lat",
            "lon",
        ),
        coords={
            "lat": lat_values,
            "lon": lon_values,
        },
        name="cell_area"
    )

    area.attrs["units"] = "m2"

    return area


def area_weighted_mean(da):
    area = calculate_latlon_cell_area(
        da.lat,
        da.lon
    )

    valid = da.notnull()

    weighted_sum = (
        (da * area)
        .where(valid)
        .sum(
            dim=(
                "lat",
                "lon",
            ),
            skipna=True
        )
    )

    valid_area = (
        area
        .where(valid)
        .sum(
            dim=(
                "lat",
                "lon",
            ),
            skipna=True
        )
    )

    result = weighted_sum / valid_area
    result.name = da.name

    return result


def to_monthly_area_series(da):
    series = (
        area_weighted_mean(da)
        .to_series()
    )

    return ensure_month_start_index(
        series
    )


# ============================================================
# 11. DOMAIN FUNCTIONS
# ============================================================

def get_historical_domain(*arrays):
    lat_min = max([
        float(array.lat.min())
        for array in arrays
    ])

    lat_max = min([
        float(array.lat.max())
        for array in arrays
    ])

    lon_min = max([
        float(array.lon.min())
        for array in arrays
    ])

    lon_max = min([
        float(array.lon.max())
        for array in arrays
    ])

    return (
        lat_min,
        lat_max,
        lon_min,
        lon_max
    )


def clip_to_domain(
    da,
    domain
):
    lat_min, lat_max, lon_min, lon_max = domain

    da = da.sortby("lat")
    da = da.sortby("lon")

    return da.sel(
        lat=slice(
            lat_min,
            lat_max
        ),
        lon=slice(
            lon_min,
            lon_max
        )
    )


# ============================================================
# 12. FUTURE COMPONENT READER
# ============================================================

def open_future_component(
    model_name,
    scenario,
    file_name,
    variable_name,
    historical_domain
):
    component_path = os.path.join(
        FUTURE_BASE_DIR,
        model_name,
        scenario,
        file_name
    )

    if not os.path.exists(component_path):
        raise FileNotFoundError(
            component_path
        )

    ds = xr.open_dataset(
        component_path
    )

    selected_variable = select_data_variable(
        ds=ds,
        preferred_variable=variable_name,
        file_path=component_path
    )

    da = ds[selected_variable]

    if "time" not in da.dims:
        raise ValueError(
            f"No time dimension found in {component_path}"
        )

    da["time"] = pd.to_datetime(
        da.time.values
    )

    da = standardize_xy_to_latlon(
        da
    )

    for dimension in [
        "time",
        "lat",
        "lon",
    ]:
        if dimension not in da.dims:
            raise ValueError(
                f"{model_name}/{scenario}/{file_name}: "
                f"dimension '{dimension}' not found."
            )

    da = da.astype("float64")
    da = da.where(np.isfinite(da))

    da = clip_to_domain(
        da,
        historical_domain
    )

    return da


# ============================================================
# 13. COMBINED FUTURE ET READER
# ============================================================

def open_future_combined_et(
    model_name,
    scenario,
    historical_domain
):
    """
    Read and combine the three future ET components.

    Output unit:
        mm/month-equivalent depth stored in the model files.

    The same downstream treatment as the previous code is retained.
    """

    total_et_m = open_future_component(
        model_name=model_name,
        scenario=scenario,
        file_name=FUTURE_TOTAL_ET_FILE,
        variable_name=FUTURE_TOTAL_ET_VAR,
        historical_domain=historical_domain
    )

    water_body_et_m = open_future_component(
        model_name=model_name,
        scenario=scenario,
        file_name=FUTURE_WATER_BODY_FILE,
        variable_name=FUTURE_WATER_BODY_VAR,
        historical_domain=historical_domain
    )

    channel_et_m3 = open_future_component(
        model_name=model_name,
        scenario=scenario,
        file_name=FUTURE_CHANNEL_FILE,
        variable_name=FUTURE_CHANNEL_VAR,
        historical_domain=historical_domain
    )

    # Align time and spatial coordinates exactly.
    (
        total_et_m,
        water_body_et_m,
        channel_et_m3
    ) = xr.align(
        total_et_m,
        water_body_et_m,
        channel_et_m3,
        join="inner"
    )

    if total_et_m.size == 0:
        raise ValueError(
            f"No common data after alignment for "
            f"{model_name}/{scenario}"
        )

    cell_area_m2 = calculate_latlon_cell_area(
        total_et_m.lat,
        total_et_m.lon
    )

    # Convert channel evaporation volume from m3 to m.
    channel_et_m = (
        channel_et_m3 /
        cell_area_m2
    )

    channel_et_m.name = "EvapoChannel_depth_m"

    # Sum the three components in metres.
    combined_et_m = (
        total_et_m
        + water_body_et_m
        + channel_et_m
    )

    # Convert m to mm.
    combined_et_mm = (
        combined_et_m * 1000.0
    )

    if MASK_NEGATIVE_FUTURE_ET:
        combined_et_mm = (
            combined_et_mm
            .where(combined_et_mm >= 0)
        )

    combined_et_mm.name = (
        f"{model_name}_{scenario}_"
        f"combined_ET_three_components"
    )

    combined_et_mm.attrs["units"] = "mm"
    combined_et_mm.attrs["future_mode"] = FUTURE_ET_MODE
    combined_et_mm.attrs["formula"] = (
        "(totalET_monthavg + "
        "EvapWaterBodyM_monthavg + "
        "EvapoChannel_monthavg / cell_area) * 1000"
    )

    components = {
        "totalET_mm": total_et_m * 1000.0,
        "EvapWaterBodyM_mm": water_body_et_m * 1000.0,
        "EvapoChannel_mm": channel_et_m * 1000.0,
        "CombinedET_mm": combined_et_mm,
    }

    return combined_et_mm, components


# ============================================================
# 14. FUTURE BIAS CORRECTION
# ============================================================

def bias_correct_future_series(
    future_series,
    reference_series
):
    """
    Multiplicative mean-ratio correction over the overlap period.
    """

    overlap = pd.concat(
        [
            future_series.rename("future"),
            reference_series.rename("reference"),
        ],
        axis=1
    ).dropna()

    n_overlap = len(overlap)

    if n_overlap < 12:
        return (
            future_series.copy(),
            np.nan,
            n_overlap
        )

    future_mean = overlap["future"].mean()
    reference_mean = overlap["reference"].mean()

    if (
        not np.isfinite(future_mean)
        or future_mean == 0
    ):
        return (
            future_series.copy(),
            np.nan,
            n_overlap
        )

    factor = (
        reference_mean /
        future_mean
    )

    corrected = (
        future_series *
        factor
    )

    corrected.name = future_series.name

    return (
        corrected,
        factor,
        n_overlap
    )


# ============================================================
# 15. HISTORICAL PRODUCT BIAS CORRECTION
# ============================================================

def calculate_additive_historical_bias(
    product_series,
    reference_series,
    product_name,
    reference_name="CWatM"
):
    overlap = pd.concat(
        [
            reference_series.rename("reference"),
            product_series.rename("product"),
        ],
        axis=1
    ).dropna()

    n_overlap = len(overlap)

    if n_overlap < MIN_OVERLAP_MONTHS_FOR_HISTORICAL_BIAS:
        return product_series.copy(), {
            "product": product_name,
            "reference": reference_name,
            "method": HISTORICAL_BIAS_METHOD,
            "status": "insufficient_overlap",
            "overlap_start": (
                overlap.index.min()
                if n_overlap > 0
                else pd.NaT
            ),
            "overlap_end": (
                overlap.index.max()
                if n_overlap > 0
                else pd.NaT
            ),
            "n_overlap_months": n_overlap,
            "reference_mean": np.nan,
            "product_mean": np.nan,
            "additive_bias_mm_day": np.nan,
            "relative_bias_percent": np.nan,
            "mae_before": np.nan,
            "mae_after": np.nan,
            "rmse_before": np.nan,
            "rmse_after": np.nan,
        }

    reference_mean = (
        overlap["reference"]
        .mean()
    )

    product_mean = (
        overlap["product"]
        .mean()
    )

    additive_bias = (
        product_mean -
        reference_mean
    )

    if reference_mean != 0:
        relative_bias_percent = (
            additive_bias /
            reference_mean
        ) * 100.0
    else:
        relative_bias_percent = np.nan

    corrected = (
        product_series -
        additive_bias
    )

    corrected.name = (
        f"{product_name}_BiasCorrected"
    )

    corrected_overlap = pd.concat(
        [
            reference_series.rename("reference"),
            corrected.rename("corrected"),
        ],
        axis=1
    ).dropna()

    error_before = (
        overlap["product"] -
        overlap["reference"]
    )

    error_after = (
        corrected_overlap["corrected"] -
        corrected_overlap["reference"]
    )

    mae_before = np.mean(
        np.abs(error_before)
    )

    mae_after = np.mean(
        np.abs(error_after)
    )

    rmse_before = np.sqrt(
        np.mean(error_before ** 2)
    )

    rmse_after = np.sqrt(
        np.mean(error_after ** 2)
    )

    record = {
        "product": product_name,
        "reference": reference_name,
        "method": HISTORICAL_BIAS_METHOD,
        "status": "corrected",
        "overlap_start": overlap.index.min(),
        "overlap_end": overlap.index.max(),
        "n_overlap_months": n_overlap,
        "reference_mean": reference_mean,
        "product_mean": product_mean,
        "additive_bias_mm_day": additive_bias,
        "relative_bias_percent": relative_bias_percent,
        "mae_before": mae_before,
        "mae_after": mae_after,
        "rmse_before": rmse_before,
        "rmse_after": rmse_after,
    }

    return corrected, record


def correct_historical_products(
    historical_monthly_df
):
    output = historical_monthly_df.copy()
    records = []

    if not APPLY_HISTORICAL_PRODUCT_BIAS_CORRECTION:
        return output, records

    reference = output[
        HISTORICAL_BIAS_REFERENCE
    ]

    for product in HISTORICAL_PRODUCTS_TO_CORRECT:
        if product not in output.columns:
            continue

        corrected, record = (
            calculate_additive_historical_bias(
                product_series=output[product],
                reference_series=reference,
                product_name=product,
                reference_name=HISTORICAL_BIAS_REFERENCE
            )
        )

        output[
            f"{product}_BiasCorrected"
        ] = corrected

        records.append(record)

    return output, records


# ============================================================
# 16. EDGE CORRECTIONS
# ============================================================

def correct_historical_monthly_edges(
    historical_monthly_df
):
    output = historical_monthly_df.copy()
    records = []

    if not CORRECT_MONTHLY_EDGE_VALUES:
        return output, records

    for column in HISTORICAL_FIXED_COLUMNS:
        if column not in output.columns:
            continue

        (
            corrected,
            original,
            replacement,
            status
        ) = replace_target_month_with_mean(
            output[column],
            target_date=HISTORICAL_TARGET_DATE,
            source_dates=HISTORICAL_SOURCE_DATES
        )

        output[column] = corrected

        records.append({
            "series": column,
            "target_date": HISTORICAL_TARGET_DATE,
            "source_dates": ", ".join(
                HISTORICAL_SOURCE_DATES
            ),
            "original_value": original,
            "replacement_value": replacement,
            "status": status,
        })

    return output, records


def correct_future_monthly_edges(
    future_monthly_df,
    scenario
):
    output = future_monthly_df.copy()
    records = []

    if not CORRECT_MONTHLY_EDGE_VALUES:
        return output, records

    for column in output.columns:
        (
            corrected,
            original,
            replacement,
            status
        ) = replace_target_month_with_mean(
            output[column],
            target_date=FUTURE_FIRST_TARGET_DATE,
            source_dates=FUTURE_FIRST_SOURCE_DATES
        )

        output[column] = corrected

        records.append({
            "scenario": scenario,
            "model": column,
            "target_date": FUTURE_FIRST_TARGET_DATE,
            "source_dates": ", ".join(
                FUTURE_FIRST_SOURCE_DATES
            ),
            "original_value": original,
            "replacement_value": replacement,
            "status": status,
        })

        (
            corrected,
            original,
            replacement,
            status
        ) = replace_target_month_with_mean(
            output[column],
            target_date=FUTURE_LAST_TARGET_DATE,
            source_dates=FUTURE_LAST_SOURCE_DATES
        )

        output[column] = corrected

        records.append({
            "scenario": scenario,
            "model": column,
            "target_date": FUTURE_LAST_TARGET_DATE,
            "source_dates": ", ".join(
                FUTURE_LAST_SOURCE_DATES
            ),
            "original_value": original,
            "replacement_value": replacement,
            "status": status,
        })

    return output, records


# ============================================================
# 17. PERIOD LINES AND LABELS
# ============================================================

PERIOD_LINES = [
    pd.Timestamp("2025-01-01"),
    pd.Timestamp("2050-01-01"),
    pd.Timestamp("2075-01-01"),
    pd.Timestamp("2100-01-01"),
]

PERIOD_LABELS = [
    (
        "Historical",
        pd.Timestamp("2000-01-01"),
        pd.Timestamp("2019-12-01")
    ),
    (
        "Near future",
        pd.Timestamp("2025-01-01"),
        pd.Timestamp("2049-12-01")
    ),
    (
        "Mid-century",
        pd.Timestamp("2050-01-01"),
        pd.Timestamp("2074-12-01")
    ),
    (
        "Far future",
        pd.Timestamp("2075-01-01"),
        pd.Timestamp("2099-12-01")
    ),
]


# ============================================================
# 18. PLOTTING
# ============================================================

def plot_scenario_annual_timeseries(
    scenario_label,
    historical_annual_df,
    future_annual_df,
    ensemble_annual_series,
    out_png
):
    fig, ax = plt.subplots(
        figsize=(
            FIG_WIDTH,
            FIG_HEIGHT
        )
    )

    historical_colors = {
        "CWatM": "black",
        "GLDAS": "tab:blue",
        "SMAP": "tab:green",
    }

    model_colors = {
        "GFDL": "tab:blue",
        "IPSL": "tab:orange",
        "MPI": "tab:green",
        "MRI": "tab:purple",
        "UKESM": "tab:brown",
    }

    # Original historical lines
    for column in [
        "CWatM",
        "GLDAS",
        "SMAP",
    ]:
        if column not in historical_annual_df.columns:
            continue

        ax.plot(
            historical_annual_df.index,
            historical_annual_df[column],
            linewidth=LINE_WIDTH_HISTORICAL,
            marker="o",
            markersize=MARKER_SIZE_HISTORICAL,
            linestyle="-",
            label=f"Historical {column}",
            color=historical_colors[column],
            zorder=3
        )

    # Historical corrected products
    corrected_settings = {
        "GLDAS_BiasCorrected": {
            "label": "GLDAS bias-corrected",
            "color": "tab:blue",
        },
        "SMAP_BiasCorrected": {
            "label": "SMAP bias-corrected",
            "color": "tab:green",
        },
    }

    for column, settings in corrected_settings.items():
        if column not in historical_annual_df.columns:
            continue

        ax.plot(
            historical_annual_df.index,
            historical_annual_df[column],
            linewidth=LINE_WIDTH_HISTORICAL_CORRECTED,
            marker="o",
            markersize=MARKER_SIZE_HISTORICAL_CORRECTED,
            linestyle="--",
            dashes=(5, 3),
            label=settings["label"],
            color=settings["color"],
            alpha=0.95,
            zorder=4
        )

    # Future models
    for column in future_annual_df.columns:
        ax.plot(
            future_annual_df.index,
            future_annual_df[column],
            linewidth=LINE_WIDTH_FUTURE_MODEL,
            alpha=ALPHA_FUTURE_MODEL,
            marker="o",
            markersize=MARKER_SIZE_FUTURE,
            linestyle="-",
            color=model_colors.get(
                column,
                None
            ),
            label=f"Future {column}",
            zorder=4
        )

    # Ensemble mean
    ax.plot(
        ensemble_annual_series.index,
        ensemble_annual_series.values,
        linewidth=LINE_WIDTH_ENSEMBLE,
        marker="o",
        markersize=MARKER_SIZE_ENSEMBLE,
        color="red",
        linestyle="-",
        alpha=0.95,
        label="Future ensemble mean",
        zorder=5
    )

    for date in PERIOD_LINES:
        ax.axvline(
            date,
            color="gray",
            linestyle="--",
            linewidth=1.2,
            alpha=0.50,
            zorder=1
        )

    ax.set_xlim(
        pd.Timestamp(PLOT_START_DATE),
        pd.Timestamp(PLOT_END_DATE)
    )

    ymin, ymax = ax.get_ylim()

    for label, start, end in PERIOD_LABELS:
        midpoint = start + (
            end - start
        ) / 2

        ax.text(
            midpoint,
            ymax - 0.035 * (
                ymax - ymin
            ),
            label,
            ha="center",
            va="top",
            fontsize=PERIOD_LABEL_FONT_SIZE,
            color="gray"
        )

    title_extra = (
        "future bias-corrected to historical CWatM"
        if APPLY_FUTURE_BIAS_CORRECTION
        else "raw future"
    )

    ax.set_title(
        f"Historical and Future Annual Mean ET Time Series "
        f"- {scenario_label}\n"
        f"Future mode: totalET + water-body ET + channel ET; "
        f"{title_extra}",
        fontsize=TITLE_FONT_SIZE
    )

    ax.set_xlabel(
        "Time",
        fontsize=AXIS_LABEL_FONT_SIZE
    )

    ax.set_ylabel(
        get_y_label(),
        fontsize=AXIS_LABEL_FONT_SIZE
    )

    ax.tick_params(
        axis="both",
        labelsize=TICK_FONT_SIZE
    )

    ax.grid(
        True,
        alpha=0.25
    )

    ax.legend(
        ncol=3,
        fontsize=LEGEND_FONT_SIZE,
        loc="lower right",
        frameon=True,
        framealpha=0.88,
        edgecolor="gray"
    )

    plt.tight_layout()

    plt.savefig(
        out_png,
        dpi=300,
        bbox_inches="tight"
    )

    plt.close()


# ============================================================
# 19. LOAD HISTORICAL DATA
# ============================================================

print("\nLoading historical ET datasets...")

cwatm = open_historical_et(
    CWATM_FILE,
    "CWatM"
)

gldas = open_historical_et(
    GLDAS_FILE,
    "GLDAS"
)

smap = open_historical_et(
    SMAP_FILE,
    "SMAP"
)

historical_domain = get_historical_domain(
    cwatm,
    gldas,
    smap
)

print("\nHistorical domain intersection:")

print(
    f"lat: {historical_domain[0]:.3f} "
    f"to {historical_domain[1]:.3f}"
)

print(
    f"lon: {historical_domain[2]:.3f} "
    f"to {historical_domain[3]:.3f}"
)

cwatm_clip = clip_to_domain(
    cwatm,
    historical_domain
)

gldas_clip = clip_to_domain(
    gldas,
    historical_domain
)

smap_clip = clip_to_domain(
    smap,
    historical_domain
)

historical_monthly_raw_df = pd.concat(
    [
        to_monthly_area_series(
            cwatm_clip
        ).rename("CWatM"),

        to_monthly_area_series(
            gldas_clip
        ).rename("GLDAS"),

        to_monthly_area_series(
            smap_clip
        ).rename("SMAP"),
    ],
    axis=1
).sort_index()

historical_monthly_raw_df = (
    historical_monthly_raw_df.loc[
        (
            historical_monthly_raw_df.index >=
            pd.Timestamp(HISTORICAL_START_DATE)
        )
        &
        (
            historical_monthly_raw_df.index <=
            pd.Timestamp(HISTORICAL_END_DATE)
        )
    ]
)

(
    historical_monthly_edge_corrected_df,
    historical_edge_records
) = correct_historical_monthly_edges(
    historical_monthly_raw_df
)

(
    historical_monthly_all_df,
    historical_bias_records
) = correct_historical_products(
    historical_monthly_edge_corrected_df
)

historical_annual_df = monthly_to_annual_mean(
    historical_monthly_all_df
)


# ============================================================
# 20. SAVE HISTORICAL TABLES
# ============================================================

if SAVE_MONTHLY_TABLES:
    historical_monthly_raw_df.to_csv(
        os.path.join(
            TABLE_DIR,
            "historical_monthly_raw_final.csv"
        ),
        encoding="utf-8-sig"
    )

    historical_monthly_all_df.to_csv(
        os.path.join(
            TABLE_DIR,
            "historical_monthly_with_bias_corrected_products_final.csv"
        ),
        encoding="utf-8-sig"
    )

if SAVE_ANNUAL_TABLES:
    historical_annual_df.to_csv(
        os.path.join(
            TABLE_DIR,
            "historical_annual_mean_with_bias_corrected_products_final.csv"
        ),
        encoding="utf-8-sig"
    )

historical_bias_df = pd.DataFrame(
    historical_bias_records
)

historical_bias_csv = os.path.join(
    TABLE_DIR,
    "historical_GLDAS_SMAP_bias_correction_summary.csv"
)

historical_bias_df.to_csv(
    historical_bias_csv,
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 21. FUTURE WORKFLOW
# ============================================================

future_summary_records = []
future_bias_records = []
future_edge_records_all = []

component_summary_records = []
combined_future_monthly_records = []
combined_future_annual_records = []

for scenario, scenario_label in SCENARIOS.items():

    print("\n" + "=" * 80)

    print(
        f"Processing scenario: "
        f"{scenario} - {scenario_label}"
    )

    print("=" * 80)

    future_monthly_series = []

    for model in CLIMATE_MODELS:

        print(
            f"\nProcessing model: "
            f"{model}, scenario: {scenario}"
        )

        try:
            (
                future_combined_da,
                component_arrays
            ) = open_future_combined_et(
                model_name=model,
                scenario=scenario,
                historical_domain=historical_domain
            )

            # Area-mean combined ET
            future_combined_raw = (
                to_monthly_area_series(
                    future_combined_da
                )
            )

            future_combined_raw.name = model

            # Save area means of individual components
            component_series = {}

            for (
                component_name,
                component_da
            ) in component_arrays.items():

                component_series[
                    component_name
                ] = to_monthly_area_series(
                    component_da
                )

            component_df = pd.concat(
                component_series,
                axis=1
            )

            component_df["model"] = model
            component_df["scenario"] = scenario

            component_df_reset = (
                component_df
                .reset_index()
                .rename(
                    columns={
                        "index": "time"
                    }
                )
            )

            component_summary_records.append(
                component_df_reset
            )

            if APPLY_FUTURE_BIAS_CORRECTION:
                (
                    future_series_used,
                    factor,
                    n_overlap
                ) = bias_correct_future_series(
                    future_series=future_combined_raw,
                    reference_series=(
                        historical_monthly_edge_corrected_df[
                            FUTURE_BIAS_REFERENCE
                        ]
                    )
                )

            else:
                future_series_used = (
                    future_combined_raw.copy()
                )

                factor = np.nan
                n_overlap = 0

            future_series_used = (
                future_series_used.loc[
                    (
                        future_series_used.index >=
                        pd.Timestamp(FUTURE_START_DATE)
                    )
                    &
                    (
                        future_series_used.index <=
                        pd.Timestamp(FUTURE_END_DATE)
                    )
                ].copy()
            )

            future_series_used.name = model

            future_monthly_series.append(
                future_series_used
            )

            future_bias_records.append({
                "model": model,
                "scenario": scenario,
                "future_mode": FUTURE_ET_MODE,
                "bias_reference": FUTURE_BIAS_REFERENCE,
                "bias_method": (
                    FUTURE_BIAS_METHOD
                    if APPLY_FUTURE_BIAS_CORRECTION
                    else "none"
                ),
                "bias_factor": factor,
                "n_overlap_months": n_overlap,
                "raw_combined_mean": (
                    future_combined_raw.mean()
                ),
                "corrected_mean": (
                    future_series_used.mean()
                ),
            })

        except Exception as error:
            print(
                f"ERROR for {model} / {scenario}: "
                f"{error}"
            )

    if len(future_monthly_series) == 0:
        print(
            f"No valid future data for "
            f"scenario {scenario}. Plot skipped."
        )
        continue

    future_monthly_df = pd.concat(
        future_monthly_series,
        axis=1
    ).sort_index()

    available_models = [
        model
        for model in CLIMATE_MODELS
        if model in future_monthly_df.columns
    ]

    future_monthly_df = future_monthly_df[
        available_models
    ]

    (
        future_monthly_corrected_df,
        future_edge_records
    ) = correct_future_monthly_edges(
        future_monthly_df,
        scenario=scenario
    )

    future_edge_records_all.extend(
        future_edge_records
    )

    future_annual_df = monthly_to_annual_mean(
        future_monthly_corrected_df
    )

    ensemble_annual_series = (
        future_annual_df.mean(
            axis=1,
            skipna=True
        )
    )

    ensemble_annual_series.name = (
        "Future_Ensemble_Mean"
    )

    for model in future_annual_df.columns:
        valid = (
            future_annual_df[model]
            .dropna()
        )

        future_summary_records.append({
            "model": model,
            "scenario": scenario,
            "scenario_label": scenario_label,
            "future_mode": FUTURE_ET_MODE,
            "bias_corrected": APPLY_FUTURE_BIAS_CORRECTION,
            "start": (
                valid.index.min()
                if len(valid) > 0
                else pd.NaT
            ),
            "end": (
                valid.index.max()
                if len(valid) > 0
                else pd.NaT
            ),
            "n_years": len(valid),
            "annual_mean": valid.mean(),
            "annual_min": valid.min(),
            "annual_max": valid.max(),
            "domain_lat_min": historical_domain[0],
            "domain_lat_max": historical_domain[1],
            "domain_lon_min": historical_domain[2],
            "domain_lon_max": historical_domain[3],
        })

    if SAVE_MONTHLY_TABLES:
        future_monthly_corrected_df.to_csv(
            os.path.join(
                TABLE_DIR,
                f"future_monthly_three_components_"
                f"scenario_{scenario}_final.csv"
            ),
            encoding="utf-8-sig"
        )

    if SAVE_ANNUAL_TABLES:
        future_annual_df.to_csv(
            os.path.join(
                TABLE_DIR,
                f"future_annual_three_components_"
                f"scenario_{scenario}_final.csv"
            ),
            encoding="utf-8-sig"
        )

        ensemble_annual_series.to_frame().to_csv(
            os.path.join(
                TABLE_DIR,
                f"future_ensemble_annual_three_components_"
                f"scenario_{scenario}_final.csv"
            ),
            encoding="utf-8-sig"
        )

    temporary_monthly = (
        future_monthly_corrected_df.copy()
    )

    temporary_monthly["scenario"] = scenario
    temporary_monthly["scenario_label"] = scenario_label

    temporary_monthly = (
        temporary_monthly
        .reset_index()
        .rename(
            columns={
                "index": "time"
            }
        )
    )

    combined_future_monthly_records.append(
        temporary_monthly
    )

    temporary_annual = (
        future_annual_df.copy()
    )

    temporary_annual[
        "Ensemble_Mean"
    ] = ensemble_annual_series

    temporary_annual["scenario"] = scenario
    temporary_annual["scenario_label"] = scenario_label

    temporary_annual = (
        temporary_annual
        .reset_index()
        .rename(
            columns={
                "index": "time"
            }
        )
    )

    combined_future_annual_records.append(
        temporary_annual
    )

    output_figure = os.path.join(
        FIG_DIR,
        f"annual_historical_future_ET_"
        f"scenario_{scenario}_"
        f"three_components_test.png"
    )

    plot_scenario_annual_timeseries(
        scenario_label=scenario_label,
        historical_annual_df=historical_annual_df,
        future_annual_df=future_annual_df,
        ensemble_annual_series=ensemble_annual_series,
        out_png=output_figure
    )


# ============================================================
# 22. SAVE SUMMARY TABLES
# ============================================================

future_summary_df = pd.DataFrame(
    future_summary_records
)

future_summary_csv = os.path.join(
    TABLE_DIR,
    "future_models_annual_three_components_summary.csv"
)

future_summary_df.to_csv(
    future_summary_csv,
    index=False,
    encoding="utf-8-sig"
)

future_bias_df = pd.DataFrame(
    future_bias_records
)

future_bias_csv = os.path.join(
    TABLE_DIR,
    "future_three_components_bias_factors.csv"
)

future_bias_df.to_csv(
    future_bias_csv,
    index=False,
    encoding="utf-8-sig"
)

historical_edge_df = pd.DataFrame(
    historical_edge_records
)

historical_edge_csv = os.path.join(
    TABLE_DIR,
    "historical_monthly_edge_corrections_final.csv"
)

historical_edge_df.to_csv(
    historical_edge_csv,
    index=False,
    encoding="utf-8-sig"
)

future_edge_df = pd.DataFrame(
    future_edge_records_all
)

future_edge_csv = os.path.join(
    TABLE_DIR,
    "future_monthly_edge_corrections_three_components.csv"
)

future_edge_df.to_csv(
    future_edge_csv,
    index=False,
    encoding="utf-8-sig"
)

if len(component_summary_records) > 0:
    all_components_df = pd.concat(
        component_summary_records,
        axis=0,
        ignore_index=True
    )

    components_csv = os.path.join(
        TABLE_DIR,
        "future_three_components_monthly_area_means_all.csv"
    )

    all_components_df.to_csv(
        components_csv,
        index=False,
        encoding="utf-8-sig"
    )

else:
    components_csv = None

if len(combined_future_monthly_records) > 0:
    combined_monthly_df = pd.concat(
        combined_future_monthly_records,
        axis=0,
        ignore_index=True
    )

    combined_monthly_csv = os.path.join(
        TABLE_DIR,
        "future_monthly_three_components_all_scenarios.csv"
    )

    combined_monthly_df.to_csv(
        combined_monthly_csv,
        index=False,
        encoding="utf-8-sig"
    )

else:
    combined_monthly_csv = None

if len(combined_future_annual_records) > 0:
    combined_annual_df = pd.concat(
        combined_future_annual_records,
        axis=0,
        ignore_index=True
    )

    combined_annual_csv = os.path.join(
        TABLE_DIR,
        "future_annual_three_components_all_scenarios.csv"
    )

    combined_annual_df.to_csv(
        combined_annual_csv,
        index=False,
        encoding="utf-8-sig"
    )

else:
    combined_annual_csv = None


# ============================================================
# 23. SAVE CONFIGURATION
# ============================================================

configuration_df = pd.DataFrame([
    {
        "parameter": "analysis_type",
        "value": "annual_mean_three_future_ET_components_test"
    },
    {
        "parameter": "future_mode",
        "value": FUTURE_ET_MODE
    },
    {
        "parameter": "future_formula",
        "value": (
            "(totalET_monthavg + "
            "EvapWaterBodyM_monthavg + "
            "EvapoChannel_monthavg/cell_area) * 1000"
        )
    },
    {
        "parameter": "total_et_file",
        "value": FUTURE_TOTAL_ET_FILE
    },
    {
        "parameter": "water_body_file",
        "value": FUTURE_WATER_BODY_FILE
    },
    {
        "parameter": "channel_file",
        "value": FUTURE_CHANNEL_FILE
    },
    {
        "parameter": "channel_input_unit",
        "value": "m3"
    },
    {
        "parameter": "channel_conversion",
        "value": "m3 / cell_area_m2 = m"
    },
    {
        "parameter": "final_future_unit",
        "value": "mm"
    },
    {
        "parameter": "historical_bias_method",
        "value": HISTORICAL_BIAS_METHOD
    },
    {
        "parameter": "future_bias_method",
        "value": FUTURE_BIAS_METHOD
    },
    {
        "parameter": "legend_location",
        "value": "lower right"
    },
    {
        "parameter": "historical_start",
        "value": HISTORICAL_START_DATE
    },
    {
        "parameter": "historical_end",
        "value": HISTORICAL_END_DATE
    },
    {
        "parameter": "future_start",
        "value": FUTURE_START_DATE
    },
    {
        "parameter": "future_end",
        "value": FUTURE_END_DATE
    },
    {
        "parameter": "domain_lat_min",
        "value": historical_domain[0]
    },
    {
        "parameter": "domain_lat_max",
        "value": historical_domain[1]
    },
    {
        "parameter": "domain_lon_min",
        "value": historical_domain[2]
    },
    {
        "parameter": "domain_lon_max",
        "value": historical_domain[3]
    },
])

configuration_csv = os.path.join(
    TABLE_DIR,
    "run_configuration_three_components_test.csv"
)

configuration_df.to_csv(
    configuration_csv,
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 24. FINAL REPORT
# ============================================================

print("\n" + "=" * 80)
print("DONE")
print("=" * 80)

print("\nFuture ET formula:")
print(
    "Combined ET = "
    "(totalET + EvapWaterBodyM + "
    "EvapoChannel / cell area) * 1000"
)

print("\nMain output folder:")
print(OUT_DIR)

print("\nMain tables:")
print(historical_bias_csv)
print(future_summary_csv)
print(future_bias_csv)
print(historical_edge_csv)
print(future_edge_csv)
print(configuration_csv)

if components_csv is not None:
    print(components_csv)

if combined_monthly_csv is not None:
    print(combined_monthly_csv)

if combined_annual_csv is not None:
    print(combined_annual_csv)

print("\nFigures:")

for scenario in SCENARIOS:
    print(
        os.path.join(
            FIG_DIR,
            f"annual_historical_future_ET_"
            f"scenario_{scenario}_"
            f"three_components_test.png"
        )
    )

print("\nImportant:")
print(
    "The channel evaporation variable is converted from m3 "
    "to equivalent water depth by dividing by grid-cell area."
)
print(
    "All other plotting details are retained from the "
    "previous accepted version."
)


Loading historical ET datasets...

Opening historical dataset: CWatM
E:\ET_Article_With_MM_SA\Future data ST\ET_CWatM_mm_day.nc
Variable: ET_final_mm_day
Shape: (468, 13, 14)
Time: 1981-01 to 2019-12
Mean: 1.102419

Opening historical dataset: GLDAS
E:\ET_Article_With_MM_SA\Future data ST\GLDAS_ET_2000_2019.nc
Variable: __xarray_dataarray_variable__
Shape: (240, 13, 14)
Time: 2000-01 to 2019-12
Mean: 0.793585

Opening historical dataset: SMAP
E:\ET_Article_With_MM_SA\Future data ST\SMAP_ET_2000_2019.nc
Variable: __xarray_dataarray_variable__
Shape: (58, 13, 14)
Time: 2015-03 to 2019-12
Mean: 1.057844

Historical domain intersection:
lat: 30.750 to 36.750
lon: 45.250 to 51.750

Processing scenario: 126 - Optimistic / SSP126

Processing model: GFDL, scenario: 126

Processing model: IPSL, scenario: 126

Processing model: MPI, scenario: 126

Processing model: MRI, scenario: 126

Processing model: UKESM, scenario: 126

Processing scenario: 370 - Medium / SSP370

Processing model: GFDL, sce